# CardioSentry — ConvNeXt-V2 Tiny domain-adaptive FCMAE

**Baseline v1:** ImageNet-1K self-supervised FCMAE initialization → continued
ECG-image FCMAE → exported encoder for later supervised fine-tuning.

| Setting | Baseline |
|---|---|
| Image | RGB, **768 height × 1024 width** |
| Resize | Preserve aspect ratio, center-pad; no flips or random crops |
| Encoder | Official ConvNeXt-V2 Tiny sparse encoder |
| Mask | Uniform random 60% of patches containing valid image pixels |
| Patch grid | 32-pixel patches; 24 × 32 grid |
| Objective | Masked patch MSE, normalized targets, padding pixels excluded |
| Decoder | Official depth 1, width 512; randomly initialized (release is encoder-only) |
| Optimization | AdamW, warmup/cosine, configurable accumulation and FP16 |

This is a **rectangular/padding adaptation** of Meta's FCMAE, not an unmodified
reproduction. It retains the actual Minkowski sparse encoder and its GRN
behavior. The original sparse GRN aggregates active sites across a microbatch;
gradient accumulation does not make that statistic equivalent to a larger
microbatch. Record both batch size and accumulation in comparisons.

The notebook contains its supporting Python files; upload this `.ipynb` alone
to `MyDrive/CardioSentry/codes/fcmae_tiny_v1/`. Run cells in order on a CUDA Colab
runtime (L4/A100 preferred; capacity must be measured by the pilot).
It builds a separate Python 3.10/CUDA 11.8 runtime for the legacy sparse engine.
Allow at least 20 GiB for this environment **in addition to the image archive
and extracted images**. The notebook kernel itself is not replaced.

**Execution status:** CPU utility tests are supplied. CUDA compilation, the real
checkpoint, sparse parity, and the full-size training step are validated by the
runtime/pilot cells on your GPU. Do not interpret an unexecuted notebook as a
successful GPU run.


## 1. Mount Drive and configure inputs

The archive path matches the current diagnostic-fidelity image generator.
Generated masters remain unchanged. Extraction and batch reads use Colab's
local disk; periodic resumable checkpoints and metrics go to Drive.

Start with `RUN_FULL = False`. The pilot uses a small subset at the **actual
768 × 1024 input resolution**, and does not authorize claims about model quality.
The notebook embeds a canonical downstream validation/test exclusion CSV from
the current supervised V2 release. Its source hash is in `holdout_provenance.json`.
Rebuild it with `make_holdouts.py` if downstream splits change, or supply your
replacement through `HOLDOUT_CSV`.
Required columns: `source_dataset`, `canonical_record_id`, `patient_id`,
`patient_id_available` (1 or 0). Include every downstream validation/test record;
use the same canonical source namespace as the pretraining index. Unknown patient
IDs are reported rather than claimed to be patient-disjoint. Existing record
overlap removal in the corpus is not a substitute for this audit.

`SSL validation` below means a held-out subset of the eligible pretraining
corpus. It is separate from downstream validation/test. No pathology labels
are passed to the model.


In [ ]:
from pathlib import Path
import os, json, subprocess, shutil, sys, time
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/CardioSentry')
ARCHIVE = DRIVE/'experiments/fcmae_synthetic_images_diagnostic_v1/fcmae-synthetic-images-diagnostic-v1.tar'
ARCHIVE_SHA = Path(str(ARCHIVE)+'.sha256')
WORK = Path('/content/cardiosentry_fcmae_tiny_v1')
CODE = WORK/'code'
RUNTIME = WORK/'runtime'
EXPERIMENT = DRIVE/'experiments/fcmae_tiny_v1'
RUN = DRIVE/'runs/fcmae_tiny_v1_768_seed42'
HOLDOUT_CSV = CODE/'downstream_holdouts_v2.csv'  # Embedded current V2 snapshot; replace if splits change.
RUN_FULL = False
RUN_BENCHMARK = True
BENCHMARK_ONLY = True  # Keep True while comparing GPU costs; later set False for the pilot/training.
COMPUTE_UNITS_PER_HOUR = None  # Enter the current rate shown in Colab; 0 only for an actually free session.
PREPARATION_TIMINGS = {}
PILOT_PASSED = False
PYTHON_OVERRIDE = None  # Optional existing compatible Python executable.
for path in [WORK, CODE, EXPERIMENT, RUN]:
    path.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE/'hf_cache')

# Initial engineering schedule, not an experimentally established optimum.
# Use a new RUN directory when changing these settings.
CONFIG = dict(input_size=[768, 1024], seed=42, batch_size=2, accumulation=16,
              workers=2, epochs=100, warmup_epochs=5, learning_rate=2e-5,
              amp=True, clip_grad=1.0, ssl_val_fraction=.05, save_every=10,
              checkpoint_every_updates=100, checkpoint_every_minutes=15,
              run_dir=str(RUN), holdout_csv=str(HOLDOUT_CSV) if HOLDOUT_CSV else None)
assert all(n % 32 == 0 for n in CONFIG['input_size'])
print('Effective batch:', CONFIG['batch_size']*CONFIG['accumulation'])
print('Full run enabled:', RUN_FULL)
print('Benchmark only:', BENCHMARK_ONLY)
subprocess.run(['nvidia-smi'], check=True)


## 2. Materialize the pinned code

The embedded official model files are from ConvNeXt-V2 commit
`2553895753323c6fe0b2bf390683f5ea358a42b9`. The exact sparse-engine fork revision is
`shwoo93/MinkowskiEngine@bbc30ef581ea6deb505976b663f5fc2358a83749`.
The adaptations live in separate modules; original model files and license
are preserved. Rebuild this notebook with `make_notebook.py` after editing a
companion module.


In [ ]:
import base64, hashlib
PAYLOAD = {'data_utils.py': 'IiIiRGF0YSBhbmQgcmVjdGFuZ3VsYXIgcmVjb25zdHJ1Y3Rpb24gdXRpbGl0aWVzOyBubyBNaW5rb3dza2lFbmdpbmUgZGVwZW5kZW5jeS4iIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBQSUwgaW1wb3J0IEltYWdlCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKCk1FQU4gPSAoMC40ODUsIDAuNDU2LCAwLjQwNikKU1REID0gKDAuMjI5LCAwLjIyNCwgMC4yMjUpCgoKZGVmIHNoYTI1NihwYXRoKToKICAgIGggPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRoIG9wZW4ocGF0aCwgJ3JiJykgYXMgc3RyZWFtOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTogc3RyZWFtLnJlYWQoOCA8PCAyMCksIGInJyk6CiAgICAgICAgICAgIGgudXBkYXRlKGJsb2NrKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgcGF0Y2hpZnkoeCwgcD0zMik6CiAgICBuLCBjLCBoLCB3ID0geC5zaGFwZQogICAgaWYgaCAlIHAgb3IgdyAlIHA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQm90aCBpbWFnZSBkaW1lbnNpb25zIG11c3QgYmUgZGl2aXNpYmxlIGJ5IHBhdGNoIHNpemUnKQogICAgcmV0dXJuIHgucmVzaGFwZShuLCBjLCBoLy9wLCBwLCB3Ly9wLCBwKS5wZXJtdXRlKDAsIDIsIDQsIDMsIDUsIDEpLnJlc2hhcGUobiwgKGgvL3ApKih3Ly9wKSwgcCpwKmMpCgoKZGVmIHVucGF0Y2hpZnkoeCwgZ3JpZCwgY2hhbm5lbHM9MywgcD0zMik6CiAgICBnaCwgZ3cgPSBncmlkCiAgICByZXR1cm4geC5yZXNoYXBlKC0xLCBnaCwgZ3csIHAsIHAsIGNoYW5uZWxzKS5wZXJtdXRlKDAsIDUsIDEsIDMsIDIsIDQpLnJlc2hhcGUoLTEsIGNoYW5uZWxzLCBnaCpwLCBndypwKQoKCmRlZiB1bmlmb3JtX21hc2sodmFsaWQsIHJhdGlvPS42LCBnZW5lcmF0b3I9Tm9uZSwgcD0zMik6CiAgICAiIiJFeGFjdGx5IHJvdW5kKHJhdGlvICogZWxpZ2libGVfY291bnQpIG1hc2tlZCBwYXRjaGVzLCB3aXRoIHZpc2libGUgY29udGV4dC4iIiIKICAgIGlmIG5vdCAwIDwgcmF0aW8gPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ21hc2sgcmF0aW8gbXVzdCBiZSBiZXR3ZWVuIHplcm8gYW5kIG9uZScpCiAgICBlbGlnaWJsZSA9IHBhdGNoaWZ5KHZhbGlkLCBwKS5zdW0oLTEpID4gMAogICAgbWFzayA9IHRvcmNoLnplcm9zX2xpa2UoZWxpZ2libGUsIGR0eXBlPXZhbGlkLmR0eXBlKQogICAgZm9yIGksIHJvdyBpbiBlbnVtZXJhdGUoZWxpZ2libGUpOgogICAgICAgIGlkcyA9IHJvdy5ub256ZXJvKCkuZmxhdHRlbigpCiAgICAgICAgaWYgbGVuKGlkcykgPCAyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdOZWVkIGF0IGxlYXN0IHR3byBub24tcGFkZGluZyBwYXRjaGVzJykKICAgICAgICBjb3VudCA9IG1heCgxLCBtaW4obGVuKGlkcyktMSwgcm91bmQobGVuKGlkcykqcmF0aW8pKSkKICAgICAgICBvcmRlciA9IHRvcmNoLnJhbmRwZXJtKGxlbihpZHMpLCBkZXZpY2U9dmFsaWQuZGV2aWNlLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKQogICAgICAgIG1hc2tbaSwgaWRzW29yZGVyWzpjb3VudF1dXSA9IDEKICAgIHJldHVybiBtYXNrCgoKZGVmIHRhcmdldHMoaW1hZ2VzLCB2YWxpZCwgbm9ybT1UcnVlLCBwPTMyKToKICAgIHJhdyA9IHBhdGNoaWZ5KGltYWdlcy5mbG9hdCgpLCBwKQogICAgd2VpZ2h0cyA9IHBhdGNoaWZ5KHZhbGlkLmZsb2F0KCkuZXhwYW5kKC0xLCBpbWFnZXMuc2hhcGVbMV0sIC0xLCAtMSksIHApCiAgICBjb3VudCA9IHdlaWdodHMuc3VtKC0xLCBrZWVwZGltPVRydWUpCiAgICBtZWFuID0gKHJhdyAqIHdlaWdodHMpLnN1bSgtMSwga2VlcGRpbT1UcnVlKSAvIGNvdW50LmNsYW1wX21pbigxKQogICAgIyBNYXRjaCBvZmZpY2lhbCB1bmJpYXNlZCB2YXJpYW5jZSBmb3IgZnVsbHkgdmFsaWQgcGF0Y2hlcy4KICAgIHZhciA9ICgocmF3LW1lYW4pLnNxdWFyZSgpKndlaWdodHMpLnN1bSgtMSwga2VlcGRpbT1UcnVlKSAvIChjb3VudC0xKS5jbGFtcF9taW4oMSkKICAgIHRhcmdldCA9IChyYXctbWVhbikgLyAodmFyKzFlLTYpLnNxcnQoKSBpZiBub3JtIGVsc2UgcmF3CiAgICByZXR1cm4gdGFyZ2V0LCB3ZWlnaHRzLCBtZWFuLCB2YXIKCgpkZWYgcmVjb25zdHJ1Y3Rpb25fbG9zcyhpbWFnZXMsIHByZWQsIHZhbGlkLCBtYXNrLCBub3JtPVRydWUsIHA9MzIpOgogICAgdGFyZ2V0LCB3ZWlnaHRzLCBfLCBfID0gdGFyZ2V0cyhpbWFnZXMsIHZhbGlkLCBub3JtLCBwKQogICAgaWYgcHJlZC5uZGltID09IDQ6CiAgICAgICAgcHJlZCA9IHByZWQuZmxhdHRlbigyKS50cmFuc3Bvc2UoMSwgMikKICAgIHBlcl9wYXRjaCA9ICgocHJlZC5mbG9hdCgpLXRhcmdldCkuc3F1YXJlKCkqd2VpZ2h0cykuc3VtKC0xKSAvIHdlaWdodHMuc3VtKC0xKS5jbGFtcF9taW4oMSkKICAgIHNlbGVjdGVkID0gbWFzayAqICh3ZWlnaHRzLnN1bSgtMSkgPiAwKQogICAgaWYgc2VsZWN0ZWQuc3VtKCkgPT0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdObyBtYXNrZWQgdmFsaWQgcGF0Y2hlcycpCiAgICByZXR1cm4gKHBlcl9wYXRjaCpzZWxlY3RlZCkuc3VtKCkvc2VsZWN0ZWQuc3VtKCkKCgpkZWYgbGV0dGVyYm94KHBhdGgsIHNpemUsIGNhY2hlX3BhdGg9Tm9uZSk6CiAgICBoLCB3ID0gc2l6ZQogICAgY2FjaGVkID0gY2FjaGVfcGF0aCBpcyBub3QgTm9uZSBhbmQgUGF0aChjYWNoZV9wYXRoKS5pc19maWxlKCkKICAgIHdpdGggSW1hZ2Uub3BlbihjYWNoZV9wYXRoIGlmIGNhY2hlZCBlbHNlIHBhdGgpIGFzIG9yaWdpbmFsOgogICAgICAgIGltID0gb3JpZ2luYWwuY29udmVydCgnUkdCJykKICAgICAgICBzY2FsZSA9IG1pbih3L2ltLndpZHRoLCBoL2ltLmhlaWdodCkKICAgICAgICBudywgbmggPSBtYXgoMSwgcm91bmQoaW0ud2lkdGgqc2NhbGUpKSwgbWF4KDEsIHJvdW5kKGltLmhlaWdodCpzY2FsZSkpCiAgICAgICAgaW0gPSBpbS5yZXNpemUoKG53LCBuaCksIEltYWdlLlJlc2FtcGxpbmcuTEFOQ1pPUykKICAgICAgICBpZiBjYWNoZV9wYXRoIGlzIG5vdCBOb25lIGFuZCBub3QgY2FjaGVkOgogICAgICAgICAgICBjYWNoZV9wYXRoID0gUGF0aChjYWNoZV9wYXRoKTsgY2FjaGVfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICB0ZW1wID0gY2FjaGVfcGF0aC53aXRoX3N1ZmZpeChmJy57b3MuZ2V0cGlkKCl9LnRtcCcpCiAgICAgICAgICAgICMgQ2FjaGUgdGhlIHJlc2l6ZWQsIFVOUEFEREVEIGltYWdlLiBJdHMgZGltZW5zaW9ucyByZXRhaW4gdGhlCiAgICAgICAgICAgICMgZXhhY3QgdmFsaWQgcmVjdGFuZ2xlOyBubyBwZXItcGl4ZWwgbWFzayBuZWVkcyB0byBiZSBzdG9yZWQuCiAgICAgICAgICAgIGltLnNhdmUodGVtcCwgZm9ybWF0PSdQTkcnKTsgb3MucmVwbGFjZSh0ZW1wLCBjYWNoZV9wYXRoKQogICAgICAgIGNhbnZhcyA9IEltYWdlLm5ldygnUkdCJywgKHcsIGgpLCAoMjU1LCAyNTUsIDI1NSkpCiAgICAgICAgbGVmdCwgdG9wID0gKHctbncpLy8yLCAoaC1uaCkvLzIKICAgICAgICBjYW52YXMucGFzdGUoaW0sIChsZWZ0LCB0b3ApKQogICAgaW1hZ2UgPSB0b3JjaC5mcm9tX251bXB5KG5wLmFzYXJyYXkoY2FudmFzKS5jb3B5KCkpLnBlcm11dGUoMiwgMCwgMSkuZmxvYXQoKS8yNTUKICAgIHZhbGlkID0gdG9yY2guemVyb3MoMSwgaCwgdykKICAgIHZhbGlkWzosIHRvcDp0b3ArbmgsIGxlZnQ6bGVmdCtud10gPSAxCiAgICBpbWFnZSA9IChpbWFnZS10b3JjaC50ZW5zb3IoTUVBTilbOiwgTm9uZSwgTm9uZV0pL3RvcmNoLnRlbnNvcihTVEQpWzosIE5vbmUsIE5vbmVdCiAgICByZXR1cm4gaW1hZ2UsIHZhbGlkCgoKY2xhc3MgRUNHRGF0YXNldChEYXRhc2V0KToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBmcmFtZSwgcm9vdCwgc2l6ZSwgY2FjaGVfcm9vdD1Ob25lKToKICAgICAgICBzZWxmLmZyYW1lLCBzZWxmLnJvb3QsIHNlbGYuc2l6ZSA9IGZyYW1lLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSksIFBhdGgocm9vdCkucmVzb2x2ZSgpLCBzaXplCiAgICAgICAgc2VsZi5jYWNoZV9yb290ID0gUGF0aChjYWNoZV9yb290KSBpZiBjYWNoZV9yb290IGVsc2UgTm9uZQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBsZW4oc2VsZi5mcmFtZSkKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaSk6CiAgICAgICAgcm93ID0gc2VsZi5mcmFtZS5pbG9jW2ldCiAgICAgICAgcGF0aCA9IChzZWxmLnJvb3QgLyByb3cuaW1hZ2VfcGF0aCkucmVzb2x2ZSgpCiAgICAgICAgaWYgbm90IHBhdGguaXNfcmVsYXRpdmVfdG8oc2VsZi5yb290KToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW1hZ2UgcGF0aCBlc2NhcGVzIGRhdGFzZXQgcm9vdCcpCiAgICAgICAgY2FjaGVfcGF0aCA9IE5vbmUKICAgICAgICBpZiBzZWxmLmNhY2hlX3Jvb3Q6CiAgICAgICAgICAgIGlkZW50aXR5ID0gZid7cm93LmltYWdlX3BhdGh9Ontyb3cuZ2V0KCJpbWFnZV9zaGEyNTYiLCAiIil9OntzZWxmLnNpemV9OmxhbmN6b3MtdjEnCiAgICAgICAgICAgIGNhY2hlX3BhdGggPSBzZWxmLmNhY2hlX3Jvb3QvKGhhc2hsaWIuc2hhMjU2KGlkZW50aXR5LmVuY29kZSgpKS5oZXhkaWdlc3QoKSsnLnBuZycpCiAgICAgICAgaW1hZ2UsIHZhbGlkID0gbGV0dGVyYm94KHBhdGgsIHNlbGYuc2l6ZSwgY2FjaGVfcGF0aCkKICAgICAgICByZXR1cm4gaW1hZ2UsIHZhbGlkLCByb3cuZmNtYWVfcmVjb3JkX2lkCgoKZGVmIHByZXBhcmVfc3BsaXRzKGluZGV4X3BhdGgsIHJvb3QsIG91dCwgc2VlZCwgaG9sZG91dF9wYXRoPU5vbmUsIHZhbF9mcmFjdGlvbj0uMDUpOgogICAgIiIiQ2Fub25pY2FsIElEcyBtdXN0IHVzZSB0aGUgc2FtZSBzb3VyY2UgbmFtZXNwYWNlIGluIGluZGV4IGFuZCBob2xkb3V0IENTVi4KCiAgICBBIG1pc3NpbmcgaG9sZG91dCBDU1YgcGVybWl0cyBhIGRpYWdub3N0aWMgcGlsb3Qgb25seS4gRnVsbCB0cmFpbmluZyBjaGVja3MKICAgIHRoZSByZXR1cm5lZCBhdWRpdC4gVW5rbm93biBwYXRpZW50IElEcyBhcmUgZXhwbGljaXRseSByZWNvcmQtZ3JvdXBlZC4KICAgICIiIgogICAgZGYgPSBwZC5yZWFkX2NzdihpbmRleF9wYXRoLCBkdHlwZT1zdHIsIGtlZXBfZGVmYXVsdF9uYT1GYWxzZSkKICAgIHJlcXVpcmVkID0geydpbWFnZV9wYXRoJywgJ2ZjbWFlX3JlY29yZF9pZCcsICdzb3VyY2VfZGF0YXNldCcsICdjYW5vbmljYWxfcmVjb3JkX2lkJywgJ3BhdGllbnRfaWQnLCAncGF0aWVudF9pZF9hdmFpbGFibGUnfQogICAgaWYgbWlzc2luZyA6PSByZXF1aXJlZC1zZXQoZGYuY29sdW1ucyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ0luZGV4IG1pc3NpbmcgY29sdW1uczoge3NvcnRlZChtaXNzaW5nKX07IGpvaW4gcmF3IG1hbmlmZXN0IGJ5IGZjbWFlX3JlY29yZF9pZCBmaXJzdCcpCiAgICBpZiBub3QgZGYuZmNtYWVfcmVjb3JkX2lkLmlzX3VuaXF1ZSBvciBub3QgZGYuaW1hZ2VfcGF0aC5pc191bmlxdWU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRXhwZWN0ZWQgZXhhY3RseSBvbmUgaW1hZ2UgcGVyIHNvdXJjZSByZWNvcmQnKQogICAgZm9yIGNvbCBpbiBbJ2ZjbWFlX3JlY29yZF9pZCcsICdzb3VyY2VfZGF0YXNldCcsICdjYW5vbmljYWxfcmVjb3JkX2lkJywgJ2ltYWdlX3BhdGgnXToKICAgICAgICBpZiBkZltjb2xdLmVxKCcnKS5hbnkoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ0VtcHR5IHtjb2x9JykKICAgIGtub3duID0gZGYucGF0aWVudF9pZF9hdmFpbGFibGUuc3RyLmxvd2VyKCkuaXNpbihbJzEnLCAndHJ1ZSddKSAmIGRmLnBhdGllbnRfaWQubmUoJycpCiAgICBkZlsncmVjb3JkX2tleSddID0gZGYuc291cmNlX2RhdGFzZXQrJzo6JytkZi5jYW5vbmljYWxfcmVjb3JkX2lkCiAgICBkZlsncGF0aWVudF9rZXknXSA9IG5wLndoZXJlKGtub3duLCBkZi5zb3VyY2VfZGF0YXNldCsnOjonK2RmLnBhdGllbnRfaWQsICcnKQogICAgaWYgZGYucmVjb3JkX2tleS5kdXBsaWNhdGVkKCkuYW55KCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRHVwbGljYXRlIGNhbm9uaWNhbCBzb3VyY2UgcmVjb3JkcycpCiAgICBleGNsdWRlZCA9IG5wLnplcm9zKGxlbihkZiksIGR0eXBlPWJvb2wpCiAgICBob2xkb3V0X2hhc2ggPSBOb25lCiAgICBpZiBob2xkb3V0X3BhdGg6CiAgICAgICAgaGQgPSBwZC5yZWFkX2Nzdihob2xkb3V0X3BhdGgsIGR0eXBlPXN0ciwga2VlcF9kZWZhdWx0X25hPUZhbHNlKQogICAgICAgIG5lZWQgPSB7J3NvdXJjZV9kYXRhc2V0JywgJ2Nhbm9uaWNhbF9yZWNvcmRfaWQnLCAncGF0aWVudF9pZCcsICdwYXRpZW50X2lkX2F2YWlsYWJsZSd9CiAgICAgICAgaWYgbmVlZC1zZXQoaGQuY29sdW1ucykgb3IgaGQuZW1wdHk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0hvbGRvdXQgQ1NWIG11c3QgY29udGFpbiBhbGwgZG93bnN0cmVhbSB2YWxpZGF0aW9uL3Rlc3QgcmVjb3JkcyBhbmQgcmVxdWlyZWQgSUQgY29sdW1ucycpCiAgICAgICAgaWYgaGQuc291cmNlX2RhdGFzZXQuZXEoJycpLmFueSgpIG9yIGhkLmNhbm9uaWNhbF9yZWNvcmRfaWQuZXEoJycpLmFueSgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdIb2xkb3V0IHNvdXJjZSBhbmQgY2Fub25pY2FsIHJlY29yZCBJRHMgbXVzdCBub3QgYmUgZW1wdHknKQogICAgICAgIGhrID0gaGQucGF0aWVudF9pZF9hdmFpbGFibGUuc3RyLmxvd2VyKCkuaXNpbihbJzEnLCAndHJ1ZSddKSAmIGhkLnBhdGllbnRfaWQubmUoJycpCiAgICAgICAgZXhjbHVkZWQgPSBkZi5yZWNvcmRfa2V5LmlzaW4oaGQuc291cmNlX2RhdGFzZXQrJzo6JytoZC5jYW5vbmljYWxfcmVjb3JkX2lkKQogICAgICAgIGV4Y2x1ZGVkIHw9IGtub3duICYgZGYucGF0aWVudF9rZXkuaXNpbigoaGQuc291cmNlX2RhdGFzZXQrJzo6JytoZC5wYXRpZW50X2lkKVtoa10pCiAgICAgICAgaG9sZG91dF9oYXNoID0gc2hhMjU2KGhvbGRvdXRfcGF0aCkKICAgIGRmWydncm91cF9rZXknXSA9IG5wLndoZXJlKGtub3duLCAncGF0aWVudDo6JytkZi5wYXRpZW50X2tleSwgJ3JlY29yZDo6JytkZi5yZWNvcmRfa2V5KQogICAga2VwdCA9IGRmLmxvY1t+ZXhjbHVkZWRdLmNvcHkoKQogICAgIyBIYXNoIHRocmVzaG9sZHMga2VlcCBhc3NpZ25tZW50cyBzdGFibGUgaWYgcmVjb3JkcyBhcmUgYXBwZW5kZWQgbGF0ZXIuCiAgICB2YWx1ZXMgPSBrZXB0Lmdyb3VwX2tleS5tYXAobGFtYmRhIGtleTogaW50KGhhc2hsaWIuc2hhMjU2KGYne3NlZWR9OntrZXl9Jy5lbmNvZGUoKSkuaGV4ZGlnZXN0KClbOjE2XSwgMTYpLzIqKjY0KQogICAga2VwdFsnc3NsX3NwbGl0J10gPSBucC53aGVyZSh2YWx1ZXMgPCB2YWxfZnJhY3Rpb24sICdzc2xfdmFsaWRhdGlvbicsICdzc2xfdHJhaW4nKQogICAgdHJhaW4gPSBrZXB0LmxvY1trZXB0LnNzbF9zcGxpdC5lcSgnc3NsX3RyYWluJyldCiAgICB2YWwgPSBrZXB0LmxvY1trZXB0LnNzbF9zcGxpdC5lcSgnc3NsX3ZhbGlkYXRpb24nKV0KICAgIGlmIHRyYWluLmVtcHR5IG9yIHZhbC5lbXB0eToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdTU0wgc3BsaXQgaXMgZW1wdHk7IHVzZSBhIGxhcmdlciBkYXRhc2V0IG9yIGFkanVzdCBzcGxpdCBzZWVkL2ZyYWN0aW9uJykKICAgIGFzc2VydCBub3Qgc2V0KHRyYWluLmdyb3VwX2tleSkgJiBzZXQodmFsLmdyb3VwX2tleSkKICAgIGZvciBwYXRoIGluIGtlcHQuaW1hZ2VfcGF0aDoKICAgICAgICByZXNvbHZlZCA9IChQYXRoKHJvb3QpL3BhdGgpLnJlc29sdmUoKQogICAgICAgIGlmIG5vdCByZXNvbHZlZC5pc19yZWxhdGl2ZV90byhQYXRoKHJvb3QpLnJlc29sdmUoKSkgb3Igbm90IHJlc29sdmVkLmlzX2ZpbGUoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ01pc3Npbmcgb3IgdW5zYWZlIGltYWdlIHBhdGg6IHtwYXRofScpCiAgICBpZiAnaW1hZ2Vfc2hhMjU2JyBpbiBrZXB0OgogICAgICAgIGhhc2hlcyA9IGtlcHQubG9jW2tlcHQuaW1hZ2Vfc2hhMjU2Lm5lKCcnKV0KICAgICAgICBpZiBoYXNoZXMuaW1hZ2Vfc2hhMjU2LmR1cGxpY2F0ZWQoKS5hbnkoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRHVwbGljYXRlIGltYWdlIGhhc2hlczsgcmVzb2x2ZSBiZWZvcmUgcHJldHJhaW5pbmcnKQogICAgYXVkaXQgPSBkaWN0KGluZGV4X3NoYTI1Nj1zaGEyNTYoaW5kZXhfcGF0aCksIGhvbGRvdXRfc2hhMjU2PWhvbGRvdXRfaGFzaCwKICAgICAgICAgICAgICAgICBpbnB1dF9yZWNvcmRzPWxlbihkZiksIGV4Y2x1ZGVkX3JlY29yZHM9aW50KG5wLmFzYXJyYXkoZXhjbHVkZWQpLnN1bSgpKSwKICAgICAgICAgICAgICAgICBzc2xfdHJhaW49bGVuKHRyYWluKSwgc3NsX3ZhbGlkYXRpb249bGVuKHZhbCksCiAgICAgICAgICAgICAgICAgdW5rbm93bl9wYXRpZW50X3JlY29yZHM9aW50KChrZXB0LnBhdGllbnRfa2V5ID09ICcnKS5zdW0oKSksCiAgICAgICAgICAgICAgICAgbGltaXRhdGlvbj0nVW5rbm93biBwYXRpZW50IElEcyBhcmUgZ3JvdXBlZCBieSByZWNvcmQ7IGNyb3NzLXNvdXJjZSBwYXRpZW50IGxpbmthZ2UgaXMgbm90IGVzdGFibGlzaGVkLicpCiAgICBvdXQgPSBQYXRoKG91dCk7IG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0cmFpbi50b19jc3Yob3V0Lydzc2xfdHJhaW4uY3N2JywgaW5kZXg9RmFsc2UpOyB2YWwudG9fY3N2KG91dC8nc3NsX3ZhbGlkYXRpb24uY3N2JywgaW5kZXg9RmFsc2UpCiAgICBkZi5sb2NbZXhjbHVkZWRdLnRvX2NzdihvdXQvJ2V4Y2x1ZGVkX2hvbGRvdXRzLmNzdicsIGluZGV4PUZhbHNlKQogICAgKG91dC8nc3BsaXRfYXVkaXQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhhdWRpdCwgaW5kZW50PTIpKQogICAgcmV0dXJuIHRyYWluLCB2YWwsIGF1ZGl0Cg==', 'rect_fcmae.py': 'IiIiUmVjdGFuZ3VsYXIvcGFkZGluZyBhZGFwdGF0aW9uIG9mIHBpbm5lZCBNZXRhIEZDTUFFLCByZXRhaW5pbmcgc3BhcnNlIGJsb2Nrcy4iIiIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzeXMKaW1wb3J0IHR5cGVzCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5wYXJlbnQvJ3ZlbmRvcicpKQpmcm9tIG1vZGVscy5mY21hZSBpbXBvcnQgRkNNQUUKZnJvbSBNaW5rb3dza2lPcHMgaW1wb3J0IHRvX3NwYXJzZQpmcm9tIGRhdGFfdXRpbHMgaW1wb3J0IHBhdGNoaWZ5LCB1bmlmb3JtX21hc2ssIHJlY29uc3RydWN0aW9uX2xvc3MKCgpkZWYgbWFza2VkX3N0ZW1fZmVhdHVyZXMoZW5jb2RlciwgaW1hZ2VzLCBtYXNrLCB2YWxpZCk6CiAgICAiIiJSZXR1cm4gdGhlIGRlbnNlIHN0ZW0gZ3JpZCBhZnRlciByZW1vdmluZyBtYXNrZWQvaW52YWxpZCBsb2NhdGlvbnMuCgogICAgVGhpcyBpcyB0aGUgZGV0ZXJtaW5pc3RpYyBpc29sYXRpb24gYm91bmRhcnk6IG1hc2tlZCBpbWFnZSBjb250ZW50IG11c3QgYmUKICAgIGdvbmUgYmVmb3JlIE1pbmtvd3NraUVuZ2luZSByZWNlaXZlcyBmZWF0dXJlcyBhbmQgYnVpbGRzIGEgY29vcmRpbmF0ZSBtYXAuCiAgICAiIiIKICAgIG4sIF8sIGgsIHcgPSBpbWFnZXMuc2hhcGUKICAgIGdoLCBndyA9IGgvLzMyLCB3Ly8zMgogICAgeCA9IGVuY29kZXIuZG93bnNhbXBsZV9sYXllcnNbMF0oaW1hZ2VzICogdmFsaWQpCiAgICBhY3RpdmUgPSAoMS1tYXNrLnJlc2hhcGUobiwgMSwgZ2gsIGd3KSkucmVwZWF0X2ludGVybGVhdmUoOCwgMikucmVwZWF0X2ludGVybGVhdmUoOCwgMykKICAgIGFjdGl2ZSA9IGFjdGl2ZSAqIChGLm1heF9wb29sMmQodmFsaWQsIDQsIDQpID4gMCkKICAgIHJldHVybiB4ICogYWN0aXZlCgoKZGVmIHJlY3Rhbmd1bGFyX3NwYXJzZV9mb3J3YXJkKHNlbGYsIGltYWdlcywgbWFzaywgdmFsaWQpOgogICAgbiwgXywgaCwgdyA9IGltYWdlcy5zaGFwZQogICAgZ2gsIGd3ID0gaC8vMzIsIHcvLzMyCiAgICAjIEZpbGwgaW52YWxpZCBwaXhlbHMgd2l0aCB6ZXJvIGluIG5vcm1hbGl6ZWQgaW5wdXQgc3BhY2UuIE5vIHBpeGVsIHZhbHVlcwogICAgIyBmcm9tIG1hc2tlZCBwYXRjaGVzIGNhbiBlbnRlciB0aGUgbm9uLW92ZXJsYXBwaW5nIHN0cmlkZS00IHN0ZW0uCiAgICB4ID0gdG9fc3BhcnNlKG1hc2tlZF9zdGVtX2ZlYXR1cmVzKHNlbGYsIGltYWdlcywgbWFzaywgdmFsaWQpKQogICAgZm9yIGkgaW4gcmFuZ2UoNCk6CiAgICAgICAgeCA9IHNlbGYuZG93bnNhbXBsZV9sYXllcnNbaV0oeCkgaWYgaSBlbHNlIHgKICAgICAgICB4ID0gc2VsZi5zdGFnZXNbaV0oeCkKICAgICMgRXhwbGljaXQgb3JpZ2luL3NoYXBlIHByZXNlcnZlIG1hc2tlZCBvdXRlciByb3dzL2NvbHVtbnMgYW5kIGJhdGNoIGl0ZW1zLgogICAgcmV0dXJuIHguZGVuc2Uoc2hhcGU9dG9yY2guU2l6ZShbbiwgeC5GLnNoYXBlWzFdLCBnaCwgZ3ddKSwKICAgICAgICAgICAgICAgICAgIG1pbl9jb29yZGluYXRlPXRvcmNoLnplcm9zKDIsIGR0eXBlPXRvcmNoLmludDMyKSlbMF0KCgpjbGFzcyBSZWN0RkNNQUUoRkNNQUUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsICoqa3dhcmdzKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKGRlcHRocz1bMywgMywgOSwgM10sIGRpbXM9Wzk2LCAxOTIsIDM4NCwgNzY4XSwKICAgICAgICAgICAgICAgICAgICAgICAgIHBhdGNoX3NpemU9MzIsIG1hc2tfcmF0aW89LjYsIG5vcm1fcGl4X2xvc3M9VHJ1ZSwgKiprd2FyZ3MpCiAgICAgICAgc2VsZi5lbmNvZGVyLmZvcndhcmQgPSB0eXBlcy5NZXRob2RUeXBlKHJlY3Rhbmd1bGFyX3NwYXJzZV9mb3J3YXJkLCBzZWxmLmVuY29kZXIpCgogICAgZGVmIHBhdGNoaWZ5KHNlbGYsIGltYWdlcyk6CiAgICAgICAgcmV0dXJuIHBhdGNoaWZ5KGltYWdlcywgc2VsZi5wYXRjaF9zaXplKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGltYWdlcywgdmFsaWQsIG1hc2s9Tm9uZSwgZ2VuZXJhdG9yPU5vbmUpOgogICAgICAgIGlmIG1hc2sgaXMgTm9uZToKICAgICAgICAgICAgbWFzayA9IHVuaWZvcm1fbWFzayh2YWxpZCwgc2VsZi5tYXNrX3JhdGlvLCBnZW5lcmF0b3IpCiAgICAgICAgZmVhdHVyZXMgPSBzZWxmLmVuY29kZXIoaW1hZ2VzLCBtYXNrLCB2YWxpZCkKICAgICAgICBwcmVkID0gc2VsZi5mb3J3YXJkX2RlY29kZXIoZmVhdHVyZXMsIG1hc2spCiAgICAgICAgbG9zcyA9IHJlY29uc3RydWN0aW9uX2xvc3MoaW1hZ2VzLCBwcmVkLCB2YWxpZCwgbWFzaywgc2VsZi5ub3JtX3BpeF9sb3NzKQogICAgICAgIHJldHVybiBsb3NzLCBwcmVkLCBtYXNrCgoKZnJvbSBjaGVja3BvaW50X3V0aWxzIGltcG9ydCBleHBvcnRfZW5jb2RlciwgbG9hZF9pbml0aWFsCg==', 'checkpoint_utils.py': 'IiIiU3RyaWN0IG9mZmljaWFsIEZDTUFFIGVuY29kZXIgaW5pdGlhbGl6YXRpb24gYW5kIHNwYXJzZS9kZW5zZSB0ZW5zb3IgY29udmVyc2lvbi4iIiIKaW1wb3J0IHRvcmNoCgpkZWYgc3BhcnNlX3RvX2RlbnNlX2tleShrZXksIHZhbHVlKToKICAgICIiIk1ldGEvTWlua293c2tpIGxheW91dCB0byB0aGUgb2ZmaWNpYWwgZGVuc2UgQ29udk5lWHQtVjIgbGF5b3V0LgoKICAgIFRoZSBzcGF0aWFsIHRyYW5zcG9zZSBmb2xsb3dzIE1pbmtvd3NraUVuZ2luZSdzIGtlcm5lbC1vZmZzZXQgb3JkZXJpbmcuCiAgICBHUFUgcHJlZmxpZ2h0IGNoZWNrcyBhbGwtdmlzaWJsZSBzcGFyc2UvZGVuc2UgZmVhdHVyZSBwYXJpdHkgd2l0aCBiYXRjaD0xLgogICAgIiIiCiAgICBrZXkgPSBrZXkucmVtb3ZlcHJlZml4KCdlbmNvZGVyLicpCiAgICBpZiBrZXkuZW5kc3dpdGgoJy5rZXJuZWwnKToKICAgICAgICBrZXkgPSBrZXlbOi1sZW4oJ2tlcm5lbCcpXSsnd2VpZ2h0JwogICAgICAgIGlmIHZhbHVlLm5kaW0gPT0gMzogICMga2VybmVsLXZvbHVtZSwgaW4sIG91dAogICAgICAgICAgICBwID0gaW50KHZhbHVlLnNoYXBlWzBdKiouNSkKICAgICAgICAgICAgdmFsdWUgPSB2YWx1ZS5wZXJtdXRlKDIsIDEsIDApLnJlc2hhcGUodmFsdWUuc2hhcGVbMl0sIHZhbHVlLnNoYXBlWzFdLCBwLCBwKS50cmFuc3Bvc2UoLTEsIC0yKQogICAgICAgIGVsaWYgdmFsdWUubmRpbSA9PSAyOiAgIyBkZXB0aHdpc2U6IGtlcm5lbC12b2x1bWUsIGNoYW5uZWxzCiAgICAgICAgICAgIHAgPSBpbnQodmFsdWUuc2hhcGVbMF0qKi41KQogICAgICAgICAgICB2YWx1ZSA9IHZhbHVlLlQucmVzaGFwZSh2YWx1ZS5zaGFwZVsxXSwgMSwgcCwgcCkudHJhbnNwb3NlKC0xLCAtMikKICAgICAgICBlbHNlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnVW5leHBlY3RlZCBrZXJuZWwgc2hhcGU6IHtrZXl9IHt2YWx1ZS5zaGFwZX0nKQogICAga2V5ID0ga2V5LnJlcGxhY2UoJy5sbi4nLCAnLicpLnJlcGxhY2UoJy5saW5lYXIuJywgJy4nKQogICAgaWYgJy5ncm4uJyBpbiBrZXk6CiAgICAgICAgdmFsdWUgPSB2YWx1ZS5yZXNoYXBlKDEsIDEsIDEsIC0xKQogICAgaWYga2V5LmVuZHN3aXRoKCcuYmlhcycpOgogICAgICAgIHZhbHVlID0gdmFsdWUucmVzaGFwZSgtMSkKICAgIHJldHVybiBrZXksIHZhbHVlLmNvbnRpZ3VvdXMoKQoKCmRlZiBleHBvcnRfZW5jb2Rlcihtb2RlbCk6CiAgICByZXR1cm4gZGljdChzcGFyc2VfdG9fZGVuc2Vfa2V5KGssIHYuZGV0YWNoKCkuY3B1KCkpIGZvciBrLCB2IGluIG1vZGVsLmVuY29kZXIuc3RhdGVfZGljdCgpLml0ZW1zKCkpCgoKZGVmIGxvYWRfaW5pdGlhbChtb2RlbCwgcGF0aCk6CiAgICAjIFJlc3RyaWN0ZWQgdW5waWNrbGVyOiB0aGUgb2ZmaWNpYWwgZmlsZSBpcyBhIHRlbnNvciBtb2RlbCBkaWN0aW9uYXJ5LgogICAgY2hlY2twb2ludCA9IHRvcmNoLmxvYWQocGF0aCwgbWFwX2xvY2F0aW9uPSdjcHUnLCB3ZWlnaHRzX29ubHk9VHJ1ZSkKICAgIHN0YXRlID0gY2hlY2twb2ludC5nZXQoJ21vZGVsJywgY2hlY2twb2ludCkKICAgICMgU3VwcG9ydCByZWxlYXNlZCBkZW5zZS9KQVgtY29udmVydGVkIHRlbnNvcnMgb3IgbmF0aXZlIHNwYXJzZSB0cmFpbmluZyBrZXlzLgogICAgIyBUaGUgcmVsZWFzZWQgcHRfb25seSBjaGVja3BvaW50IGNvbnRhaW5zIDE5NiBFTkNPREVSIHRlbnNvcnMsIG5vIGRlY29kZXIuCiAgICAjIFRoZSBGQ01BRSByZWNvbnN0cnVjdGlvbiBoZWFkIHJlbWFpbnMgZXhwbGljaXRseSByYW5kb21seSBpbml0aWFsaXplZC4KICAgIGV4cGVjdGVkID0geydlbmNvZGVyLicrazogdiBmb3IgaywgdiBpbiBtb2RlbC5lbmNvZGVyLnN0YXRlX2RpY3QoKS5pdGVtcygpfQogICAgbG9hZGVkID0ge30KICAgIHVzZWQgPSBzZXQoKQogICAgZm9yIGtleSwgdGFyZ2V0IGluIGV4cGVjdGVkLml0ZW1zKCk6CiAgICAgICAgaWYga2V5IGluIHN0YXRlIGFuZCBzdGF0ZVtrZXldLnNoYXBlID09IHRhcmdldC5zaGFwZToKICAgICAgICAgICAgbG9hZGVkW2tleV0gPSBzdGF0ZVtrZXldOyB1c2VkLmFkZChrZXkpOyBjb250aW51ZQogICAgICAgIGRlbnNlX2tleSwgXyA9IHNwYXJzZV90b19kZW5zZV9rZXkoa2V5LCB0YXJnZXQpIGlmIGtleS5zdGFydHN3aXRoKCdlbmNvZGVyLicpIGVsc2UgKGtleSwgdGFyZ2V0KQogICAgICAgIGNhbmRpZGF0ZXMgPSBbJ2VuY29kZXIuJytkZW5zZV9rZXksIGRlbnNlX2tleV0gaWYga2V5LnN0YXJ0c3dpdGgoJ2VuY29kZXIuJykgZWxzZSBbZGVuc2Vfa2V5XQogICAgICAgIG1hdGNoID0gbmV4dCgoayBmb3IgayBpbiBjYW5kaWRhdGVzIGlmIGsgaW4gc3RhdGUpLCBOb25lKQogICAgICAgIGlmIG1hdGNoIGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZidDaGVja3BvaW50IGlzIG1pc3Npbmcge2tleX07IHJlZnVzaW5nIHBhcnRpYWwgaW5pdGlhbGl6YXRpb24nKQogICAgICAgIHZhbHVlID0gc3RhdGVbbWF0Y2hdCiAgICAgICAgaWYga2V5LmVuZHN3aXRoKCcua2VybmVsJyk6CiAgICAgICAgICAgIHZhbHVlID0gdmFsdWUudHJhbnNwb3NlKC0xLCAtMikKICAgICAgICAgICAgaWYgdGFyZ2V0Lm5kaW0gPT0gMzoKICAgICAgICAgICAgICAgIHZhbHVlID0gdmFsdWUucmVzaGFwZSh2YWx1ZS5zaGFwZVswXSwgdmFsdWUuc2hhcGVbMV0sIC0xKS5wZXJtdXRlKDIsIDEsIDApCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICB2YWx1ZSA9IHZhbHVlLnJlc2hhcGUodmFsdWUuc2hhcGVbMF0sIC0xKS5UCiAgICAgICAgaWYgdmFsdWUubnVtZWwoKSAhPSB0YXJnZXQubnVtZWwoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ0NoZWNrcG9pbnQgc2hhcGUgbWlzbWF0Y2g6IHtrZXl9JykKICAgICAgICBsb2FkZWRba2V5XSA9IHZhbHVlLnJlc2hhcGVfYXModGFyZ2V0KS5jb250aWd1b3VzKCk7IHVzZWQuYWRkKG1hdGNoKQogICAgdW5leHBlY3RlZCA9IHNldChzdGF0ZSktdXNlZAogICAgaWYgdW5leHBlY3RlZDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnVW5leHBlY3RlZCBjaGVja3BvaW50IGtleXM6IHtzb3J0ZWQodW5leHBlY3RlZClbOjEwXX0nKQogICAgbW9kZWwuZW5jb2Rlci5sb2FkX3N0YXRlX2RpY3Qoe2sucmVtb3ZlcHJlZml4KCdlbmNvZGVyLicpOiB2IGZvciBrLCB2IGluIGxvYWRlZC5pdGVtcygpfSwgc3RyaWN0PVRydWUpCiAgICByZXR1cm4geydsb2FkZWRfZW5jb2Rlcl90ZW5zb3JzJzogbGVuKGxvYWRlZCksICdtaXNzaW5nX2VuY29kZXInOiBbXSwgJ3VuZXhwZWN0ZWQnOiBbXSwKICAgICAgICAgICAgJ2RlY29kZXJfaW5pdGlhbGl6YXRpb24nOiAncmFuZG9tOyBvZmZpY2lhbCBwdF9vbmx5IHJlbGVhc2UgY29udGFpbnMgZW5jb2RlciBvbmx5J30K', 'train.py': 'IiIiU2luZ2xlLUdQVSBjb250aW51ZWQgRkNNQUUgdHJhaW5pbmcsIGludm9rZWQgYnkgdGhlIENvbGFiIG5vdGVib29rLiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgcmFuZG9tCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdGltZQoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlcgpmcm9tIHRyYWluaW5nX3N0YXRlIGltcG9ydCAoc2F2ZV9hdG9taWMsIGxvYWRfcmVzdW1lLCByZXN0b3JlX3JuZywgZXBvY2hfb3JkZXIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBSZXN1bWVCYXRjaFNhbXBsZXIsIHNuYXBzaG90LCB0cmFpbl9lcG9jaCkKCmZyb20gZGF0YV91dGlscyBpbXBvcnQgKEVDR0RhdGFzZXQsIE1FQU4sIFNURCwgcHJlcGFyZV9zcGxpdHMsIHNoYTI1NiwgdGFyZ2V0cywKICAgICAgICAgICAgICAgICAgICAgICAgdW5pZm9ybV9tYXNrLCB1bnBhdGNoaWZ5KQpmcm9tIHJlY3RfZmNtYWUgaW1wb3J0IFJlY3RGQ01BRSwgZXhwb3J0X2VuY29kZXIsIGxvYWRfaW5pdGlhbCwgbWFza2VkX3N0ZW1fZmVhdHVyZXMKZnJvbSBtb2RlbHMuY29udm5leHR2MiBpbXBvcnQgY29udm5leHR2Ml90aW55CgoKZGVmIHZpc3VhbGl6ZShtb2RlbCwgZGF0YXNldCwgb3V0LCBzZWVkLCBkZXZpY2UpOgogICAgaW1wb3J0IG1hdHBsb3RsaWIKICAgIG1hdHBsb3RsaWIudXNlKCdBZ2cnKQogICAgaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAogICAgaW1hZ2UsIHZhbGlkLCByZWNvcmQgPSBkYXRhc2V0WzBdCiAgICBpbWFnZSwgdmFsaWQgPSBpbWFnZVtOb25lXS50byhkZXZpY2UpLCB2YWxpZFtOb25lXS50byhkZXZpY2UpCiAgICBtb2RlbC5ldmFsKCkKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIG1hc2sgPSB1bmlmb3JtX21hc2sodmFsaWQsIGdlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoZGV2aWNlPWRldmljZSkubWFudWFsX3NlZWQoc2VlZCkpCiAgICAgICAgXywgcHJlZCwgXyA9IG1vZGVsKGltYWdlLCB2YWxpZCwgbWFzaykKICAgICAgICBfLCBfLCBtZWFuLCB2YXIgPSB0YXJnZXRzKGltYWdlLCB2YWxpZCkKICAgICAgICBwcmVkID0gcHJlZC5mbG9hdCgpLmZsYXR0ZW4oMikudHJhbnNwb3NlKDEsIDIpCiAgICAgICAgIyBEaXNwbGF5IG9ubHk6IHVubm9ybWFsaXplIHVzaW5nIHRhcmdldCBwYXRjaCBzdGF0aXN0aWNzLiBUaGlzIGlzIG5vdAogICAgICAgICMgYXV0b25vbW91cyByZWNvdmVyeSBvZiB0aGUgdW5rbm93biBwYXRjaCdzIGJyaWdodG5lc3MvY29udHJhc3QuCiAgICAgICAgcHJlZCA9IHByZWQgKiAodmFyKzFlLTYpLnNxcnQoKSArIG1lYW4KICAgICAgICBncmlkID0gKGltYWdlLnNoYXBlWy0yXS8vMzIsIGltYWdlLnNoYXBlWy0xXS8vMzIpCiAgICAgICAgcmVjb24gPSB1bnBhdGNoaWZ5KHByZWQsIGdyaWQpCiAgICAgICAgcGl4ZWxzID0gbWFzay5yZXNoYXBlKDEsIDEsICpncmlkKS5yZXBlYXRfaW50ZXJsZWF2ZSgzMiwgMikucmVwZWF0X2ludGVybGVhdmUoMzIsIDMpCiAgICAgICAgbXUgPSB0b3JjaC50ZW5zb3IoTUVBTiwgZGV2aWNlPWRldmljZSlbTm9uZSwgOiwgTm9uZSwgTm9uZV0KICAgICAgICBzZCA9IHRvcmNoLnRlbnNvcihTVEQsIGRldmljZT1kZXZpY2UpW05vbmUsIDosIE5vbmUsIE5vbmVdCiAgICAgICAgb3JpZ2luYWwgPSAoaW1hZ2Uqc2QrbXUpLmNsYW1wKDAsIDEpCiAgICAgICAgZGVjb2RlZCA9IChyZWNvbipzZCttdSkuY2xhbXAoMCwgMSkKICAgICAgICBjb21wb3NpdGUgPSBvcmlnaW5hbCooMS1waXhlbHMpK2RlY29kZWQqcGl4ZWxzCiAgICAgICAgcGFuZWxzID0gW29yaWdpbmFsLCBvcmlnaW5hbCooMS1waXhlbHMpKy41KnBpeGVscywgY29tcG9zaXRlLCB2YWxpZC5leHBhbmQoLTEsMywtMSwtMSldCiAgICAgICAgZmlnLCBheGVzID0gcGx0LnN1YnBsb3RzKDEsIDQsIGZpZ3NpemU9KDIwLCA1KSkKICAgICAgICBmb3IgYXgsIHBhbmVsLCB0aXRsZSBpbiB6aXAoYXhlcywgcGFuZWxzLCBbJ09yaWdpbmFsJywgJ01hc2tlZCBpbnB1dCcsICdSZWNvbnN0cnVjdGlvbiAodGFyZ2V0IHN0YXRzKScsICdWYWxpZCBwaXhlbHMnXSk6CiAgICAgICAgICAgIGF4Lmltc2hvdyhwYW5lbFswXS5jcHUoKS5wZXJtdXRlKDEsIDIsIDApKTsgYXguc2V0X3RpdGxlKHRpdGxlKTsgYXguYXhpcygnb2ZmJykKICAgICAgICBmaWcuc3VwdGl0bGUocmVjb3JkKTsgZmlnLnRpZ2h0X2xheW91dCgpOyBmaWcuc2F2ZWZpZyhvdXQsIGRwaT0xNDApOyBwbHQuY2xvc2UoZmlnKQoKCmRlZiBwcmVmbGlnaHQobW9kZWwsIGRldmljZSk6CiAgICAiIiJBY3R1YWwgc3BhcnNlIGV4ZWN1dGlvbjogc2hhcGUsIGhpZGRlbi1jb250ZW50IGlzb2xhdGlvbiwgZXhwb3J0IHBhcml0eS4iIiIKICAgIG1vZGVsLmV2YWwoKQogICAgaW1hZ2UgPSB0b3JjaC5yYW5kbigxLCAzLCA5NiwgMTYwLCBkZXZpY2U9ZGV2aWNlKQogICAgdmFsaWQgPSB0b3JjaC5vbmVzKDEsIDEsIDk2LCAxNjAsIGRldmljZT1kZXZpY2UpCiAgICBtYXNrID0gdW5pZm9ybV9tYXNrKHZhbGlkLCBnZW5lcmF0b3I9dG9yY2guR2VuZXJhdG9yKGRldmljZT1kZXZpY2UpLm1hbnVhbF9zZWVkKDgpKQogICAgIyBGb3JjZSBvdXRlciBjZWxscyBtYXNrZWQgdG8gY2F0Y2ggYWNjaWRlbnRhbCBzcGFyc2UgYm91bmRpbmctYm94IGNyb3BwaW5nLgogICAgbWFza1s6LCAtNTpdID0gMQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgbG9zcywgcHJlZCwgXyA9IG1vZGVsKGltYWdlLCB2YWxpZCwgbWFzaykKICAgICAgICBhc3NlcnQgcHJlZC5zaGFwZSA9PSAoMSwgMzA3MiwgMywgNSkgYW5kIHRvcmNoLmlzZmluaXRlKGxvc3MpCiAgICAgICAgcGl4ZWxzID0gbWFzay5yZXNoYXBlKDEsIDEsIDMsIDUpLnJlcGVhdF9pbnRlcmxlYXZlKDMyLCAyKS5yZXBlYXRfaW50ZXJsZWF2ZSgzMiwgMykKICAgICAgICBhbHRlcmVkID0gaW1hZ2UgKyBwaXhlbHMqdG9yY2gucmFuZG5fbGlrZShpbWFnZSkqMTAKICAgICAgICAjIENvbXBhcmUgYXQgdGhlIGRldGVybWluaXN0aWMgYm91bmRhcnkgaW1tZWRpYXRlbHkgYmVmb3JlIGNvbnZlcnNpb24KICAgICAgICAjIHRvIGEgc3BhcnNlIHRlbnNvci4gTWlua293c2tpRW5naW5lIDAuNS40IENVREEgY29udm9sdXRpb25zIHVzZSBhdG9taWMKICAgICAgICAjIGFjY3VtdWxhdGlvbiBhbmQgY2FuIHZhcnkgc2xpZ2h0bHkgYmV0d2VlbiBvdGhlcndpc2UgaWRlbnRpY2FsIGZ1bGwKICAgICAgICAjIGZvcndhcmRzLCBzbyBkZWNvZGVyLW91dHB1dCBlcXVhbGl0eSBpcyBub3QgYSB2YWxpZCBsZWFrYWdlIHRlc3QuCiAgICAgICAgc3RlbSA9IG1hc2tlZF9zdGVtX2ZlYXR1cmVzKG1vZGVsLmVuY29kZXIsIGltYWdlLCBtYXNrLCB2YWxpZCkKICAgICAgICBjaGFuZ2VkX3N0ZW0gPSBtYXNrZWRfc3RlbV9mZWF0dXJlcyhtb2RlbC5lbmNvZGVyLCBhbHRlcmVkLCBtYXNrLCB2YWxpZCkKICAgICAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShzdGVtLCBjaGFuZ2VkX3N0ZW0sIHJ0b2w9MCwgYXRvbD0wKQogICAgICAgIGRlbnNlID0gY29udm5leHR2Ml90aW55KCkudG8oZGV2aWNlKS5ldmFsKCkKICAgICAgICBtaXNzaW5nLCB1bmV4cGVjdGVkID0gZGVuc2UubG9hZF9zdGF0ZV9kaWN0KGV4cG9ydF9lbmNvZGVyKG1vZGVsKSwgc3RyaWN0PUZhbHNlKQogICAgICAgIGFzc2VydCBzZXQobWlzc2luZykgPT0geydub3JtLndlaWdodCcsJ25vcm0uYmlhcycsJ2hlYWQud2VpZ2h0JywnaGVhZC5iaWFzJ30gYW5kIG5vdCB1bmV4cGVjdGVkCiAgICAgICAgc3BhcnNlX2ZlYXR1cmUgPSBtb2RlbC5lbmNvZGVyKGltYWdlLCB0b3JjaC56ZXJvc19saWtlKG1hc2spLCB2YWxpZCkKICAgICAgICBkZW5zZV9mZWF0dXJlID0gaW1hZ2UKICAgICAgICBmb3IgaSBpbiByYW5nZSg0KToKICAgICAgICAgICAgZGVuc2VfZmVhdHVyZSA9IGRlbnNlLnN0YWdlc1tpXShkZW5zZS5kb3duc2FtcGxlX2xheWVyc1tpXShkZW5zZV9mZWF0dXJlKSkKICAgICAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShzcGFyc2VfZmVhdHVyZSwgZGVuc2VfZmVhdHVyZSwgcnRvbD0yZS0zLCBhdG9sPTJlLTMpCiAgICBkZWwgZGVuc2UKICAgIG1vZGVsLnRyYWluKCkKICAgIGxvc3MsIF8sIF8gPSBtb2RlbChpbWFnZSwgdmFsaWQsIG1hc2spCiAgICBsb3NzLmJhY2t3YXJkKCkKICAgIGdyYWRzID0gW3AuZ3JhZCBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcC5ncmFkIGlzIG5vdCBOb25lXQogICAgYXNzZXJ0IGdyYWRzIGFuZCBhbGwodG9yY2guaXNmaW5pdGUoZykuYWxsKCkgZm9yIGcgaW4gZ3JhZHMpCiAgICBtb2RlbC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKCgpkZWYgbWFpbihjZmcsIHBpbG90PUZhbHNlKToKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQSBDVURBIEdQVSBhbmQgdGhlIHBpbm5lZCBNaW5rb3dza2lFbmdpbmUgZW52aXJvbm1lbnQgYXJlIHJlcXVpcmVkJykKICAgIGRldmljZSA9ICdjdWRhJwogICAgc2VlZCA9IGNmZ1snc2VlZCddCiAgICByYW5kb20uc2VlZChzZWVkKTsgbnAucmFuZG9tLnNlZWQoc2VlZCk7IHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpOyB0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChzZWVkKQogICAgdG9yY2guYmFja2VuZHMuY3Vkbm4uYmVuY2htYXJrID0gRmFsc2UKICAgIHRvcmNoLmJhY2tlbmRzLmN1ZGEubWF0bXVsLmFsbG93X3RmMzIgPSBGYWxzZQogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDQpCiAgICBvdXQgPSBQYXRoKGNmZ1sncnVuX2RpciddKSAvICgncGlsb3QnIGlmIHBpbG90IGVsc2UgJ2Z1bGwnKQogICAgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIGNoZWNrcG9pbnRzLCBlbmNvZGVycywgcHJldmlld3MgPSBvdXQvJ2NoZWNrcG9pbnRzJywgb3V0LydlbmNvZGVycycsIG91dC8ncHJldmlld3MnCiAgICBmb3IgZGlyZWN0b3J5IGluIChjaGVja3BvaW50cywgZW5jb2RlcnMsIHByZXZpZXdzKToKICAgICAgICBkaXJlY3RvcnkubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgc3BsaXRfc3RhZ2UgPSBQYXRoKHRlbXBmaWxlLm1rZHRlbXAocHJlZml4PSdzcGxpdF9hdWRpdF8nLCBkaXI9b3V0KSkKICAgIHRyYWluX2ZyYW1lLCB2YWxfZnJhbWUsIGF1ZGl0ID0gcHJlcGFyZV9zcGxpdHMoY2ZnWydpbmRleCddLCBjZmdbJ2ltYWdlX3Jvb3QnXSwgc3BsaXRfc3RhZ2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VlZCwgY2ZnLmdldCgnaG9sZG91dF9jc3YnKSwgY2ZnWydzc2xfdmFsX2ZyYWN0aW9uJ10pCiAgICBpZiBub3QgcGlsb3QgYW5kIG5vdCBhdWRpdFsnaG9sZG91dF9zaGEyNTYnXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdGdWxsIHJ1biByZXF1aXJlcyBkb3duc3RyZWFtIHZhbGlkYXRpb24vdGVzdCBleGNsdXNpb24gQ1NWOyBwaWxvdCByZW1haW5zIGF2YWlsYWJsZScpCiAgICBpZiBwaWxvdDoKICAgICAgICB0cmFpbl9mcmFtZSA9IHRyYWluX2ZyYW1lLmlsb2NbOm1heChjZmdbJ2JhdGNoX3NpemUnXSpjZmdbJ2FjY3VtdWxhdGlvbiddKjIsIDgpXQogICAgICAgIHZhbF9mcmFtZSA9IHZhbF9mcmFtZS5pbG9jWzo0XQogICAgdHJhaW5fZHMgPSBFQ0dEYXRhc2V0KHRyYWluX2ZyYW1lLCBjZmdbJ2ltYWdlX3Jvb3QnXSwgY2ZnWydpbnB1dF9zaXplJ10sIGNmZy5nZXQoJ3Jlc2l6ZV9jYWNoZScpKQogICAgdmFsX2RzID0gRUNHRGF0YXNldCh2YWxfZnJhbWUsIGNmZ1snaW1hZ2Vfcm9vdCddLCBjZmdbJ2lucHV0X3NpemUnXSwgY2ZnLmdldCgncmVzaXplX2NhY2hlJykpCiAgICAjIERlZGljYXRlZCBsb2FkZXIgZ2VuZXJhdG9yIHByZXZlbnRzIGl0ZXJhdG9yIGNvbnN0cnVjdGlvbiBvbiByZXN1bWUgZnJvbQogICAgIyBjb25zdW1pbmcgdGhlIFJORyBzdHJlYW0gdXNlZCBmb3IgbWFza3MvbW9kZWwgY29tcHV0YXRpb25zLgogICAgdmFsX2xvYWRlciA9IERhdGFMb2FkZXIodmFsX2RzLCBiYXRjaF9zaXplPWNmZ1snYmF0Y2hfc2l6ZSddLCBzaHVmZmxlPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgbnVtX3dvcmtlcnM9Y2ZnWyd3b3JrZXJzJ10sIHBpbl9tZW1vcnk9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGdlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChzZWVkKzUwMDAwMCkpCiAgICBtb2RlbCA9IFJlY3RGQ01BRSgpLnRvKGRldmljZSkKICAgIHJlcG9ydCA9IGxvYWRfaW5pdGlhbChtb2RlbCwgY2ZnWydjaGVja3BvaW50J10pCiAgICBwcmludCgnSW5pdGlhbGl6YXRpb246JywgcmVwb3J0LCBmbHVzaD1UcnVlKQogICAgcHJlZmxpZ2h0KG1vZGVsLCBkZXZpY2UpCiAgICBwcmludCgnU3BhcnNlIHByZWZsaWdodCBwYXNzZWQ6IHJlY3Rhbmd1bGFyIHNoYXBlLCBoaWRkZW4tY29udGVudCBpc29sYXRpb24sIGV4cG9ydCBwYXJpdHksIGdyYWRpZW50cycsIGZsdXNoPVRydWUpCiAgICBja3B0X2hhc2ggPSBzaGEyNTYoY2ZnWydjaGVja3BvaW50J10pCiAgICBzb3VyY2VfaGFzaGVzID0ge3N0cihwLnJlbGF0aXZlX3RvKFBhdGgoX19maWxlX18pLnBhcmVudCkpOiBzaGEyNTYocCkKICAgICAgICAgICAgICAgICAgICAgZm9yIHAgaW4gUGF0aChfX2ZpbGVfXykucGFyZW50LnJnbG9iKCcqLnB5Jyl9CiAgICBpZGVudGl0eSA9IGRpY3QoY29uZmlnPWNmZywgc3BsaXQ9YXVkaXQsIGNoZWNrcG9pbnRfc2hhMjU2PWNrcHRfaGFzaCwgc291cmNlcz1zb3VyY2VfaGFzaGVzKQogICAgZmluZ2VycHJpbnQgPSBoYXNobGliLnNoYTI1Nihqc29uLmR1bXBzKGlkZW50aXR5LCBzb3J0X2tleXM9VHJ1ZSkuZW5jb2RlKCkpLmhleGRpZ2VzdCgpCiAgICBwcm92ZW5hbmNlX3BhdGggPSBvdXQvJ3Byb3ZlbmFuY2UuanNvbicKICAgIGlmIHByb3ZlbmFuY2VfcGF0aC5leGlzdHMoKSBhbmQganNvbi5sb2Fkcyhwcm92ZW5hbmNlX3BhdGgucmVhZF90ZXh0KCkpWydmaW5nZXJwcmludCddICE9IGZpbmdlcnByaW50OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0V4aXN0aW5nIHJ1biBjb25maWd1cmF0aW9uL2RhdGEvc291cmNlIG1pc21hdGNoOiB1c2UgYSBuZXcgcnVuX2RpcicpCiAgICBmb3IgcGF0aCBpbiBzcGxpdF9zdGFnZS5pdGVyZGlyKCk6CiAgICAgICAgb3MucmVwbGFjZShwYXRoLCBvdXQvcGF0aC5uYW1lKQogICAgc3BsaXRfc3RhZ2Uucm1kaXIoKQogICAgKG91dC8ncHJvdmVuYW5jZS5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKGRpY3QoKippZGVudGl0eSwgZmluZ2VycHJpbnQ9ZmluZ2VycHJpbnQsCiAgICAgICAgdG9yY2g9dG9yY2guX192ZXJzaW9uX18sIGN1ZGE9dG9yY2gudmVyc2lvbi5jdWRhLCBncHU9dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoKSwKICAgICAgICBpbml0aWFsaXphdGlvbj1yZXBvcnQpLCBpbmRlbnQ9MikpCiAgICAjIE1hdGNoZWQgZG93bnN0cmVhbSBjb250cm9sOiBleGFjdGx5IHRoZSBlbmNvZGVyIGJlZm9yZSBFQ0cgdXBkYXRlcy4KICAgIHNhdmVfYXRvbWljKHsnbW9kZWwnOiBleHBvcnRfZW5jb2Rlcihtb2RlbCksICdraW5kJzogJ0ltYWdlTmV0LTFLIEZDTUFFIGNvbnRyb2wnLAogICAgICAgICAgICAgICAgICdpbnB1dF9zaXplJzogY2ZnWydpbnB1dF9zaXplJ10sICdjaGVja3BvaW50X3NoYTI1Nic6IGNrcHRfaGFzaH0sIGVuY29kZXJzLydlbmNvZGVyX2luaXRpYWwucHQnKQogICAgZGVjYXksIG5vX2RlY2F5ID0gW10sIFtdCiAgICBmb3IgbmFtZSwgcGFyYW1ldGVyIGluIG1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKToKICAgICAgICAobm9fZGVjYXkgaWYgcGFyYW1ldGVyLm5kaW0gPD0gMSBvciBuYW1lLmVuZHN3aXRoKCdiaWFzJykgb3IgJ2dybicgaW4gbmFtZSBlbHNlIGRlY2F5KS5hcHBlbmQocGFyYW1ldGVyKQogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbVcoW3sncGFyYW1zJzogZGVjYXksICd3ZWlnaHRfZGVjYXknOiAuMDV9LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHsncGFyYW1zJzogbm9fZGVjYXksICd3ZWlnaHRfZGVjYXknOiAwLn1dLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbHI9Y2ZnWydsZWFybmluZ19yYXRlJ10sIGJldGFzPSguOSwgLjk1KSkKICAgICMgRlAxNiBpcyBhdmFpbGFibGUgb24gVDQvTDQvQTEwMC4gQWN0dWFsIHNwYXJzZSBBTVAgc3VwcG9ydCBpcyBleGVyY2lzZWQKICAgICMgYnkgdGhlIHBpbG90OyBubyBhdXRvbWF0aWMgY2hhbmdlIG9mIHByZWNpc2lvbiBvciBiYXRjaCBzaXplIG9uIGZhaWx1cmUuCiAgICBhbXAgPSBjZmdbJ2FtcCddCiAgICBzY2FsZXIgPSB0b3JjaC5jdWRhLmFtcC5HcmFkU2NhbGVyKGVuYWJsZWQ9YW1wKQogICAgZXBvY2hzID0gMSBpZiBwaWxvdCBlbHNlIGNmZ1snZXBvY2hzJ10KICAgIHRvdGFsX2JhdGNoZXMgPSBtYXRoLmNlaWwobGVuKHRyYWluX2RzKS9jZmdbJ2JhdGNoX3NpemUnXSkKICAgIHVwZGF0ZXNfcGVyX2Vwb2NoID0gbWF0aC5jZWlsKHRvdGFsX2JhdGNoZXMvY2ZnWydhY2N1bXVsYXRpb24nXSkKICAgIHRvdGFsX3N0ZXBzID0gZXBvY2hzKnVwZGF0ZXNfcGVyX2Vwb2NoCiAgICB3YXJtdXAgPSBtaW4odG90YWxfc3RlcHMtMSwgKDAgaWYgcGlsb3QgZWxzZSBjZmdbJ3dhcm11cF9lcG9jaHMnXSkqdXBkYXRlc19wZXJfZXBvY2gpCiAgICBzdGFydCwgc3RlcCwgYmVzdCwgaGlzdG9yeSA9IDAsIDAsIGZsb2F0KCdpbmYnKSwgW10KICAgIHJlc3VtZV9wYXRoID0gY2hlY2twb2ludHMvJ2xhc3QucHQnCiAgICByZXN1bWVkID0gbG9hZF9yZXN1bWUocmVzdW1lX3BhdGgsIGZpbmdlcnByaW50KQogICAgaWYgcmVzdW1lZCBpcyBub3QgTm9uZToKICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QocmVzdW1lZFsnbW9kZWwnXSwgc3RyaWN0PVRydWUpCiAgICAgICAgb3B0aW1pemVyLmxvYWRfc3RhdGVfZGljdChyZXN1bWVkWydvcHRpbWl6ZXInXSk7IHNjYWxlci5sb2FkX3N0YXRlX2RpY3QocmVzdW1lZFsnc2NhbGVyJ10pCiAgICAgICAgc3RhcnQgPSByZXN1bWVkWydlcG9jaCddICsgaW50KHJlc3VtZWRbJ2Vwb2NoX2NvbXBsZXRlJ10pCiAgICAgICAgc3RlcCwgYmVzdCwgaGlzdG9yeSA9IHJlc3VtZWRbJ3Byb2dyZXNzJ11bJ3N0ZXAnXSwgcmVzdW1lZFsnYmVzdCddLCByZXN1bWVkWydoaXN0b3J5J10KICAgICAgICByZXN0b3JlX3JuZyhyZXN1bWVkWydybmcnXSkKICAgICAgICBwcmludChmIlJlc3VtaW5nIGF0IGVwb2NoIHtzdGFydCsxfSwgbmV4dCB0cmFpbmluZyBiYXRjaCAiCiAgICAgICAgICAgICAgZiJ7MSBpZiByZXN1bWVkWydlcG9jaF9jb21wbGV0ZSddIGVsc2UgcmVzdW1lZFsncHJvZ3Jlc3MnXVsnbmV4dF9iYXRjaCddKzF9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICAjIERvIG5vdCBrZWVwIGEgc2Vjb25kIGZ1bGwgQ1BVIG1vZGVsL29wdGltaXplciBjaGVja3BvaW50IGluIFJBTS4KICAgICAgICByZXN1bWVkID0ge2s6IHJlc3VtZWRba10gZm9yIGsgaW4gWydlcG9jaF9jb21wbGV0ZScsICdvcmRlcicsICdwcm9ncmVzcyddfQogICAgZm9yIGVwb2NoIGluIHJhbmdlKHN0YXJ0LCBlcG9jaHMpOgogICAgICAgIHBhcnRpYWwgPSByZXN1bWVkIGlzIG5vdCBOb25lIGFuZCBub3QgcmVzdW1lZFsnZXBvY2hfY29tcGxldGUnXSBhbmQgZXBvY2ggPT0gc3RhcnQKICAgICAgICBvcmRlciA9IHJlc3VtZWRbJ29yZGVyJ10gaWYgcGFydGlhbCBlbHNlIGVwb2NoX29yZGVyKGxlbih0cmFpbl9kcyksIHNlZWQsIGVwb2NoKQogICAgICAgIHByb2dyZXNzID0gZGljdChyZXN1bWVkWydwcm9ncmVzcyddKSBpZiBwYXJ0aWFsIGVsc2UgZGljdCgKICAgICAgICAgICAgbmV4dF9iYXRjaD0wLCBzdGVwPXN0ZXAsIGxvc3Nfc3VtPTAuLCBzZWVuPTAsIHNlY29uZHM9MC4sIHBlYWtfZ3B1X2dpYj0wLikKICAgICAgICBzYW1wbGVyID0gUmVzdW1lQmF0Y2hTYW1wbGVyKG9yZGVyLCBjZmdbJ2JhdGNoX3NpemUnXSwgcHJvZ3Jlc3NbJ25leHRfYmF0Y2gnXSwgY2ZnWydhY2N1bXVsYXRpb24nXSkKICAgICAgICBpZiBsZW4ob3JkZXIpICE9IGxlbih0cmFpbl9kcykgb3IgcHJvZ3Jlc3NbJ3NlZW4nXSAhPSBtaW4ocHJvZ3Jlc3NbJ25leHRfYmF0Y2gnXSpjZmdbJ2JhdGNoX3NpemUnXSwgbGVuKHRyYWluX2RzKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1NhdmVkIGN1cnNvci9zYW1wbGUgY291bnQgZG9lcyBub3QgbWF0Y2ggZGF0YXNldCcpCiAgICAgICAgdHJhaW5fbG9hZGVyID0gRGF0YUxvYWRlcih0cmFpbl9kcywgYmF0Y2hfc2FtcGxlcj1zYW1wbGVyLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbnVtX3dvcmtlcnM9Y2ZnWyd3b3JrZXJzJ10sIHBpbl9tZW1vcnk9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGdlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChzZWVkKzEwMDAwMCtlcG9jaCkpCiAgICAgICAgZGVmIGNoZWNrcG9pbnRfcGFydGlhbChjdXJyZW50KToKICAgICAgICAgICAgc3RhdGUgPSBzbmFwc2hvdChtb2RlbCwgb3B0aW1pemVyLCBzY2FsZXIsIGZpbmdlcnByaW50LCBlcG9jaCwgRmFsc2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb3JkZXIsIGN1cnJlbnQsIGJlc3QsIGhpc3RvcnkpCiAgICAgICAgICAgIHNhdmVfYXRvbWljKHN0YXRlLCByZXN1bWVfcGF0aCwga2VlcF9wcmV2aW91cz1UcnVlKQogICAgICAgICAgICBwcmludChmIlNhdmVkIGVwb2NoIHtlcG9jaCsxfSwge2N1cnJlbnRbJ25leHRfYmF0Y2gnXX0ve3RvdGFsX2JhdGNoZXN9IGJhdGNoZXMsICIKICAgICAgICAgICAgICAgICAgZiJvcHRpbWl6ZXIgdXBkYXRlIHtjdXJyZW50WydzdGVwJ119IHRvIHtyZXN1bWVfcGF0aH0iLCBmbHVzaD1UcnVlKQogICAgICAgIHByb2dyZXNzID0gdHJhaW5fZXBvY2gobW9kZWwsIHRyYWluX2xvYWRlciwgb3B0aW1pemVyLCBzY2FsZXIsIGNmZywgZGV2aWNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZXBvY2gsIGxlbih0cmFpbl9kcyksIHByb2dyZXNzLCB0b3RhbF9zdGVwcywgd2FybXVwLCBjaGVja3BvaW50X3BhcnRpYWwpCiAgICAgICAgc3RlcCA9IHByb2dyZXNzWydzdGVwJ10KICAgICAgICByZXN1bWVkID0gTm9uZQogICAgICAgIHZhbGlkYXRpb25fc3RhcnRlZCA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICBtb2RlbC5ldmFsKCk7IHZhbF9zdW0sIHZhbF9zZWVuID0gMC4sIDAKICAgICAgICB2YWxfcm5nID0gdG9yY2guR2VuZXJhdG9yKGRldmljZT1kZXZpY2UpLm1hbnVhbF9zZWVkKHNlZWQrMTAwMCkKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgZm9yIGltYWdlcywgdmFsaWQsIF8gaW4gdmFsX2xvYWRlcjoKICAgICAgICAgICAgICAgIGltYWdlcywgdmFsaWQgPSBpbWFnZXMudG8oZGV2aWNlKSwgdmFsaWQudG8oZGV2aWNlKQogICAgICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgnY3VkYScsIGR0eXBlPXRvcmNoLmZsb2F0MTYsIGVuYWJsZWQ9YW1wKToKICAgICAgICAgICAgICAgICAgICBsb3NzLCBfLCBfID0gbW9kZWwoaW1hZ2VzLCB2YWxpZCwgZ2VuZXJhdG9yPXZhbF9ybmcpCiAgICAgICAgICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUobG9zcyk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKCdOb25maW5pdGUgdmFsaWRhdGlvbiBsb3NzJykKICAgICAgICAgICAgICAgIHZhbF9zdW0gKz0gbG9zcy5pdGVtKCkqbGVuKGltYWdlcyk7IHZhbF9zZWVuICs9IGxlbihpbWFnZXMpCiAgICAgICAgdmFsX2xvc3MgPSB2YWxfc3VtL3ZhbF9zZWVuCiAgICAgICAgcm93ID0gZGljdChlcG9jaD1lcG9jaCsxLCB0cmFpbl9sb3NzPXByb2dyZXNzWydsb3NzX3N1bSddL3Byb2dyZXNzWydzZWVuJ10sIHNzbF92YWxfbG9zcz12YWxfbG9zcywKICAgICAgICAgICAgICAgICAgIGxyPW9wdGltaXplci5wYXJhbV9ncm91cHNbMF1bJ2xyJ10sIHNlY29uZHM9cHJvZ3Jlc3NbJ3NlY29uZHMnXSt0aW1lLm1vbm90b25pYygpLXZhbGlkYXRpb25fc3RhcnRlZCwKICAgICAgICAgICAgICAgICAgIHBlYWtfZ3B1X2dpYj1tYXgocHJvZ3Jlc3NbJ3BlYWtfZ3B1X2dpYiddLCB0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkvMioqMzApKQogICAgICAgIGhpc3RvcnkuYXBwZW5kKHJvdyk7IHByaW50KHJvdywgZmx1c2g9VHJ1ZSkKICAgICAgICBpbXByb3ZlZCA9IHZhbF9sb3NzIDwgYmVzdAogICAgICAgIGJlc3QgPSBtaW4oYmVzdCwgdmFsX2xvc3MpCiAgICAgICAgcGQuRGF0YUZyYW1lKGhpc3RvcnkpLnRvX2NzdihvdXQvJ21ldHJpY3MuY3N2JywgaW5kZXg9RmFsc2UpCiAgICAgICAgaWYgaW1wcm92ZWQ6CiAgICAgICAgICAgIHNhdmVfYXRvbWljKHsnbW9kZWwnOiBleHBvcnRfZW5jb2Rlcihtb2RlbCksICdlcG9jaCc6IGVwb2NoKzEsCiAgICAgICAgICAgICAgICAgICAgICAgICAna2luZCc6ICdiZXN0IFNTTCByZWNvbnN0cnVjdGlvbjsgbm90IGRvd25zdHJlYW0tc2VsZWN0ZWQnLAogICAgICAgICAgICAgICAgICAgICAgICAgJ2lucHV0X3NpemUnOiBjZmdbJ2lucHV0X3NpemUnXSwgJ2ZpbmdlcnByaW50JzogZmluZ2VycHJpbnR9LCBlbmNvZGVycy8nZW5jb2Rlcl9iZXN0X3NzbC5wdCcpCiAgICAgICAgaWYgKGVwb2NoKzEpJWNmZ1snc2F2ZV9ldmVyeSddID09IDAgb3IgZXBvY2grMSA9PSBlcG9jaHM6CiAgICAgICAgICAgICMgUHJlc2VydmUgZnVsbCByZXN1bWFibGUgbWlsZXN0b25lcywgaW5jbHVkaW5nIGVwb2NoIDIwLCBldmVuCiAgICAgICAgICAgICMgYWZ0ZXIgbGFzdC5wdCBhZHZhbmNlcy4gQm90aCBjaGVja3BvaW50IHR5cGVzIGxpdmUgb24gRHJpdmUuCiAgICAgICAgICAgIHNhdmVfYXRvbWljKHsnbW9kZWwnOiBleHBvcnRfZW5jb2Rlcihtb2RlbCksICdlcG9jaCc6IGVwb2NoKzEsCiAgICAgICAgICAgICAgICAgICAgICAgICAnaW5wdXRfc2l6ZSc6IGNmZ1snaW5wdXRfc2l6ZSddLCAnZmluZ2VycHJpbnQnOiBmaW5nZXJwcmludH0sIGVuY29kZXJzL2YnZW5jb2Rlcl9lcG9jaF97ZXBvY2grMTowNGR9LnB0JykKICAgICAgICAgICAgdmlzdWFsaXplKG1vZGVsLCB2YWxfZHMsIHByZXZpZXdzL2YncmVjb25zdHJ1Y3Rpb25fe2Vwb2NoKzE6MDRkfS5wbmcnLCBzZWVkKzIwMDAsIGRldmljZSkKICAgICAgICAjIFB1Ymxpc2ggY29tcGxldGlvbiBhZnRlciB2YWxpZGF0aW9uIGFydGlmYWN0cy4gQW4gaW50ZXJydXB0aW9uIGR1cmluZwogICAgICAgICMgdmFsaWRhdGlvbi9leHBvcnQgcmVzdW1lcyB0aGUgcHJlLXZhbGlkYXRpb24gY2hlY2twb2ludCBhbmQgcmV0cmllcyBpdC4KICAgICAgICBzdGF0ZSA9IHNuYXBzaG90KG1vZGVsLCBvcHRpbWl6ZXIsIHNjYWxlciwgZmluZ2VycHJpbnQsIGVwb2NoLCBUcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgb3JkZXIsIHByb2dyZXNzLCBiZXN0LCBoaXN0b3J5KQogICAgICAgIGlmIChlcG9jaCsxKSVjZmdbJ3NhdmVfZXZlcnknXSA9PSAwIG9yIGVwb2NoKzEgPT0gZXBvY2hzOgogICAgICAgICAgICBzYXZlX2F0b21pYyhzdGF0ZSwgY2hlY2twb2ludHMvZidjaGVja3BvaW50X2Vwb2NoX3tlcG9jaCsxOjA0ZH0ucHQnKQogICAgICAgIHNhdmVfYXRvbWljKHN0YXRlLCByZXN1bWVfcGF0aCwga2VlcF9wcmV2aW91cz1UcnVlKQogICAgICAgIHByaW50KGYnQ29tcGxldGVkIGVwb2NoIHtlcG9jaCsxfTsgY2hlY2twb2ludCBzYXZlZDoge3Jlc3VtZV9wYXRofScsIGZsdXNoPVRydWUpCiAgICAob3V0Lydjb21wbGV0ZWQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkaWN0KGVwb2Nocz1lcG9jaHMsIGZpbmdlcnByaW50PWZpbmdlcnByaW50LCBwaWxvdD1waWxvdCksIGluZGVudD0yKSkKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1jb25maWcnLCByZXF1aXJlZD1UcnVlKTsgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1waWxvdCcsIGFjdGlvbj0nc3RvcmVfdHJ1ZScpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgbWFpbihqc29uLmxvYWRzKFBhdGgoYXJncy5jb25maWcpLnJlYWRfdGV4dCgpKSwgYXJncy5waWxvdCkK', 'setup_runtime.py': 'IiIiQnVpbGQgYW4gaXNvbGF0ZWQgQ1VEQSAxMS44L1B5dGhvbiAzLjEwIHNwYXJzZS10cmFpbmluZyBydW50aW1lIG9uIENvbGFiLiIiIgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCB0YXJmaWxlCmltcG9ydCB1cmxsaWIucmVxdWVzdAoKCkNVREFfQ0hBTk5FTCA9ICdudmlkaWEvbGFiZWwvY3VkYS0xMS44LjAnClNFVFVQX1ZFUlNJT04gPSA1CgoKZGVmIHBhdGNoX2VuZ2luZV9oZWFkZXJzKGVuZ2luZSk6CiAgICAiIiJQeVRvcmNoIDIuMSBDVURBVXRpbHMuaCBuZWVkcyBhIGNvbXBsZXRlIGF0OjpUZW5zb3IgYmVmb3JlIGluY2x1c2lvbi4iIiIKICAgIHBhdGggPSBQYXRoKGVuZ2luZSkvJ3NyYy9zcG1tLmN1JwogICAgc291cmNlID0gcGF0aC5yZWFkX3RleHQoKQogICAgYW5jaG9yID0gJyNpbmNsdWRlIDxBVGVuL2N1ZGEvQ1VEQUNvbnRleHQuaD4nCiAgICByZXBsYWNlbWVudCA9ICcjaW5jbHVkZSA8QVRlbi9BVGVuLmg+ICAvLyBDYXJkaW9TZW50cnk6IGNvbXBsZXRlIFRlbnNvciBiZWZvcmUgQ1VEQVV0aWxzXG4nK2FuY2hvcgogICAgaWYgcmVwbGFjZW1lbnQgaW4gc291cmNlOgogICAgICAgIHJldHVybgogICAgaWYgc291cmNlLmNvdW50KGFuY2hvcikgIT0gMSBvciAnI2luY2x1ZGUgPEFUZW4vY3VkYS9DVURBVXRpbHMuaD4nIG5vdCBpbiBzb3VyY2U6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdVbmV4cGVjdGVkIE1pbmtvd3NraUVuZ2luZSBzcG1tLmN1IGhlYWRlcnM7IHJlZnVzaW5nIGNvbXBhdGliaWxpdHkgcGF0Y2gnKQogICAgcGF0aC53cml0ZV90ZXh0KHNvdXJjZS5yZXBsYWNlKGFuY2hvciwgcmVwbGFjZW1lbnQsIDEpKQogICAgcHJpbnQoJ0FwcGxpZWQgc3BtbS5jdSBjb21wbGV0ZS1UZW5zb3IgaGVhZGVyIGZpeCAoc2V0dXAgdjMpJywgZmx1c2g9VHJ1ZSkKCgpkZWYgcGF0Y2hfZW5naW5lX3B5dGhvbihlbmdpbmUpOgogICAgIiIiUmVwYWlyIGFsbCBmaXZlIGxlZ2FjeSBBQkMgaW1wb3J0cyBpbiB0aGUgcGlubmVkIFB5dGhvbiBwYWNrYWdlLiIiIgogICAgcGFja2FnZSA9IFBhdGgoZW5naW5lKS8nTWlua293c2tpRW5naW5lJwogICAgZm9yIG5hbWUgaW4gWydNaW5rb3dza2lDb29yZGluYXRlTWFuYWdlci5weScsICdNaW5rb3dza2lLZXJuZWxHZW5lcmF0b3IucHknLAogICAgICAgICAgICAgICAgICdNaW5rb3dza2lDb21tb24ucHknLCAnTWlua293c2tpVGVuc29yRmllbGQucHknLCAndXRpbHMvcXVhbnRpemF0aW9uLnB5J106CiAgICAgICAgcGF0aCA9IHBhY2thZ2UvbmFtZQogICAgICAgIHNvdXJjZSA9IHBhdGgucmVhZF90ZXh0KCkKICAgICAgICBvbGQgPSAoJ2Zyb20gY29sbGVjdGlvbnMgaW1wb3J0IFNlcXVlbmNlLCBuYW1lZHR1cGxlJyBpZiBuYW1lID09ICdNaW5rb3dza2lLZXJuZWxHZW5lcmF0b3IucHknCiAgICAgICAgICAgICAgIGVsc2UgJ2Zyb20gY29sbGVjdGlvbnMgaW1wb3J0IFNlcXVlbmNlJykKICAgICAgICBuZXcgPSAoJ2Zyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBTZXF1ZW5jZVxuZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgbmFtZWR0dXBsZScKICAgICAgICAgICAgICAgaWYgbmFtZSA9PSAnTWlua293c2tpS2VybmVsR2VuZXJhdG9yLnB5JyBlbHNlICdmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgU2VxdWVuY2UnKQogICAgICAgIGlmIG5ldyBpbiBzb3VyY2U6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgc291cmNlLmNvdW50KG9sZCkgIT0gMToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYnVW5leHBlY3RlZCBsZWdhY3kgaW1wb3J0cyBpbiB7cGF0aH07IHJlZnVzaW5nIGNvbXBhdGliaWxpdHkgcGF0Y2gnKQogICAgICAgIHBhdGgud3JpdGVfdGV4dChzb3VyY2UucmVwbGFjZShvbGQsIG5ldywgMSkpCiAgICAjIFVwc3RyZWFtIGRlbGV0ZXMgYnVpbGQvIG9uIGV2ZXJ5IGluc3RhbGwuIExldCBOaW5qYSByZXVzZSB2YWxpZCBvYmplY3RzCiAgICAjIGFuZCB0cmFjayBjaGFuZ2VkIHNvdXJjZS9oZWFkZXIgZGVwZW5kZW5jaWVzIGFuZCBjb21waWxlciBjb21tYW5kIGxpbmVzLgogICAgcGF0aCA9IFBhdGgoZW5naW5lKS8nc2V0dXAucHknCiAgICBzb3VyY2UgPSBwYXRoLnJlYWRfdGV4dCgpCiAgICBvbGQgPSAncnVuX2NvbW1hbmQoInJtIiwgIi1yZiIsICJidWlsZCIpJwogICAgbmV3ID0gJyMgQ2FyZGlvU2VudHJ5OiByZXRhaW4gYnVpbGQgb2JqZWN0cyBmb3IgTmluamEgaW5jcmVtZW50YWwgcmVidWlsZHMnCiAgICBpZiBuZXcgbm90IGluIHNvdXJjZToKICAgICAgICBpZiBzb3VyY2UuY291bnQob2xkKSAhPSAxOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1VuZXhwZWN0ZWQgdXBzdHJlYW0gYnVpbGQgY2xlYW51cDsgcmVmdXNpbmcgcGF0Y2gnKQogICAgICAgIHBhdGgud3JpdGVfdGV4dChzb3VyY2UucmVwbGFjZShvbGQsIG5ldywgMSkpCiAgICBwcmludCgnVmVyaWZpZWQgUHl0aG9uIDMuMTAgaW1wb3J0cyBpbiBmaXZlIGZpbGVzOyBpbmNyZW1lbnRhbCBidWlsZHMgZW5hYmxlZCAodjQpJywgZmx1c2g9VHJ1ZSkKCgpkZWYgY2hlY2tfY3VkYShlbnYsIGJ1aWxkX2Vudik6CiAgICAiIiJDaGVjayB0aGUgZXhhY3QgY29tcGlsZXIgUHlUb3JjaCB3aWxsIHVzZSwgbm90IG52aWRpYS1zbWkncyBkcml2ZXIgY2VpbGluZy4iIiIKICAgIG52Y2MgPSBQYXRoKGVudikvJ2Jpbi9udmNjJwogICAgb3V0cHV0ID0gc3VicHJvY2Vzcy5jaGVja19vdXRwdXQoW3N0cihudmNjKSwgJy0tdmVyc2lvbiddLCBlbnY9YnVpbGRfZW52LCB0ZXh0PVRydWUpCiAgICBtYXRjaCA9IHJlLnNlYXJjaChyJ3JlbGVhc2VccysoXGQrXC5cZCspJywgb3V0cHV0KQogICAgaWYgbm90IG1hdGNoIG9yIG1hdGNoLmdyb3VwKDEpICE9ICcxMS44JzoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZidFeHBlY3RlZCBDVURBIGNvbXBpbGVyIDExLjggYXQge252Y2N9OyBnb3Q6IHtvdXRwdXR9JykKICAgIHByaW50KG91dHB1dCwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBtYXRjaC5ncm91cCgxKQoKCmRlZiBzZXR1cChyb290KToKICAgIHJvb3QgPSBQYXRoKHJvb3QpLnJlc29sdmUoKTsgcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBlbnYgPSByb290LydlbnYnCiAgICByZWFkeSA9IHJvb3QvJ3J1bnRpbWVfcmVhZHkuanNvbicKICAgIGlmIHJlYWR5LmV4aXN0cygpIGFuZCBqc29uLmxvYWRzKHJlYWR5LnJlYWRfdGV4dCgpKS5nZXQoJ3NldHVwX3ZlcnNpb24nKSA9PSBTRVRVUF9WRVJTSU9OOgogICAgICAgIGNoZWNrX2N1ZGEoZW52LCBvcy5lbnZpcm9uLmNvcHkoKSkKICAgICAgICByZXR1cm4gc3RyKGVudi8nYmluL3B5dGhvbicpCiAgICBpZiBzaHV0aWwuZGlza191c2FnZShyb290KS5mcmVlIDwgMjAqMioqMzA6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdSZXNlcnZlIGF0IGxlYXN0IDIwIEdpQiBmb3IgdGhlIGNvbXBpbGVyL3J1bnRpbWUsIHBsdXMgZXh0cmFjdGVkIGRhdGFzZXQgc3RvcmFnZScpCiAgICBhcmNoaXZlID0gcm9vdC8nbWljcm9tYW1iYS50YXIuYnoyJwogICAgaWYgbm90IGFyY2hpdmUuZXhpc3RzKCk6CiAgICAgICAgdXJsbGliLnJlcXVlc3QudXJscmV0cmlldmUoJ2h0dHBzOi8vbWljcm8ubWFtYmEucG0vYXBpL21pY3JvbWFtYmEvbGludXgtNjQvMi4wLjUnLCBhcmNoaXZlKQogICAgd2l0aCB0YXJmaWxlLm9wZW4oYXJjaGl2ZSkgYXMgdGY6CiAgICAgICAgaXRlbSA9IHRmLmdldG1lbWJlcignYmluL21pY3JvbWFtYmEnKQogICAgICAgIGlmIG5vdCBpdGVtLmlzZmlsZSgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVbmV4cGVjdGVkIG1pY3JvbWFtYmEgYXJjaGl2ZScpCiAgICAgICAgKHJvb3QvJ2JpbicpLm1rZGlyKGV4aXN0X29rPVRydWUpCiAgICAgICAgd2l0aCB0Zi5leHRyYWN0ZmlsZShpdGVtKSBhcyBzcmMsIG9wZW4ocm9vdC8nYmluL21pY3JvbWFtYmEnLCAnd2InKSBhcyBkc3Q6CiAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihzcmMsIGRzdCkKICAgIG1hbWJhID0gcm9vdC8nYmluL21pY3JvbWFtYmEnOyBtYW1iYS5jaG1vZCgwbzc1NSkKICAgIGJ1aWxkX2VudiA9IGRpY3Qob3MuZW52aXJvbiwgTUFNQkFfUk9PVF9QUkVGSVg9c3RyKHJvb3QvJ21hbWJhJyksCiAgICAgICAgICAgICAgICAgICAgIENVREFfSE9NRT1zdHIoZW52KSwgQ1VEQV9QQVRIPXN0cihlbnYpLCBDVURBQ1hYPXN0cihlbnYvJ2Jpbi9udmNjJyksIE1BWF9KT0JTPScyJywgT01QX05VTV9USFJFQURTPSc0JykKICAgIGRlZiBydW4oYXJncyk6CiAgICAgICAgc3VicHJvY2Vzcy5ydW4oW3N0cihhKSBmb3IgYSBpbiBhcmdzXSwgZW52PWJ1aWxkX2VudiwgY2hlY2s9VHJ1ZSkKICAgICMgQSB0b29sa2l0IG1ldGFwYWNrYWdlIGFsb25lIHBlcm1pdHMgbmV3ZXIgdHJhbnNpdGl2ZSBDVURBIGNvbXBvbmVudHMuCiAgICAjIFRoZSByZWxlYXNlIGxhYmVsIHBsdXMgc3RyaWN0IHByaW9yaXR5IHNlbGVjdHMgdGhlIG1hdGNoaW5nIGNvbXBvbmVudCBzZXQuCiAgICAjIEluc3RhbGwgaW50byBhbiBleGlzdGluZyBwcmVmaXggdG9vLCByZXBhaXJpbmcgZmFpbGVkIG9sZGVyIHNldHVwIGF0dGVtcHRzLgogICAgYWN0aW9uID0gJ2luc3RhbGwnIGlmIChlbnYvJ2NvbmRhLW1ldGEnKS5leGlzdHMoKSBlbHNlICdjcmVhdGUnCiAgICBydW4oW21hbWJhLCBhY3Rpb24sICcteScsICctcCcsIGVudiwgJy0tb3ZlcnJpZGUtY2hhbm5lbHMnLCAnLS1zdHJpY3QtY2hhbm5lbC1wcmlvcml0eScsCiAgICAgICAgICctYycsIENVREFfQ0hBTk5FTCwgJy1jJywgJ2NvbmRhLWZvcmdlJywKICAgICAgICAgJ3B5dGhvbj0zLjEwLjE0JywgJ3BpcD0yNC4wJywgJ2N1ZGEtdG9vbGtpdD0xMS44LjAnLCAnY3VkYS1udmNjPTExLjguODknLAogICAgICAgICAnb3BlbmJsYXM9MC4zLjI3JywgJ2d4eF9saW51eC02ND0xMS40LjAnLCAnbmluamE9MS4xMi4xJ10pCiAgICBweSA9IGVudi8nYmluL3B5dGhvbicKICAgIGJ1aWxkX2VudlsnUEFUSCddID0gc3RyKGVudi8nYmluJykrb3MucGF0aHNlcCtvcy5lbnZpcm9uWydQQVRIJ10KICAgIGJ1aWxkX2VudlsnTERfTElCUkFSWV9QQVRIJ10gPSBzdHIoZW52LydsaWInKStvcy5wYXRoc2VwK29zLmVudmlyb24uZ2V0KCdMRF9MSUJSQVJZX1BBVEgnLCAnJykKICAgIGJ1aWxkX2VudlsnQ0MnXSA9IHN0cihlbnYvJ2Jpbi94ODZfNjQtY29uZGEtbGludXgtZ251LWNjJykKICAgIGJ1aWxkX2VudlsnQ1hYJ10gPSBzdHIoZW52LydiaW4veDg2XzY0LWNvbmRhLWxpbnV4LWdudS1jKysnKQogICAgY2hlY2tfY3VkYShlbnYsIGJ1aWxkX2VudikKICAgIHJ1bihbcHksICctbScsICdwaXAnLCAnaW5zdGFsbCcsICdzZXR1cHRvb2xzPT02OC4yLjInLCAnd2hlZWw9PTAuNDMuMCcsICdudW1weT09MS4yNi40J10pCiAgICBydW4oW3B5LCAnLW0nLCAncGlwJywgJ2luc3RhbGwnLCAndG9yY2g9PTIuMS4yJywgJ3RvcmNodmlzaW9uPT0wLjE2LjInLAogICAgICAgICAnLS1pbmRleC11cmwnLCAnaHR0cHM6Ly9kb3dubG9hZC5weXRvcmNoLm9yZy93aGwvY3UxMTgnXSkKICAgIHJ1bihbcHksICctbScsICdwaXAnLCAnaW5zdGFsbCcsICd0aW1tPT0wLjkuMTYnLCAncGFuZGFzPT0yLjIuMicsICdQaWxsb3c9PTEwLjQuMCcsCiAgICAgICAgICdtYXRwbG90bGliPT0zLjguNCddKQogICAgcnVuKFtweSwgJy1jJywgJ2ltcG9ydCB0b3JjaDsgZnJvbSB0b3JjaC51dGlscy5jcHBfZXh0ZW5zaW9uIGltcG9ydCBDVURBX0hPTUU7ICcKICAgICAgICAgJ2ltcG9ydCBvczsgYXNzZXJ0IHRvcmNoLnZlcnNpb24uY3VkYSA9PSBcIjExLjhcIjsgJwogICAgICAgICAnYXNzZXJ0IENVREFfSE9NRSA9PSBvcy5lbnZpcm9uW1wiQ1VEQV9IT01FXCJdOyBwcmludChcIkJ1aWxkIENVREFfSE9NRTpcIiwgQ1VEQV9IT01FKSddKQogICAgZW5naW5lID0gcm9vdC8nTWlua293c2tpRW5naW5lJwogICAgcmV2aXNpb24gPSAnYmJjMzBlZjU4MWVhNmRlYjUwNTk3NmI2NjNmNWZjMjM1OGE4Mzc0OScKICAgIGlmIG5vdCBlbmdpbmUuZXhpc3RzKCk6CiAgICAgICAgcnVuKFsnZ2l0JywgJ2Nsb25lJywgJ2h0dHBzOi8vZ2l0aHViLmNvbS9zaHdvbzkzL01pbmtvd3NraUVuZ2luZS5naXQnLCBlbmdpbmVdKQogICAgcnVuKFsnZ2l0JywgJy1DJywgZW5naW5lLCAnY2hlY2tvdXQnLCAnLS1kZXRhY2gnLCByZXZpc2lvbl0pCiAgICBwYXRjaF9lbmdpbmVfaGVhZGVycyhlbmdpbmUpCiAgICBwYXRjaF9lbmdpbmVfcHl0aG9uKGVuZ2luZSkKICAgICMgVGhlIE1ldGEtcGlubmVkIGZvcmssIG5vdCB0aGUgTlZJRElBIHVwc3RyZWFtIGJyYW5jaC4gU2V0dXAgb3B0aW9ucyBtdXN0CiAgICAjIHJlYWNoIHNldHVwLnB5IGRpcmVjdGx5OyB3aGVlbCBidWlsZHMgb3RoZXJ3aXNlIGRpc2NhcmQgdGhlc2UgZmxhZ3MuCiAgICBzdWJwcm9jZXNzLnJ1bihbc3RyKHB5KSwgJ3NldHVwLnB5JywgJ2luc3RhbGwnLCAnLS1mb3JjZV9jdWRhJywgJy0tYmxhcz1vcGVuYmxhcycsCiAgICAgICAgICAgICAgICAgICAgJy0tYmxhc19pbmNsdWRlX2RpcnM9JytzdHIoZW52LydpbmNsdWRlJyksICctLWJsYXNfbGlicmFyeV9kaXJzPScrc3RyKGVudi8nbGliJyldLAogICAgICAgICAgICAgICAgICAgY3dkPWVuZ2luZSwgZW52PWJ1aWxkX2VudiwgY2hlY2s9VHJ1ZSkKICAgIHJ1bihbcHksICctYycsICdpbXBvcnQgdG9yY2gsIE1pbmtvd3NraUVuZ2luZTsgYXNzZXJ0IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk7IHByaW50KHRvcmNoLl9fdmVyc2lvbl9fLCBNaW5rb3dza2lFbmdpbmUuX192ZXJzaW9uX18pJ10pCiAgICAjIG1pY3JvbWFtYmEgMi54IGV4cG9zZXMgZXhwbGljaXQgbG9jayBleHBvcnQgdW5kZXIgYGBlbnYgZXhwb3J0YGA7CiAgICAjIGBgbGlzdCAtLWV4cGxpY2l0YGAgaXMgY29uZGEgc3ludGF4IGFuZCBleGl0cyAxMDkgYWZ0ZXIgYW4gb3RoZXJ3aXNlCiAgICAjIHN1Y2Nlc3NmdWwgcnVudGltZSBidWlsZC4KICAgIGxvY2sgPSBzdWJwcm9jZXNzLmNoZWNrX291dHB1dCgKICAgICAgICBbc3RyKG1hbWJhKSwgJ2VudicsICdleHBvcnQnLCAnLXAnLCBzdHIoZW52KSwgJy0tZXhwbGljaXQnXSwKICAgICAgICBlbnY9YnVpbGRfZW52LAogICAgICAgIHRleHQ9VHJ1ZSwKICAgICkKICAgIGlmICdARVhQTElDSVQnIG5vdCBpbiBsb2NrLnNwbGl0bGluZXMoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ21pY3JvbWFtYmEgZGlkIG5vdCBwcm9kdWNlIGFuIGV4cGxpY2l0IGVudmlyb25tZW50IGxvY2snKQogICAgKHJvb3QvJ2NvbmRhLWV4cGxpY2l0LnR4dCcpLndyaXRlX3RleHQobG9jaykKICAgIChyb290LydwaXAtZnJlZXplLnR4dCcpLndyaXRlX3RleHQoc3VicHJvY2Vzcy5jaGVja19vdXRwdXQoW3N0cihweSksICctbScsICdwaXAnLCAnZnJlZXplJ10sIGVudj1idWlsZF9lbnYsIHRleHQ9VHJ1ZSkpCiAgICByZWFkeS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdChweXRob249c3RyKHB5KSwgZW5naW5lX3JldmlzaW9uPXJldmlzaW9uLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNldHVwX3ZlcnNpb249U0VUVVBfVkVSU0lPTiwgY3VkYV9jaGFubmVsPUNVREFfQ0hBTk5FTCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbmdpbmVfcGF0Y2hlcz1bJ3NwbW0tY29tcGxldGUtdGVuc29yLWhlYWRlcicsICdjb2xsZWN0aW9ucy1hYmMtZml2ZS1maWxlcycsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICdyZXRhaW4tbmluamEtYnVpbGQtb2JqZWN0cyddKSwgaW5kZW50PTIpKQogICAgcmV0dXJuIHN0cihweSkKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgaW1wb3J0IHN5cwogICAgcHJpbnQoc2V0dXAoc3lzLmFyZ3ZbMV0pKQo=', 'test_data_utils.py': 'IiIiQ1BVIHJlZ3Jlc3Npb24gY2hlY2tzIGZvciB0aGUgbm90ZWJvb2sncyBkYXRhL21hc2tpbmcvbG9zcyBhZGFwdGF0aW9ucy4iIiIKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB1bml0dGVzdApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBQSUwgaW1wb3J0IEltYWdlCmltcG9ydCB0b3JjaApmcm9tIGRhdGFfdXRpbHMgaW1wb3J0IChsZXR0ZXJib3gsIHBhdGNoaWZ5LCB1bnBhdGNoaWZ5LCB1bmlmb3JtX21hc2ssCiAgICAgICAgICAgICAgICAgICAgICAgIHRhcmdldHMsIHJlY29uc3RydWN0aW9uX2xvc3MsIHByZXBhcmVfc3BsaXRzKQoKCmNsYXNzIERhdGFUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9yZWN0YW5ndWxhcl9yb3VuZHRyaXAoc2VsZik6CiAgICAgICAgZm9yIGgsIHcgaW4gWyg2NCwgOTYpLCAoNzY4LCAxMDI0KV06CiAgICAgICAgICAgIHggPSB0b3JjaC5yYW5kbigxLCAzLCBoLCB3KQogICAgICAgICAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZSh1bnBhdGNoaWZ5KHBhdGNoaWZ5KHgpLCAoaC8vMzIsIHcvLzMyKSksIHgpCgogICAgZGVmIHRlc3RfcGFkZGluZ19pbnZhcmlhbmNlX2xvc3NfYW5kX3N0YXRzKHNlbGYpOgogICAgICAgIHggPSB0b3JjaC5yYW5kbigxLCAzLCA2NCwgOTYpCiAgICAgICAgdmFsaWQgPSB0b3JjaC56ZXJvcygxLCAxLCA2NCwgOTYpOyB2YWxpZFs6LCA6LCA3OjU3LCAxMDo4MF0gPSAxCiAgICAgICAgY2hhbmdlZCA9IHggKyAoMS12YWxpZCkqMTAwCiAgICAgICAgbWFzayA9IHVuaWZvcm1fbWFzayh2YWxpZCkKICAgICAgICBwcmVkID0gdG9yY2gucmFuZG4oMSwgMzA3MiwgMiwgMywgcmVxdWlyZXNfZ3JhZD1UcnVlKQogICAgICAgIGEgPSByZWNvbnN0cnVjdGlvbl9sb3NzKHgsIHByZWQsIHZhbGlkLCBtYXNrKQogICAgICAgIGIgPSByZWNvbnN0cnVjdGlvbl9sb3NzKGNoYW5nZWQsIHByZWQsIHZhbGlkLCBtYXNrKQogICAgICAgIHRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGEsIGIpCiAgICAgICAgZm9yIGFhLCBiYiBpbiB6aXAodGFyZ2V0cyh4LCB2YWxpZClbMjpdLCB0YXJnZXRzKGNoYW5nZWQsIHZhbGlkKVsyOl0pOgogICAgICAgICAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhYSwgYmIpCiAgICAgICAgYS5iYWNrd2FyZCgpOyBzZWxmLmFzc2VydFRydWUodG9yY2guaXNmaW5pdGUocHJlZC5ncmFkKS5hbGwoKSkKCiAgICBkZWYgdGVzdF9mdWxsX3ZhbGlkX21hdGNoZXNfb2ZmaWNpYWxfbG9zcyhzZWxmKToKICAgICAgICB4ID0gdG9yY2gucmFuZG4oMiwgMywgNjQsIDk2KQogICAgICAgIHZhbGlkID0gdG9yY2gub25lcygyLCAxLCA2NCwgOTYpOyBtYXNrID0gdW5pZm9ybV9tYXNrKHZhbGlkKQogICAgICAgIHByZWQgPSB0b3JjaC5yYW5kbigyLCA2LCAzMDcyKQogICAgICAgIHQgPSBwYXRjaGlmeSh4KTsgdCA9ICh0LXQubWVhbigtMSwga2VlcGRpbT1UcnVlKSkvKHQudmFyKC0xLCBrZWVwZGltPVRydWUpKzFlLTYpLnNxcnQoKQogICAgICAgIGV4cGVjdGVkID0gKCgocHJlZC10KS5zcXVhcmUoKS5tZWFuKC0xKSkqbWFzaykuc3VtKCkvbWFzay5zdW0oKQogICAgICAgIHRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKHJlY29uc3RydWN0aW9uX2xvc3MoeCwgcHJlZCwgdmFsaWQsIG1hc2spLCBleHBlY3RlZCkKCiAgICBkZWYgdGVzdF91bmlmb3JtX2lnbm9yZXNfZW1wdHlfcGF0Y2hlcyhzZWxmKToKICAgICAgICB2YWxpZCA9IHRvcmNoLnplcm9zKDIsIDEsIDk2LCAxNjApOyB2YWxpZFs6LCA6LCA6NjQsIDoxMjhdID0gMQogICAgICAgIGEgPSB1bmlmb3JtX21hc2sodmFsaWQsIGdlbmVyYXRvcj10b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZCgzKSkKICAgICAgICBiID0gdW5pZm9ybV9tYXNrKHZhbGlkLCBnZW5lcmF0b3I9dG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoMykpCiAgICAgICAgdG9yY2gudGVzdGluZy5hc3NlcnRfY2xvc2UoYSwgYikKICAgICAgICBzZWxmLmFzc2VydFRydWUoKGEuc3VtKC0xKSA9PSA1KS5hbGwoKSkKICAgICAgICBzZWxmLmFzc2VydFRydWUoKGFbcGF0Y2hpZnkodmFsaWQpLnN1bSgtMSkgPT0gMF0gPT0gMCkuYWxsKCkpCgogICAgZGVmIHRlc3RfbGV0dGVyYm94KHNlbGYpOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wOgogICAgICAgICAgICBwYXRoID0gUGF0aCh0bXApLydpbWFnZS5wbmcnOyBJbWFnZS5uZXcoJ1JHQicsICgxMDAsIDUwKSkuc2F2ZShwYXRoKQogICAgICAgICAgICBpbWFnZSwgdmFsaWQgPSBsZXR0ZXJib3gocGF0aCwgKDY0LCA5NikpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwodHVwbGUoaW1hZ2Uuc2hhcGUpLCAoMyw2NCw5NikpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwodmFsaWQuc3VtKCksIDQ4Kjk2KQoKICAgIGRlZiB0ZXN0X2NhY2hlX3ByZXNlcnZlc19waXhlbHNfYW5kX3ZhbGlkaXR5KHNlbGYpOgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wOgogICAgICAgICAgICBwYXRoID0gUGF0aCh0bXApLydvcmlnaW5hbC5wbmcnCiAgICAgICAgICAgIEltYWdlLmZyb21hcnJheShucC5yYW5kb20uZGVmYXVsdF9ybmcoMCkuaW50ZWdlcnMoMCwyNTYsKDkxLDEzNywzKSxkdHlwZT1ucC51aW50OCkpLnNhdmUocGF0aCkKICAgICAgICAgICAgY2FjaGUgPSBQYXRoKHRtcCkvJ2NhY2hlLnBuZycKICAgICAgICAgICAgYSwgYXYgPSBsZXR0ZXJib3gocGF0aCwgKDY0LDk2KSwgY2FjaGUpCiAgICAgICAgICAgIGIsIGJ2ID0gbGV0dGVyYm94KHBhdGgsICg2NCw5NiksIGNhY2hlKQogICAgICAgICAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhLCBiLCBydG9sPTAsIGF0b2w9MCkKICAgICAgICAgICAgdG9yY2gudGVzdGluZy5hc3NlcnRfY2xvc2UoYXYsIGJ2LCBydG9sPTAsIGF0b2w9MCkKCiAgICBkZWYgdGVzdF9wYXRpZW50X2hvbGRvdXRfYW5kX3N0YWJsZV9zcGxpdChzZWxmKToKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRtcDoKICAgICAgICAgICAgcm9vdCA9IFBhdGgodG1wKQogICAgICAgICAgICByb3dzID0gW10KICAgICAgICAgICAgZm9yIGkgaW4gcmFuZ2UoMTAwKToKICAgICAgICAgICAgICAgIHBhdGggPSBmJ3tpfS5wbmcnOyBJbWFnZS5uZXcoJ1JHQicsICg4LDgpKS5zYXZlKHJvb3QvcGF0aCkKICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKGRpY3QoaW1hZ2VfcGF0aD1wYXRoLCBmY21hZV9yZWNvcmRfaWQ9c3RyKGkpLCBzb3VyY2VfZGF0YXNldD0ncycsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNhbm9uaWNhbF9yZWNvcmRfaWQ9c3RyKGkpLCBwYXRpZW50X2lkPXN0cihpLy8yKSwgcGF0aWVudF9pZF9hdmFpbGFibGU9JzEnKSkKICAgICAgICAgICAgZnJhbWUgPSBwZC5EYXRhRnJhbWUocm93cyk7IGZyYW1lLnRvX2Nzdihyb290LydpbmRleC5jc3YnLCBpbmRleD1GYWxzZSkKICAgICAgICAgICAgZnJhbWUuaWxvY1tbMF1dLnRvX2Nzdihyb290Lydob2xkb3V0LmNzdicsIGluZGV4PUZhbHNlKQogICAgICAgICAgICBhLCBiLCBhdWRpdCA9IHByZXBhcmVfc3BsaXRzKHJvb3QvJ2luZGV4LmNzdicsIHJvb3QsIHJvb3QvJ291dCcsIDQyLCByb290Lydob2xkb3V0LmNzdicsIC4yKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGF1ZGl0WydleGNsdWRlZF9yZWNvcmRzJ10sIDIpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RmFsc2Uoc2V0KGEuZ3JvdXBfa2V5KSZzZXQoYi5ncm91cF9rZXkpKQogICAgICAgICAgICBzZWxmLmFzc2VydE5vdEluKCcwJywgc2V0KGEucGF0aWVudF9pZCl8c2V0KGIucGF0aWVudF9pZCkpCiAgICAgICAgICAgIGZyYW1lLmxvY1syLCAnY2Fub25pY2FsX3JlY29yZF9pZCddID0gZnJhbWUubG9jWzMsICdjYW5vbmljYWxfcmVjb3JkX2lkJ10KICAgICAgICAgICAgZnJhbWUudG9fY3N2KHJvb3QvJ2JhZC5jc3YnLCBpbmRleD1GYWxzZSkKICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgICAgIHByZXBhcmVfc3BsaXRzKHJvb3QvJ2JhZC5jc3YnLCByb290LCByb290LydiYWRvdXQnLCA0MikKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDIpCiAgICB1bml0dGVzdC5tYWluKCkK', 'test_checkpoint_utils.py': 'IiIiQ2hlY2twb2ludC1sYXlvdXQgdGVzdHMgb25seTsgYWN0dWFsIHNwYXJzZSBleGVjdXRpb24gaXMgYSBHUFUgcHJlZmxpZ2h0LiIiIgppbXBvcnQgYXJncGFyc2UKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdHlwZXMKaW1wb3J0IHVuaXR0ZXN0CmltcG9ydCB0b3JjaApmcm9tIGNoZWNrcG9pbnRfdXRpbHMgaW1wb3J0IGxvYWRfaW5pdGlhbCwgZXhwb3J0X2VuY29kZXIKCgpjbGFzcyBTaGFwZU9ubHlFbmNvZGVyOgogICAgIiIiRGVzY3JpYmUgc3BhcnNlIHRlbnNvciBzaGFwZXMgd2l0aG91dCBwcmV0ZW5kaW5nIHRvIGV4ZWN1dGUgTWlua293c2tpLiIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRlbnNlKToKICAgICAgICBzZWxmLnN0YXRlID0ge30KICAgICAgICBmb3Iga2V5LCB2YWx1ZSBpbiBkZW5zZS5pdGVtcygpOgogICAgICAgICAgICBzaGFwZSA9IHZhbHVlLnNoYXBlCiAgICAgICAgICAgIGlmIGtleS5zdGFydHN3aXRoKCdkb3duc2FtcGxlX2xheWVycy4nKSBhbmQga2V5LnNwbGl0KCcuJylbMV0gIT0gJzAnOgogICAgICAgICAgICAgICAgaWYga2V5LnNwbGl0KCcuJylbMl0gPT0gJzAnOgogICAgICAgICAgICAgICAgICAgIGtleSA9IGtleS5yZXBsYWNlKCcuMC4nLCAnLjAubG4uJykKICAgICAgICAgICAgICAgIGVsaWYga2V5LmVuZHN3aXRoKCd3ZWlnaHQnKToKICAgICAgICAgICAgICAgICAgICBrZXkgPSBrZXkucmVtb3Zlc3VmZml4KCd3ZWlnaHQnKSsna2VybmVsJwogICAgICAgICAgICAgICAgICAgIHNoYXBlID0gKDQsIHZhbHVlLnNoYXBlWzFdLCB2YWx1ZS5zaGFwZVswXSkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgc2hhcGUgPSAoMSwgdmFsdWUubnVtZWwoKSkKICAgICAgICAgICAgZWxpZiBrZXkuc3RhcnRzd2l0aCgnc3RhZ2VzLicpOgogICAgICAgICAgICAgICAgaWYgJy5kd2NvbnYuJyBpbiBrZXkgYW5kIGtleS5lbmRzd2l0aCgnd2VpZ2h0Jyk6CiAgICAgICAgICAgICAgICAgICAga2V5ID0ga2V5LnJlbW92ZXN1ZmZpeCgnd2VpZ2h0JykrJ2tlcm5lbCc7IHNoYXBlID0gKDQ5LCB2YWx1ZS5zaGFwZVswXSkKICAgICAgICAgICAgICAgIGVsaWYgJy5kd2NvbnYuJyBpbiBrZXk6CiAgICAgICAgICAgICAgICAgICAgc2hhcGUgPSAoMSwgdmFsdWUubnVtZWwoKSkKICAgICAgICAgICAgICAgIGtleSA9IGtleS5yZXBsYWNlKCcubm9ybS4nLCAnLm5vcm0ubG4uJykucmVwbGFjZSgnLnB3Y29udjEuJywgJy5wd2NvbnYxLmxpbmVhci4nKS5yZXBsYWNlKCcucHdjb252Mi4nLCAnLnB3Y29udjIubGluZWFyLicpCiAgICAgICAgICAgICAgICBpZiAnLmdybi4nIGluIGtleToKICAgICAgICAgICAgICAgICAgICBzaGFwZSA9ICgxLCB2YWx1ZS5udW1lbCgpKQogICAgICAgICAgICBzZWxmLnN0YXRlW2tleV0gPSB0b3JjaC5lbXB0eShzaGFwZSwgZHR5cGU9dmFsdWUuZHR5cGUpCgogICAgZGVmIHN0YXRlX2RpY3Qoc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYuc3RhdGUKCiAgICBkZWYgbG9hZF9zdGF0ZV9kaWN0KHNlbGYsIHN0YXRlLCBzdHJpY3QpOgogICAgICAgIGFzc2VydCBzdHJpY3QgYW5kIHNldChzdGF0ZSkgPT0gc2V0KHNlbGYuc3RhdGUpCiAgICAgICAgYXNzZXJ0IGFsbChzdGF0ZVtrXS5zaGFwZSA9PSB2LnNoYXBlIGZvciBrLCB2IGluIHNlbGYuc3RhdGUuaXRlbXMoKSkKICAgICAgICBzZWxmLnN0YXRlID0gc3RhdGUKCgpkZWYgdmVyaWZ5KHBhdGgsIGV4cGVjdGVkX2NvdW50PU5vbmUpOgogICAgZGVuc2UgPSB0b3JjaC5sb2FkKHBhdGgsIG1hcF9sb2NhdGlvbj0nY3B1Jywgd2VpZ2h0c19vbmx5PVRydWUpWydtb2RlbCddCiAgICBpZiBleHBlY3RlZF9jb3VudDoKICAgICAgICBhc3NlcnQgbGVuKGRlbnNlKSA9PSBleHBlY3RlZF9jb3VudAogICAgbW9kZWwgPSB0eXBlcy5TaW1wbGVOYW1lc3BhY2UoZW5jb2Rlcj1TaGFwZU9ubHlFbmNvZGVyKGRlbnNlKSkKICAgIHJlcG9ydCA9IGxvYWRfaW5pdGlhbChtb2RlbCwgcGF0aCkKICAgIGNvbnZlcnRlZCA9IGV4cG9ydF9lbmNvZGVyKG1vZGVsKQogICAgYXNzZXJ0IHNldChjb252ZXJ0ZWQpID09IHNldChkZW5zZSkKICAgIGZvciBrZXksIHZhbHVlIGluIGRlbnNlLml0ZW1zKCk6CiAgICAgICAgZXhwZWN0ZWQgPSB2YWx1ZS5yZXNoYXBlKDEsMSwxLC0xKSBpZiAnLmdybi4nIGluIGtleSBlbHNlIHZhbHVlCiAgICAgICAgdG9yY2gudGVzdGluZy5hc3NlcnRfY2xvc2UoY29udmVydGVkW2tleV0sIGV4cGVjdGVkLCBydG9sPTAsIGF0b2w9MCkKICAgIHByaW50KCdFeGFjdCBlbmNvZGVyIHRlbnNvciByb3VuZHRyaXA6JywgbGVuKGRlbnNlKSwgcmVwb3J0KQoKCmNsYXNzIENoZWNrcG9pbnRUZXN0cyh1bml0dGVzdC5UZXN0Q2FzZSk6CiAgICBkZWYgdGVzdF9jb252ZXJzaW9uX2FuZF9zdHJpY3RfbWlzc2luZyhzZWxmKToKICAgICAgICBzdGF0ZSA9IHsnZG93bnNhbXBsZV9sYXllcnMuMS4wLndlaWdodCc6IHRvcmNoLnJhbmRuKDMpLAogICAgICAgICAgICAgICAgICdkb3duc2FtcGxlX2xheWVycy4xLjEud2VpZ2h0JzogdG9yY2guYXJhbmdlKDI0LikucmVzaGFwZSgyLDMsMiwyKSwKICAgICAgICAgICAgICAgICAnZG93bnNhbXBsZV9sYXllcnMuMS4xLmJpYXMnOiB0b3JjaC5yYW5kbigyKSwKICAgICAgICAgICAgICAgICAnc3RhZ2VzLjAuMC5kd2NvbnYud2VpZ2h0JzogdG9yY2guYXJhbmdlKDE0Ny4pLnJlc2hhcGUoMywxLDcsNyksCiAgICAgICAgICAgICAgICAgJ3N0YWdlcy4wLjAubm9ybS53ZWlnaHQnOiB0b3JjaC5yYW5kbigzKSwKICAgICAgICAgICAgICAgICAnc3RhZ2VzLjAuMC5wd2NvbnYxLndlaWdodCc6IHRvcmNoLnJhbmRuKDEyLDMpLAogICAgICAgICAgICAgICAgICdzdGFnZXMuMC4wLmdybi5nYW1tYSc6IHRvcmNoLnJhbmRuKDEsMTIpfQogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KCkgYXMgdG1wOgogICAgICAgICAgICBwYXRoID0gUGF0aCh0bXApLydjaGVja3BvaW50LnB0JzsgdG9yY2guc2F2ZSh7J21vZGVsJzpzdGF0ZX0sIHBhdGgpCiAgICAgICAgICAgIHZlcmlmeShwYXRoKQogICAgICAgICAgICBtb2RlbCA9IHR5cGVzLlNpbXBsZU5hbWVzcGFjZShlbmNvZGVyPVNoYXBlT25seUVuY29kZXIoc3RhdGUpKQogICAgICAgICAgICBkZWwgc3RhdGVbJ3N0YWdlcy4wLjAubm9ybS53ZWlnaHQnXTsgdG9yY2guc2F2ZSh7J21vZGVsJzpzdGF0ZX0sIHBhdGgpCiAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBsb2FkX2luaXRpYWwobW9kZWwsIHBhdGgpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7IHBhcnNlci5hZGRfYXJndW1lbnQoJy0tY2hlY2twb2ludCcpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDIpCiAgICBpZiBhcmdzLmNoZWNrcG9pbnQ6CiAgICAgICAgdmVyaWZ5KGFyZ3MuY2hlY2twb2ludCwgMTk2KQogICAgZWxzZToKICAgICAgICB1bml0dGVzdC5tYWluKGFyZ3Y9Wyd0ZXN0X2NoZWNrcG9pbnRfdXRpbHMnXSkK', 'training_state.py': 'IiIiT3B0aW1pemVyLWJvdW5kYXJ5IGNoZWNrcG9pbnRpbmcgYW5kIGRldGVybWluaXN0aWMgcmVzdW1hYmxlIGJhdGNoIG9yZGVyLgoKVGhlIGRhdGFzZXQgbXVzdCBiZSBkZXRlcm1pbmlzdGljIHBlciBpbmRleCAoYXMgRUNHRGF0YXNldCBjdXJyZW50bHkgaXMpLgpUaGUgdHJhaW5lciwgbm90IERhdGFMb2FkZXIgcHJlZmV0Y2gsIG93bnMgdGhlIGNvbW1pdHRlZCBiYXRjaCBjdXJzb3IuCiIiIgppbXBvcnQgZXJybm8KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgcmFuZG9tCmltcG9ydCB0aW1lCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgU2FtcGxlcgoKCmRlZiBzYXZlX2F0b21pYyh2YWx1ZSwgcGF0aCwga2VlcF9wcmV2aW91cz1GYWxzZSk6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgdG1wID0gcGF0aC53aXRoX3N1ZmZpeChwYXRoLnN1ZmZpeCsnLnRtcCcpCiAgICBwcmV2aW91cyA9IHBhdGgud2l0aF9uYW1lKCdwcmV2aW91cy5wdCcpCiAgICB0cnk6CiAgICAgICAgd2l0aCBvcGVuKHRtcCwgJ3diJykgYXMgc3RyZWFtOgogICAgICAgICAgICB0b3JjaC5zYXZlKHZhbHVlLCBzdHJlYW0pCiAgICAgICAgICAgIHN0cmVhbS5mbHVzaCgpCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKICAgICAgICAgICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICAgICAgICAgIyBTb21lIG1vdW50ZWQgZmlsZXN5c3RlbXMgbGFjayBmc3luYy4gUmVuYW1lIHN0aWxsIHByZXZlbnRzCiAgICAgICAgICAgICAgICAjIGFuIHVuZmluaXNoZWQgc2VyaWFsaXphdGlvbiBmcm9tIHJlcGxhY2luZyB0aGUgbGFzdCBmaWxlLgogICAgICAgICAgICAgICAgaWYgZXhjLmVycm5vIG5vdCBpbiB7ZXJybm8uRUlOVkFMLCBlcnJuby5FTk9UU1VQLCBlcnJuby5FTk9TWVN9OgogICAgICAgICAgICAgICAgICAgIHJhaXNlCiAgICAgICAgaWYga2VlcF9wcmV2aW91cyBhbmQgcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgb3MucmVwbGFjZShwYXRoLCBwcmV2aW91cykKICAgICAgICBvcy5yZXBsYWNlKHRtcCwgcGF0aCkKICAgIGZpbmFsbHk6CiAgICAgICAgdG1wLnVubGluayhtaXNzaW5nX29rPVRydWUpCgoKZGVmIGxvYWRfcmVzdW1lKHBhdGgsIGZpbmdlcnByaW50KToKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBjYW5kaWRhdGVzID0gW3BhdGgsIHBhdGgud2l0aF9uYW1lKCdwcmV2aW91cy5wdCcpXQogICAgZm91bmQsIGVycm9ycyA9IEZhbHNlLCBbXQogICAgZm9yIGNhbmRpZGF0ZSBpbiBjYW5kaWRhdGVzOgogICAgICAgIGlmIG5vdCBjYW5kaWRhdGUuZXhpc3RzKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm91bmQgPSBUcnVlCiAgICAgICAgdHJ5OgogICAgICAgICAgICAjIFRoZXNlIGFyZSB0aGlzIHJ1bidzIG93biBmaWxlcywgaW5jbHVkaW5nIFB5dGhvbi9OdW1QeSBSTkcgc3RhdGUuCiAgICAgICAgICAgIHN0YXRlID0gdG9yY2gubG9hZChjYW5kaWRhdGUsIG1hcF9sb2NhdGlvbj0nY3B1Jywgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICBlcnJvcnMuYXBwZW5kKGYne2NhbmRpZGF0ZS5uYW1lfToge3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9JykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBzdGF0ZS5nZXQoJ2Zvcm1hdF92ZXJzaW9uJykgIT0gMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignT2xkZXIgY2hlY2twb2ludCBmb3JtYXQ6IHJlc3VtZSB3aXRoIGl0cyBvcmlnaW5hbCBub3RlYm9vazsgbm8gaW1wbGljaXQgbWlncmF0aW9uJykKICAgICAgICBpZiBzdGF0ZS5nZXQoJ2ZpbmdlcnByaW50JykgIT0gZmluZ2VycHJpbnQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1Jlc3VtZSBjb25maWd1cmF0aW9uL2RhdGEvc291cmNlIG1pc21hdGNoOiB1c2UgdGhlIG9yaWdpbmFsIG5vdGVib29rL3NldHRpbmdzIG9yIGEgbmV3IHJ1bl9kaXInKQogICAgICAgIGlmIGNhbmRpZGF0ZSAhPSBwYXRoOgogICAgICAgICAgICBwcmludChmJ1JlY292ZXJpbmcgZnJvbSB7Y2FuZGlkYXRlfTsgbGF0ZXN0IGNoZWNrcG9pbnQgbWlzc2luZy91bnJlYWRhYmxlLiB7ZXJyb3JzfScsIGZsdXNoPVRydWUpCiAgICAgICAgICAgIGlmIHBhdGguZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBvcy5yZXBsYWNlKHBhdGgsIHBhdGgud2l0aF9uYW1lKCdsYXN0LnVucmVhZGFibGUucHQnKSkKICAgICAgICByZXR1cm4gc3RhdGUKICAgIGlmIGZvdW5kOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignTm8gcmVhZGFibGUgcmVzdW1lIGNoZWNrcG9pbnQ6ICcrJzsgJy5qb2luKGVycm9ycykpCiAgICByZXR1cm4gTm9uZQoKCmRlZiBjYXB0dXJlX3JuZygpOgogICAgcmV0dXJuIGRpY3QocHl0aG9uPXJhbmRvbS5nZXRzdGF0ZSgpLCBudW1weT1ucC5yYW5kb20uZ2V0X3N0YXRlKCksCiAgICAgICAgICAgICAgICB0b3JjaD10b3JjaC5nZXRfcm5nX3N0YXRlKCksCiAgICAgICAgICAgICAgICBjdWRhPXRvcmNoLmN1ZGEuZ2V0X3JuZ19zdGF0ZV9hbGwoKSBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgW10pCgoKZGVmIHJlc3RvcmVfcm5nKHN0YXRlKToKICAgIHJhbmRvbS5zZXRzdGF0ZShzdGF0ZVsncHl0aG9uJ10pOyBucC5yYW5kb20uc2V0X3N0YXRlKHN0YXRlWydudW1weSddKQogICAgdG9yY2guc2V0X3JuZ19zdGF0ZShzdGF0ZVsndG9yY2gnXSkKICAgIGlmIHN0YXRlWydjdWRhJ106CiAgICAgICAgdG9yY2guY3VkYS5zZXRfcm5nX3N0YXRlX2FsbChzdGF0ZVsnY3VkYSddKQoKCmRlZiBlcG9jaF9vcmRlcihzaXplLCBzZWVkLCBlcG9jaCk6CiAgICByZXR1cm4gdG9yY2gucmFuZHBlcm0oc2l6ZSwgZ2VuZXJhdG9yPXRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKHNlZWQrZXBvY2gpKQoKCmNsYXNzIFJlc3VtZUJhdGNoU2FtcGxlcihTYW1wbGVyKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBvcmRlciwgYmF0Y2hfc2l6ZSwgc3RhcnRfYmF0Y2g9MCwgYWNjdW11bGF0aW9uPTEpOgogICAgICAgIHNlbGYub3JkZXIgPSB0b3JjaC5hc190ZW5zb3Iob3JkZXIsIGR0eXBlPXRvcmNoLmxvbmcpLmNwdSgpCiAgICAgICAgc2VsZi5iYXRjaF9zaXplLCBzZWxmLnN0YXJ0X2JhdGNoID0gYmF0Y2hfc2l6ZSwgc3RhcnRfYmF0Y2gKICAgICAgICB0b3RhbCA9IG1hdGguY2VpbChsZW4oc2VsZi5vcmRlcikvYmF0Y2hfc2l6ZSkKICAgICAgICBpZiBub3QgMCA8PSBzdGFydF9iYXRjaCA8PSB0b3RhbDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignSW52YWxpZCBzYXZlZCBiYXRjaCBjdXJzb3InKQogICAgICAgIGlmIHN0YXJ0X2JhdGNoICE9IHRvdGFsIGFuZCBzdGFydF9iYXRjaCAlIGFjY3VtdWxhdGlvbjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQ2Fubm90IHJlc3VtZSB3aXRoaW4gYSBncmFkaWVudC1hY2N1bXVsYXRpb24gZ3JvdXAnKQogICAgICAgIGlmIG5vdCB0b3JjaC5lcXVhbChzZWxmLm9yZGVyLnNvcnQoKS52YWx1ZXMsIHRvcmNoLmFyYW5nZShsZW4oc2VsZi5vcmRlcikpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignU2F2ZWQgaW1hZ2Ugb3JkZXIgaXMgbm90IGEgcGVybXV0YXRpb24gb2YgdGhlIGRhdGFzZXQnKQoKICAgIGRlZiBfX2l0ZXJfXyhzZWxmKToKICAgICAgICBmb3Igb2Zmc2V0IGluIHJhbmdlKHNlbGYuc3RhcnRfYmF0Y2gqc2VsZi5iYXRjaF9zaXplLCBsZW4oc2VsZi5vcmRlciksIHNlbGYuYmF0Y2hfc2l6ZSk6CiAgICAgICAgICAgIHlpZWxkIHNlbGYub3JkZXJbb2Zmc2V0Om9mZnNldCtzZWxmLmJhdGNoX3NpemVdLnRvbGlzdCgpCgogICAgZGVmIF9fbGVuX18oc2VsZik6CiAgICAgICAgcmV0dXJuIG1hdGguY2VpbChsZW4oc2VsZi5vcmRlcikvc2VsZi5iYXRjaF9zaXplKS1zZWxmLnN0YXJ0X2JhdGNoCgoKZGVmIHNuYXBzaG90KG1vZGVsLCBvcHRpbWl6ZXIsIHNjYWxlciwgZmluZ2VycHJpbnQsIGVwb2NoLCBjb21wbGV0ZSwgb3JkZXIsCiAgICAgICAgICAgICBwcm9ncmVzcywgYmVzdCwgaGlzdG9yeSk6CiAgICByZXR1cm4gZGljdChmb3JtYXRfdmVyc2lvbj0yLCBmaW5nZXJwcmludD1maW5nZXJwcmludCwgbW9kZWw9bW9kZWwuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgb3B0aW1pemVyPW9wdGltaXplci5zdGF0ZV9kaWN0KCksIHNjYWxlcj1zY2FsZXIuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgZXBvY2g9ZXBvY2gsIGVwb2NoX2NvbXBsZXRlPWNvbXBsZXRlLAogICAgICAgICAgICAgICAgb3JkZXI9Tm9uZSBpZiBjb21wbGV0ZSBlbHNlIG9yZGVyLmNsb25lKCkuY3B1KCksCiAgICAgICAgICAgICAgICBwcm9ncmVzcz1kaWN0KHByb2dyZXNzKSwgYmVzdD1iZXN0LCBoaXN0b3J5PWxpc3QoaGlzdG9yeSksIHJuZz1jYXB0dXJlX3JuZygpKQoKCmRlZiB0cmFpbl9lcG9jaChtb2RlbCwgbG9hZGVyLCBvcHRpbWl6ZXIsIHNjYWxlciwgY2ZnLCBkZXZpY2UsIGVwb2NoLAogICAgICAgICAgICAgICAgc2FtcGxlcywgcHJvZ3Jlc3MsIHRvdGFsX3N0ZXBzLCB3YXJtdXAsIGNoZWNrcG9pbnQpOgogICAgIiIiQ2hlY2twb2ludCBvbmx5IGF0IGNsZWFuIGFjY3VtdWxhdGlvbiBib3VuZGFyaWVzIGFmdGVyIHplcm9fZ3JhZC4KCiAgICBMb2FkZXIgUk5HIG11c3QgYmUgc2VwYXJhdGUgZnJvbSBtb2RlbCBSTkcuIFdvcmtlciB0cmFuc2Zvcm1zIG11c3Qgbm90IGRyYXcKICAgIHJhbmRvbW5lc3MuIEJvdGggY29uZGl0aW9ucyBhcmUgZW5mb3JjZWQgYnkgY29uc3RydWN0aW9uIGluIHRyYWluLnB5LgogICAgIiIiCiAgICBwcm9ncmVzcyA9IGRpY3QocHJvZ3Jlc3MpCiAgICB0b3RhbF9iYXRjaGVzID0gbWF0aC5jZWlsKHNhbXBsZXMvY2ZnWydiYXRjaF9zaXplJ10pCiAgICBzdGFydF9iYXRjaCA9IHByb2dyZXNzWyduZXh0X2JhdGNoJ10KICAgIGlmIHN0YXJ0X2JhdGNoID09IHRvdGFsX2JhdGNoZXM6CiAgICAgICAgcmV0dXJuIHByb2dyZXNzICAjIFRyYWluaW5nIGlzIGRvbmU7IHJlcGxheSB2YWxpZGF0aW9uIG9ubHkuCiAgICBtb2RlbC50cmFpbigpOyBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICBzdGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgYmFzZV9zZWNvbmRzID0gcHJvZ3Jlc3NbJ3NlY29uZHMnXQogICAgaWYgZGV2aWNlID09ICdjdWRhJzoKICAgICAgICB0b3JjaC5jdWRhLnJlc2V0X3BlYWtfbWVtb3J5X3N0YXRzKCkKICAgIGxhc3Rfc2F2ZWQgPSBzdGFydGVkCiAgICBldmVyeV91cGRhdGVzID0gY2ZnLmdldCgnY2hlY2twb2ludF9ldmVyeV91cGRhdGVzJywgMTAwKQogICAgZXZlcnlfc2Vjb25kcyA9IGNmZy5nZXQoJ2NoZWNrcG9pbnRfZXZlcnlfbWludXRlcycsIDE1KSo2MAogICAgaWYgZXZlcnlfdXBkYXRlcyA8IDAgb3IgZXZlcnlfc2Vjb25kcyA8IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQ2hlY2twb2ludCBpbnRlcnZhbHMgbXVzdCBiZSBub25uZWdhdGl2ZTsgemVybyBkaXNhYmxlcyBhbiBpbnRlcnZhbCcpCiAgICBmb3IgaSwgKGltYWdlcywgdmFsaWQsIF8pIGluIGVudW1lcmF0ZShsb2FkZXIsIHN0YXJ0PXN0YXJ0X2JhdGNoKToKICAgICAgICBpbWFnZXMsIHZhbGlkID0gaW1hZ2VzLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpLCB2YWxpZC50byhkZXZpY2UsIG5vbl9ibG9ja2luZz1UcnVlKQogICAgICAgIGdyb3VwX3N0YXJ0ID0gKGkvL2NmZ1snYWNjdW11bGF0aW9uJ10pKmNmZ1snYWNjdW11bGF0aW9uJ10KICAgICAgICBncm91cF9zYW1wbGVzID0gbWluKGNmZ1snYWNjdW11bGF0aW9uJ10qY2ZnWydiYXRjaF9zaXplJ10sIHNhbXBsZXMtZ3JvdXBfc3RhcnQqY2ZnWydiYXRjaF9zaXplJ10pCiAgICAgICAgc3RlcCA9IHByb2dyZXNzWydzdGVwJ10KICAgICAgICBmYWN0b3IgPSAoc3RlcCsxKS9tYXgoMSx3YXJtdXApIGlmIHN0ZXAgPCB3YXJtdXAgZWxzZSAuNSooMSttYXRoLmNvcyhtYXRoLnBpKihzdGVwLXdhcm11cCkvbWF4KDEsdG90YWxfc3RlcHMtd2FybXVwLTEpKSkKICAgICAgICBmb3IgZ3JvdXAgaW4gb3B0aW1pemVyLnBhcmFtX2dyb3VwczoKICAgICAgICAgICAgZ3JvdXBbJ2xyJ10gPSBjZmdbJ2xlYXJuaW5nX3JhdGUnXSpmYWN0b3IKICAgICAgICB3aXRoIHRvcmNoLmF1dG9jYXN0KGRldmljZSwgZHR5cGU9dG9yY2guZmxvYXQxNiBpZiBkZXZpY2UgPT0gJ2N1ZGEnIGVsc2UgdG9yY2guYmZsb2F0MTYsIGVuYWJsZWQ9Y2ZnWydhbXAnXSk6CiAgICAgICAgICAgIGxvc3MsIF8sIF8gPSBtb2RlbChpbWFnZXMsIHZhbGlkKQogICAgICAgIGlmIG5vdCB0b3JjaC5pc2Zpbml0ZShsb3NzKToKICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKGYnTm9uZmluaXRlIGxvc3MgZXBvY2g9e2Vwb2NofSBiYXRjaD17aX0nKQogICAgICAgIHNjYWxlci5zY2FsZShsb3NzKmxlbihpbWFnZXMpL2dyb3VwX3NhbXBsZXMpLmJhY2t3YXJkKCkKICAgICAgICBwcm9ncmVzc1snbG9zc19zdW0nXSArPSBsb3NzLml0ZW0oKSpsZW4oaW1hZ2VzKQogICAgICAgIHByb2dyZXNzWydzZWVuJ10gKz0gbGVuKGltYWdlcykKICAgICAgICBib3VuZGFyeSA9IChpKzEpJWNmZ1snYWNjdW11bGF0aW9uJ10gPT0gMCBvciBpKzEgPT0gdG90YWxfYmF0Y2hlcwogICAgICAgIGlmIGJvdW5kYXJ5OgogICAgICAgICAgICBzY2FsZXIudW5zY2FsZV8ob3B0aW1pemVyKQogICAgICAgICAgICBub3JtID0gdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgY2ZnWydjbGlwX2dyYWQnXSkKICAgICAgICAgICAgaWYgbm90IHRvcmNoLmlzZmluaXRlKG5vcm0pOgogICAgICAgICAgICAgICAgaWYgbm90IGNmZ1snYW1wJ106CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9yKCdOb25maW5pdGUgZ3JhZGllbnRzIHdpdGggQU1QIGRpc2FibGVkOyByZWR1Y2UgTFIgaW4gYSBuZXcgcnVuJykKICAgICAgICAgICAgICAgICMgR3JhZFNjYWxlciByZWNvcmRzIG5vbmZpbml0ZSBncmFkaWVudHMgZHVyaW5nIHVuc2NhbGVfLiBMZXQgaXQKICAgICAgICAgICAgICAgICMgc2tpcCB0aGUgb3B0aW1pemVyIHVwZGF0ZSBhbmQgcmVkdWNlIGl0cyBzY2FsZSBpbnN0ZWFkIG9mCiAgICAgICAgICAgICAgICAjIHR1cm5pbmcgYW4gZXhwZWN0ZWQsIHJlY292ZXJhYmxlIEZQMTYgb3ZlcmZsb3cgaW50byBhIGZhdGFsCiAgICAgICAgICAgICAgICAjIHRyYWluaW5nIGVycm9yLgogICAgICAgICAgICAgICAgb2xkX3NjYWxlID0gc2NhbGVyLmdldF9zY2FsZSgpCiAgICAgICAgICAgICAgICBzY2FsZXIuc3RlcChvcHRpbWl6ZXIpOyBzY2FsZXIudXBkYXRlKCk7IG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICAgICAgICAgIG5ld19zY2FsZSA9IHNjYWxlci5nZXRfc2NhbGUoKQogICAgICAgICAgICAgICAgaWYgbm90IG5ld19zY2FsZSA8IG9sZF9zY2FsZToKICAgICAgICAgICAgICAgICAgICByYWlzZSBGbG9hdGluZ1BvaW50RXJyb3IoJ05vbmZpbml0ZSBBTVAgZ3JhZGllbnRzIGJ1dCBHcmFkU2NhbGVyIGRpZCBub3QgcmVkdWNlIGl0cyBzY2FsZScpCiAgICAgICAgICAgICAgICBwcm9ncmVzc1snYW1wX3NraXBwZWRfdXBkYXRlcyddID0gcHJvZ3Jlc3MuZ2V0KCdhbXBfc2tpcHBlZF91cGRhdGVzJywgMCkgKyAxCiAgICAgICAgICAgICAgICBwcm9ncmVzc1snbmV4dF9iYXRjaCddID0gaSsxCiAgICAgICAgICAgICAgICBwcm9ncmVzc1snc2Vjb25kcyddID0gYmFzZV9zZWNvbmRzICsgdGltZS5tb25vdG9uaWMoKS1zdGFydGVkCiAgICAgICAgICAgICAgICBpZiBkZXZpY2UgPT0gJ2N1ZGEnOgogICAgICAgICAgICAgICAgICAgIHByb2dyZXNzWydwZWFrX2dwdV9naWInXSA9IG1heChwcm9ncmVzc1sncGVha19ncHVfZ2liJ10sIHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKS8yKiozMCkKICAgICAgICAgICAgICAgIHByaW50KGYiU2tpcHBlZCBub25maW5pdGUgQU1QIHVwZGF0ZSBhdCBlcG9jaCB7ZXBvY2grMX0sIGJhdGNoIHtpKzF9OyAiCiAgICAgICAgICAgICAgICAgICAgICBmInNjYWxlIHtvbGRfc2NhbGU6Z30gLT4ge25ld19zY2FsZTpnfSIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICAjIFBlcnNpc3QgdGhlIHJlZHVjZWQgc2NhbGUgYW5kIGNvbnN1bWVkLWJhdGNoIGN1cnNvcgogICAgICAgICAgICAgICAgIyBpbW1lZGlhdGVseSBzbyBhIGRpc2Nvbm5lY3QgY2Fubm90IHJlY3JlYXRlIHRoZSBvdmVyZmxvdy4KICAgICAgICAgICAgICAgIGNoZWNrcG9pbnQocHJvZ3Jlc3MpCiAgICAgICAgICAgICAgICBsYXN0X3NhdmVkID0gdGltZS5tb25vdG9uaWMoKQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgc2NhbGVyLnN0ZXAob3B0aW1pemVyKTsgc2NhbGVyLnVwZGF0ZSgpOyBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgICAgIHByb2dyZXNzWydzdGVwJ10gKz0gMQogICAgICAgICAgICBwcm9ncmVzc1snbmV4dF9iYXRjaCddID0gaSsxCiAgICAgICAgICAgIHByb2dyZXNzWydzZWNvbmRzJ10gPSBiYXNlX3NlY29uZHMgKyB0aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQKICAgICAgICAgICAgaWYgZGV2aWNlID09ICdjdWRhJzoKICAgICAgICAgICAgICAgIHByb2dyZXNzWydwZWFrX2dwdV9naWInXSA9IG1heChwcm9ncmVzc1sncGVha19ncHVfZ2liJ10sIHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKS8yKiozMCkKICAgICAgICAgICAgZHVlID0gKChldmVyeV91cGRhdGVzIGFuZCBwcm9ncmVzc1snc3RlcCddJWV2ZXJ5X3VwZGF0ZXMgPT0gMCkKICAgICAgICAgICAgICAgICAgIG9yIChldmVyeV9zZWNvbmRzIGFuZCB0aW1lLm1vbm90b25pYygpLWxhc3Rfc2F2ZWQgPj0gZXZlcnlfc2Vjb25kcykKICAgICAgICAgICAgICAgICAgIG9yIGkrMSA9PSB0b3RhbF9iYXRjaGVzKQogICAgICAgICAgICBpZiBkdWU6CiAgICAgICAgICAgICAgICBjaGVja3BvaW50KHByb2dyZXNzKQogICAgICAgICAgICAgICAgbGFzdF9zYXZlZCA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICBpZiBpJTIwID09IDA6CiAgICAgICAgICAgIHByaW50KGYnZXBvY2gge2Vwb2NoKzF9IGJhdGNoIHtpKzF9L3t0b3RhbF9iYXRjaGVzfSBsb3NzPXtsb3NzLml0ZW0oKTouNWZ9JywgZmx1c2g9VHJ1ZSkKICAgIGFzc2VydCBwcm9ncmVzc1snbmV4dF9iYXRjaCddID09IHRvdGFsX2JhdGNoZXMgYW5kIHByb2dyZXNzWydzZWVuJ10gPT0gc2FtcGxlcwogICAgcHJvZ3Jlc3NbJ3NlY29uZHMnXSA9IGJhc2Vfc2Vjb25kcyt0aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQKICAgIHJldHVybiBwcm9ncmVzcwo=', 'test_training_state.py': 'IiIiQ1BVIGludGVycnVwdGVkLXZzLXVuaW50ZXJydXB0ZWQgdGVzdHMgb2YgdGhlIHByb2R1Y3Rpb24gZXBvY2ggbG9vcC4iIiIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBvcwppbXBvcnQgcmFuZG9tCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdW5pdHRlc3QKZnJvbSB1bml0dGVzdC5tb2NrIGltcG9ydCBwYXRjaAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQsIERhdGFMb2FkZXIKZnJvbSB0cmFpbmluZ19zdGF0ZSBpbXBvcnQgKFJlc3VtZUJhdGNoU2FtcGxlciwgZXBvY2hfb3JkZXIsIHNuYXBzaG90LCBzYXZlX2F0b21pYywKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxvYWRfcmVzdW1lLCByZXN0b3JlX3JuZywgdHJhaW5fZXBvY2gpCgoKY2xhc3MgUmVjb3JkcyhEYXRhc2V0KToKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiAyMwoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpbmRleCk6CiAgICAgICAgcmV0dXJuIHRvcmNoLnRlbnNvcihbaW5kZXgvMjMuLCAuMywgLS4yLCAuN10pLCB0b3JjaC5vbmVzKDQpLCBzdHIoaW5kZXgpCgoKY2xhc3MgTWFza2VkTW9kZWwodG9yY2gubm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm5ldCA9IHRvcmNoLm5uLlNlcXVlbnRpYWwodG9yY2gubm4uTGluZWFyKDQsIDkpLCB0b3JjaC5ubi5Ecm9wb3V0KC4yKSwgdG9yY2gubm4uTGluZWFyKDksIDQpKQogICAgICAgIHNlbGYudmlzaXRlZCA9IFtdCgogICAgZGVmIGZvcndhcmQoc2VsZiwgaW1hZ2VzLCB2YWxpZCk6CiAgICAgICAgc2VsZi52aXNpdGVkLmV4dGVuZChyb3VuZChmbG9hdCh4KSoyMykgZm9yIHggaW4gaW1hZ2VzWzosIDBdKQogICAgICAgIG1hc2sgPSAodG9yY2gucmFuZF9saWtlKGltYWdlcykgPiAuNCkuZmxvYXQoKQogICAgICAgICMgRXhlcmNpc2UgYWxsIHNhdmVkIFJORyBzdHJlYW1zLCBhcyB3ZWxsIGFzIGRyb3BvdXQgYW5kIHJhbmRvbSBtYXNrcy4KICAgICAgICB0YXJnZXQgPSBpbWFnZXMgKiAocmFuZG9tLnJhbmRvbSgpK2Zsb2F0KG5wLnJhbmRvbS5yYW5kb20oKSkpCiAgICAgICAgbG9zcyA9IChzZWxmLm5ldChpbWFnZXMqKDEtbWFzaykpLXRhcmdldCkuc3F1YXJlKCkubWVhbigpCiAgICAgICAgcmV0dXJuIGxvc3MsIE5vbmUsIG1hc2sKCgpkZWYgc2VlZF9hbGwoKToKICAgIHJhbmRvbS5zZWVkKDgxKTsgbnAucmFuZG9tLnNlZWQoODEpOyB0b3JjaC5tYW51YWxfc2VlZCg4MSkKCgpkZWYgbWFrZV9vYmplY3RzKCk6CiAgICBtb2RlbCA9IE1hc2tlZE1vZGVsKCkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW1XKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9LjAwMSkKICAgIHNjYWxlciA9IHRvcmNoLmN1ZGEuYW1wLkdyYWRTY2FsZXIoZW5hYmxlZD1GYWxzZSkKICAgIHJldHVybiBtb2RlbCwgb3B0aW1pemVyLCBzY2FsZXIKCgpkZWYgbG9hZGVyKG9yZGVyLCBzdGFydCwgd29ya2Vycyk6CiAgICByZXR1cm4gRGF0YUxvYWRlcihSZWNvcmRzKCksIGJhdGNoX3NhbXBsZXI9UmVzdW1lQmF0Y2hTYW1wbGVyKG9yZGVyLCAyLCBzdGFydCwgMyksCiAgICAgICAgICAgICAgICAgICAgICBudW1fd29ya2Vycz13b3JrZXJzLCBnZW5lcmF0b3I9dG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoNTAwKSkKCgpDRkcgPSBkaWN0KGJhdGNoX3NpemU9MiwgYWNjdW11bGF0aW9uPTMsIGxlYXJuaW5nX3JhdGU9LjAwMSwgYW1wPUZhbHNlLCBjbGlwX2dyYWQ9MS4sCiAgICAgICAgICAgY2hlY2twb2ludF9ldmVyeV91cGRhdGVzPTEsIGNoZWNrcG9pbnRfZXZlcnlfbWludXRlcz0wKQoKCmRlZiBpbml0aWFsX3Byb2dyZXNzKCk6CiAgICByZXR1cm4gZGljdChuZXh0X2JhdGNoPTAsIHN0ZXA9MCwgbG9zc19zdW09MC4sIHNlZW49MCwgc2Vjb25kcz0wLiwgcGVha19ncHVfZ2liPTAuKQoKCmNsYXNzIEludGVycnVwdGVkKEV4Y2VwdGlvbik6CiAgICBwYXNzCgoKY2xhc3MgT3ZlcmZsb3dPbmNlU2NhbGVyOgogICAgIiIiU21hbGwgQ1BVIHRlc3QgZG91YmxlIGZvciBHcmFkU2NhbGVyJ3Mgc2tpcC1hbmQtYmFja29mZiBiZWhhdmlvci4iIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBzZWxmLnZhbHVlID0gOC4KICAgICAgICBzZWxmLmNhbGxzID0gMAoKICAgIGRlZiBzY2FsZShzZWxmLCBsb3NzKToKICAgICAgICByZXR1cm4gbG9zcwoKICAgIGRlZiB1bnNjYWxlXyhzZWxmLCBvcHRpbWl6ZXIpOgogICAgICAgIHBhc3MKCiAgICBkZWYgZ2V0X3NjYWxlKHNlbGYpOgogICAgICAgIHJldHVybiBzZWxmLnZhbHVlCgogICAgZGVmIHN0ZXAoc2VsZiwgb3B0aW1pemVyKToKICAgICAgICBzZWxmLmNhbGxzICs9IDEKICAgICAgICBpZiBzZWxmLmNhbGxzID4gMToKICAgICAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQoKICAgIGRlZiB1cGRhdGUoc2VsZik6CiAgICAgICAgaWYgc2VsZi5jYWxscyA9PSAxOgogICAgICAgICAgICBzZWxmLnZhbHVlICo9IC41CgoKY2xhc3MgUmVzdW1lVGVzdHModW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIGFzc2VydF9uZXN0ZWRfZXF1YWwoc2VsZiwgYSwgYik6CiAgICAgICAgaWYgaXNpbnN0YW5jZShhLCB0b3JjaC5UZW5zb3IpOgogICAgICAgICAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZShhLCBiLCBydG9sPTAsIGF0b2w9MCkKICAgICAgICBlbGlmIGlzaW5zdGFuY2UoYSwgZGljdCk6CiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoYS5rZXlzKCksIGIua2V5cygpKQogICAgICAgICAgICBmb3Iga2V5IGluIGE6CiAgICAgICAgICAgICAgICBzZWxmLmFzc2VydF9uZXN0ZWRfZXF1YWwoYVtrZXldLCBiW2tleV0pCiAgICAgICAgZWxpZiBpc2luc3RhbmNlKGEsIChsaXN0LCB0dXBsZSkpOgogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGxlbihhKSwgbGVuKGIpKQogICAgICAgICAgICBmb3IgeCwgeSBpbiB6aXAoYSwgYik6CiAgICAgICAgICAgICAgICBzZWxmLmFzc2VydF9uZXN0ZWRfZXF1YWwoeCwgeSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGEsIGIpCgogICAgZGVmIHJ1bl9pbnRlcnJ1cHRpb25fY2FzZShzZWxmLCBzdG9wX3VwZGF0ZSwgd29ya2Vycyk6CiAgICAgICAgb3JkZXIgPSBlcG9jaF9vcmRlcigyMywgNDIsIDApCiAgICAgICAgc2VlZF9hbGwoKTsgcmVmZXJlbmNlLCByb3B0LCByc2NhbGUgPSBtYWtlX29iamVjdHMoKQogICAgICAgIGV4cGVjdGVkID0gdHJhaW5fZXBvY2gocmVmZXJlbmNlLCBsb2FkZXIob3JkZXIsMCx3b3JrZXJzKSwgcm9wdCwgcnNjYWxlLCBDRkcsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnY3B1JywwLDIzLGluaXRpYWxfcHJvZ3Jlc3MoKSwxMCwyLGxhbWJkYSBwOiBOb25lKQogICAgICAgIGV4cGVjdGVkX25leHRfZHJhdyA9IHRvcmNoLnJhbmQoNSkKICAgICAgICBzZWVkX2FsbCgpOyBtb2RlbCwgb3B0LCBzY2FsZSA9IG1ha2Vfb2JqZWN0cygpCiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXA6CiAgICAgICAgICAgIHBhdGggPSBQYXRoKHRtcCkvJ2xhc3QucHQnCiAgICAgICAgICAgIGRlZiBzYXZlKHByb2dyZXNzKToKICAgICAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShhbGwocC5ncmFkIGlzIE5vbmUgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKSkKICAgICAgICAgICAgICAgIHNhdmVfYXRvbWljKHNuYXBzaG90KG1vZGVsLG9wdCxzY2FsZSwndGVzdCcsMCxGYWxzZSxvcmRlcixwcm9ncmVzcyxmbG9hdCgnaW5mJyksW10pLCBwYXRoLCBUcnVlKQogICAgICAgICAgICAgICAgaWYgcHJvZ3Jlc3NbJ3N0ZXAnXSA9PSBzdG9wX3VwZGF0ZToKICAgICAgICAgICAgICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZCgpCiAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoSW50ZXJydXB0ZWQpOgogICAgICAgICAgICAgICAgdHJhaW5fZXBvY2gobW9kZWwsbG9hZGVyKG9yZGVyLDAsd29ya2Vycyksb3B0LHNjYWxlLENGRywnY3B1JywwLDIzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgaW5pdGlhbF9wcm9ncmVzcygpLDEwLDIsc2F2ZSkKICAgICAgICAgICAgZmlyc3RfdmlzaXRzID0gbGlzdChtb2RlbC52aXNpdGVkKQogICAgICAgICAgICAjIFNpbXVsYXRlIGEgY29tcGxldGVseSBuZXcgcHJvY2VzcyBjb25zdHJ1Y3RpbmcgbW9kZWwgYW5kIGxvYWRlcnMuCiAgICAgICAgICAgIHNlZWRfYWxsKCk7IG1vZGVsLCBvcHQsIHNjYWxlID0gbWFrZV9vYmplY3RzKCkKICAgICAgICAgICAgc2F2ZWQgPSBsb2FkX3Jlc3VtZShwYXRoLCd0ZXN0JykKICAgICAgICAgICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHNhdmVkWydtb2RlbCddKTsgb3B0LmxvYWRfc3RhdGVfZGljdChzYXZlZFsnb3B0aW1pemVyJ10pCiAgICAgICAgICAgIHNjYWxlLmxvYWRfc3RhdGVfZGljdChzYXZlZFsnc2NhbGVyJ10pOyByZXN0b3JlX3JuZyhzYXZlZFsncm5nJ10pCiAgICAgICAgICAgIHJlc3VtZWRfbG9hZGVyID0gbG9hZGVyKHNhdmVkWydvcmRlciddLHNhdmVkWydwcm9ncmVzcyddWyduZXh0X2JhdGNoJ10sd29ya2VycykKICAgICAgICAgICAgYWN0dWFsID0gdHJhaW5fZXBvY2gobW9kZWwscmVzdW1lZF9sb2FkZXIsb3B0LHNjYWxlLENGRywnY3B1JywwLDIzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzYXZlZFsncHJvZ3Jlc3MnXSwxMCwyLGxhbWJkYSBwOiBOb25lKQogICAgICAgIHNlbGYuYXNzZXJ0X25lc3RlZF9lcXVhbChyZWZlcmVuY2Uuc3RhdGVfZGljdCgpLG1vZGVsLnN0YXRlX2RpY3QoKSkKICAgICAgICBzZWxmLmFzc2VydF9uZXN0ZWRfZXF1YWwocm9wdC5zdGF0ZV9kaWN0KCksb3B0LnN0YXRlX2RpY3QoKSkKICAgICAgICB0b3JjaC50ZXN0aW5nLmFzc2VydF9jbG9zZSh0b3JjaC5yYW5kKDUpLGV4cGVjdGVkX25leHRfZHJhdyxydG9sPTAsYXRvbD0wKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZmlyc3RfdmlzaXRzK21vZGVsLnZpc2l0ZWQscmVmZXJlbmNlLnZpc2l0ZWQpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChyZWZlcmVuY2UudmlzaXRlZCxvcmRlci50b2xpc3QoKSkKICAgICAgICBmb3Iga2V5IGluIFsnc3RlcCcsJ25leHRfYmF0Y2gnLCdsb3NzX3N1bScsJ3NlZW4nXToKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChhY3R1YWxba2V5XSxleHBlY3RlZFtrZXldKQoKICAgIGRlZiB0ZXN0X21pZF9lcG9jaF93aXRoX3ByZWZldGNoKHNlbGYpOgogICAgICAgIHNlbGYucnVuX2ludGVycnVwdGlvbl9jYXNlKHN0b3BfdXBkYXRlPTIsd29ya2Vycz0yKQoKICAgIGRlZiB0ZXN0X3Nob3J0X2ZpbmFsX2FjY3VtdWxhdGlvbl9ncm91cChzZWxmKToKICAgICAgICBzZWxmLnJ1bl9pbnRlcnJ1cHRpb25fY2FzZShzdG9wX3VwZGF0ZT0zLHdvcmtlcnM9MCkKCiAgICBkZWYgdGVzdF90cmFpbmluZ19kb25lX3ZhbGlkYXRpb25fcGVuZGluZyhzZWxmKToKICAgICAgICBzZWxmLnJ1bl9pbnRlcnJ1cHRpb25fY2FzZShzdG9wX3VwZGF0ZT00LHdvcmtlcnM9MikKCiAgICBkZWYgdGVzdF90aW1lX3RyaWdnZXJfb25seV9hdF91cGRhdGVfYm91bmRhcmllcyhzZWxmKToKICAgICAgICBzZWVkX2FsbCgpOyBtb2RlbCxvcHQsc2NhbGU9bWFrZV9vYmplY3RzKCk7IHNhdmVkPVtdCiAgICAgICAgY2ZnID0gZGljdChDRkcsY2hlY2twb2ludF9ldmVyeV91cGRhdGVzPTAsY2hlY2twb2ludF9ldmVyeV9taW51dGVzPTFlLTEyKQogICAgICAgIHRyYWluX2Vwb2NoKG1vZGVsLGxvYWRlcihlcG9jaF9vcmRlcigyMyw0MiwwKSwwLDApLG9wdCxzY2FsZSxjZmcsJ2NwdScsMCwyMywKICAgICAgICAgICAgICAgICAgICBpbml0aWFsX3Byb2dyZXNzKCksMTAsMixsYW1iZGEgcDogc2F2ZWQuYXBwZW5kKHBbJ25leHRfYmF0Y2gnXSkpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChzYXZlZCxbMyw2LDksMTJdKQoKICAgIGRlZiB0ZXN0X2FtcF9vdmVyZmxvd19za2lwc191cGRhdGVfYmFja3Nfb2ZmX2FuZF9hZHZhbmNlc19jdXJzb3Ioc2VsZik6CiAgICAgICAgc2VlZF9hbGwoKTsgbW9kZWwsb3B0aW1pemVyLF89bWFrZV9vYmplY3RzKCk7IHNjYWxlcj1PdmVyZmxvd09uY2VTY2FsZXIoKTsgc2F2ZWQ9W10KICAgICAgICBjZmc9ZGljdChDRkcsYW1wPVRydWUsY2hlY2twb2ludF9ldmVyeV91cGRhdGVzPTAsY2hlY2twb2ludF9ldmVyeV9taW51dGVzPTApCiAgICAgICAgcmVhbF9jbGlwPXRvcmNoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXwogICAgICAgIGNhbGxzPWl0ZXIoW3RvcmNoLnRlbnNvcihmbG9hdCgnaW5mJykpXStbTm9uZV0qMykKICAgICAgICBkZWYgb3ZlcmZsb3dfb25jZShwYXJhbWV0ZXJzLG1heF9ub3JtKToKICAgICAgICAgICAgcmVzdWx0PW5leHQoY2FsbHMpCiAgICAgICAgICAgIHJldHVybiByZWFsX2NsaXAocGFyYW1ldGVycyxtYXhfbm9ybSkgaWYgcmVzdWx0IGlzIE5vbmUgZWxzZSByZXN1bHQKICAgICAgICB3aXRoIHBhdGNoKCd0cmFpbmluZ19zdGF0ZS50b3JjaC5hdXRvY2FzdCcpIGFzIGF1dG9jYXN0LCBcCiAgICAgICAgICAgICBwYXRjaCgndHJhaW5pbmdfc3RhdGUudG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fJyxzaWRlX2VmZmVjdD1vdmVyZmxvd19vbmNlKToKICAgICAgICAgICAgYXV0b2Nhc3QucmV0dXJuX3ZhbHVlLl9fZW50ZXJfXy5yZXR1cm5fdmFsdWU9Tm9uZQogICAgICAgICAgICBwcm9ncmVzcz10cmFpbl9lcG9jaChtb2RlbCxsb2FkZXIoZXBvY2hfb3JkZXIoMjMsNDIsMCksMCwwKSxvcHRpbWl6ZXIsc2NhbGVyLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjZmcsJ2NwdScsMCwyMyxpbml0aWFsX3Byb2dyZXNzKCksMTAsMiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbGFtYmRhIHA6c2F2ZWQuYXBwZW5kKGRpY3QocCkpKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocHJvZ3Jlc3NbJ25leHRfYmF0Y2gnXSwxMikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByb2dyZXNzWydzdGVwJ10sMykKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHByb2dyZXNzWydhbXBfc2tpcHBlZF91cGRhdGVzJ10sMSkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNjYWxlci5nZXRfc2NhbGUoKSw0LikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNhdmVkWzBdWyduZXh0X2JhdGNoJ10sMykKCiAgICBkZWYgdGVzdF9iYWRfY3Vyc29yX3JlamVjdGVkKHNlbGYpOgogICAgICAgIHdpdGggc2VsZi5hc3NlcnRSYWlzZXMoVmFsdWVFcnJvcik6CiAgICAgICAgICAgIFJlc3VtZUJhdGNoU2FtcGxlcihlcG9jaF9vcmRlcigyMyw0MiwwKSwyLDEsMykKCiAgICBkZWYgdGVzdF9mYWlsZWRfc2F2ZV9wcmVzZXJ2ZXNfcmVjb3ZlcnlfYW5kX2JhZF9sYXRlc3RfZmFsbHNfYmFjayhzZWxmKToKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRtcDoKICAgICAgICAgICAgcGF0aD1QYXRoKHRtcCkvJ2xhc3QucHQnCiAgICAgICAgICAgIG9sZD1kaWN0KGZvcm1hdF92ZXJzaW9uPTIsZmluZ2VycHJpbnQ9J3Rlc3QnLHZhbHVlPTEpCiAgICAgICAgICAgIG5ldz1kaWN0KG9sZCx2YWx1ZT0yKQogICAgICAgICAgICBzYXZlX2F0b21pYyhvbGQscGF0aCxUcnVlKQogICAgICAgICAgICB3aXRoIHBhdGNoKCd0cmFpbmluZ19zdGF0ZS50b3JjaC5zYXZlJyxzaWRlX2VmZmVjdD1PU0Vycm9yKCdkaXNrIGZ1bGwnKSk6CiAgICAgICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKE9TRXJyb3IpOgogICAgICAgICAgICAgICAgICAgIHNhdmVfYXRvbWljKG5ldyxwYXRoLFRydWUpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwobG9hZF9yZXN1bWUocGF0aCwndGVzdCcpWyd2YWx1ZSddLDEpCiAgICAgICAgICAgIHNhdmVfYXRvbWljKG5ldyxwYXRoLFRydWUpCiAgICAgICAgICAgIHBhdGgud3JpdGVfYnl0ZXMoYidpbnRlcnJ1cHRlZC9jb3JydXB0JykKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsb2FkX3Jlc3VtZShwYXRoLCd0ZXN0JylbJ3ZhbHVlJ10sMSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShwYXRoLmV4aXN0cygpKSAgIyBjb3JydXB0IGxhc3QgY2Fubm90IHJlcGxhY2UgZ29vZCBwcmV2aW91cwogICAgICAgICAgICBzYXZlX2F0b21pYyhuZXcscGF0aCxUcnVlKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKGxvYWRfcmVzdW1lKHBhdGgsJ3Rlc3QnKVsndmFsdWUnXSwyKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHRvcmNoLmxvYWQocGF0aC53aXRoX25hbWUoJ3ByZXZpb3VzLnB0Jyksd2VpZ2h0c19vbmx5PUZhbHNlKVsndmFsdWUnXSwxKQogICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0UmFpc2VzKFZhbHVlRXJyb3IpOgogICAgICAgICAgICAgICAgbG9hZF9yZXN1bWUocGF0aCwnZGlmZmVyZW50IGV4cGVyaW1lbnQnKQoKICAgIGRlZiB0ZXN0X3B1Ymxpc2hfZmFpbHVyZV9hZnRlcl9yb3RhdGlvbl9yZWNvdmVyc19wcmV2aW91cyhzZWxmKToKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRtcDoKICAgICAgICAgICAgcGF0aD1QYXRoKHRtcCkvJ2xhc3QucHQnCiAgICAgICAgICAgIHN0YXRlPWRpY3QoZm9ybWF0X3ZlcnNpb249MixmaW5nZXJwcmludD0ndGVzdCcsdmFsdWU9MSkKICAgICAgICAgICAgc2F2ZV9hdG9taWMoc3RhdGUscGF0aCxUcnVlKQogICAgICAgICAgICByZXBsYWNlPW9zLnJlcGxhY2UKICAgICAgICAgICAgZGVmIGZhaWxfcHVibGlzaChzb3VyY2UsdGFyZ2V0KToKICAgICAgICAgICAgICAgIGlmIHN0cihzb3VyY2UpLmVuZHN3aXRoKCcudG1wJyk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgT1NFcnJvcignZGlzY29ubmVjdGVkIGR1cmluZyBmaW5hbCByZW5hbWUnKQogICAgICAgICAgICAgICAgcmVwbGFjZShzb3VyY2UsdGFyZ2V0KQogICAgICAgICAgICB3aXRoIHBhdGNoKCd0cmFpbmluZ19zdGF0ZS5vcy5yZXBsYWNlJyxzaWRlX2VmZmVjdD1mYWlsX3B1Ymxpc2gpOgogICAgICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhPU0Vycm9yKToKICAgICAgICAgICAgICAgICAgICBzYXZlX2F0b21pYyhkaWN0KHN0YXRlLHZhbHVlPTIpLHBhdGgsVHJ1ZSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsb2FkX3Jlc3VtZShwYXRoLCd0ZXN0JylbJ3ZhbHVlJ10sMSkKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgdG9yY2guc2V0X251bV90aHJlYWRzKDIpCiAgICB1bml0dGVzdC5tYWluKCkK', 'benchmark.py': 'IiIiTWVhc3VyZSB0aGUgYXNzaWduZWQgR1BVIHVzaW5nIHRoZSByZWFsIEZDTUFFL2RhdGEvb3B0aW1pemVyIHBpcGVsaW5lLgoKVGhlIG1vZGVsIGlzIGRpc3Bvc2FibGUuIFRoaXMgY29tbWFuZCBuZXZlciByZWFkcyBvciBjaGFuZ2VzIHByb2R1Y3Rpb24gcmVzdW1lCmNoZWNrcG9pbnRzLiBPbmx5IGxpZ2h0d2VpZ2h0IHJlcG9ydHMgcmVtYWluIGluIGl0cyB1bmlxdWUgYmVuY2htYXJrIGRpcmVjdG9yeS4KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQgaXRlcnRvb2xzCmltcG9ydCBpbXBvcnRsaWIubWV0YWRhdGEKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCByYW5kb20KaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB0aW1lCmltcG9ydCB0cmFjZWJhY2sKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCgpmcm9tIGJlbmNobWFya19yZXBvcnQgaW1wb3J0IHByb2plY3Rpb25zLCB3cml0ZV9yZXBvcnRzCmZyb20gZGF0YV91dGlscyBpbXBvcnQgRUNHRGF0YXNldCwgcHJlcGFyZV9zcGxpdHMsIHNoYTI1Ngpmcm9tIHRyYWluaW5nX3N0YXRlIGltcG9ydCB0cmFpbl9lcG9jaCwgc25hcHNob3QsIHNhdmVfYXRvbWljCgoKZGVmIGJlbmNobWFyayhjZmcsIG9wdGlvbnMsIGRpcmVjdG9yeSk6CiAgICBkaXJlY3Rvcnk9UGF0aChkaXJlY3RvcnkpOyBkaXJlY3RvcnkubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBzdGFydGVkPXRpbWUucGVyZl9jb3VudGVyKCkKICAgIHJlcG9ydD1kaWN0KHN0YXR1cz0ncnVubmluZycsdW5pdHNfcGVyX2hvdXI9b3B0aW9ucy5nZXQoJ3VuaXRzX3Blcl9ob3VyJyksCiAgICAgICAgICAgICAgICBjb25maWd1cmF0aW9uPWNmZyxiZW5jaG1hcmtfb3B0aW9ucz1vcHRpb25zLG5vdGVzPVsKICAgICAgICAnTWVhc3VyZW1lbnRzIGFwcGx5IG9ubHkgdG8gdGhlIEdQVSBhbGxvY2F0ZWQgdG8gdGhpcyBzZXNzaW9uOyBubyBleHRyYXBvbGF0aW9uIGJldHdlZW4gR1BVIG1vZGVscy4nLAogICAgICAgICdCZW5jaG1hcmtzIHVzZSByZWFsIEVDRyBpbWFnZXMsIHNwYXJzZSBGQ01BRSwgYmFja3dhcmQgcGFzc2VzLCBBZGFtVyBhbmQgdGhlIGNvbmZpZ3VyZWQgYWNjdW11bGF0aW9uL3ByZWNpc2lvbi4nLAogICAgICAgICdSZXBlYXRlZCB0cmFpbmluZy92YWxpZGF0aW9uIHRpbWluZ3MgdXNlIGEgcG9wdWxhdGVkIHJlc2l6ZWQtaW1hZ2UgY2FjaGUgYW5kIHBlcnNpc3RlbnQgd29ya2Vycy4nLAogICAgICAgICdQcm9kdWN0aW9uIHJlc3RhcnRzIGxvYWRlciB3b3JrZXJzIGVhY2ggZXBvY2g7IHRoYXQgc21hbGwgcmVjdXJyaW5nIHN0YXJ0dXAgb3ZlcmhlYWQgaXMgbm90IGV4dHJhcG9sYXRlZCBmcm9tIHRoZXNlIHdhcm0td29ya2VyIG1lYXN1cmVtZW50cy4nLAogICAgICAgICdGYXN0L3Nsb3cgc2NlbmFyaW9zIGFyZSBvYnNlcnZlZCByb3VuZCBleHRyZW1hLCBub3QgY29uZmlkZW5jZSBpbnRlcnZhbHMgb3IgZ3VhcmFudGVlZCBydW50aW1lIGJvdW5kcy4nLAogICAgICAgICdGaXJzdC1wYXNzIGNhY2hlIHByZXBhcmF0aW9uIGlzIG1lYXN1cmVkIG9uIGEgc2FtcGxlIGFuZCBleHRyYXBvbGF0ZWQgc2VwYXJhdGVseTsgYWRkaW5nIGl0IHNlcmlhbGx5IGlzIGNvbnNlcnZhdGl2ZSBiZWNhdXNlIHJlYWwgdHJhaW5pbmcgY2FuIG92ZXJsYXAgcHJlcHJvY2Vzc2luZy4nLAogICAgICAgICdQbGFubmluZyB0b3RhbHMgaW5jbHVkZSBvbmUgcmVjb3JkZWQgcHJlcGFyYXRpb24gc2Vzc2lvbiBhbmQgb25lIGV4dHJhcG9sYXRlZCBjYWNoZSBidWlsZCwgYnV0IGV4Y2x1ZGUgYmVuY2htYXJraW5nIGNvc3QsIGZ1dHVyZSByZXN0YXJ0cywgaWRsZSB0aW1lIGFuZCB1bm1lYXN1cmVkIGNsb3VkLXN5bmMgZGVsYXlzLicsCiAgICAgICAgJ0NoZWNrcG9pbnQvZW5jb2Rlci9wcmV2aWV3IHdyaXRlcyBhcmUgbWVhc3VyZWQgb24gbW91bnRlZCBEcml2ZTsgYmVzdC1TU0wgZXhwb3J0IGlzIGNvbnNlcnZhdGl2ZWx5IGJ1ZGdldGVkIGV2ZXJ5IGVwb2NoLicsCiAgICAgICAgJ0NvbXBhcmUgb25seSByZXBvcnRzIHdpdGggdGhlIHNhbWUgY29tcGFyaXNvbl9rZXkuIENvbXB1dGUtdW5pdCByYXRlcyBhcmUgbWFudWFsbHkgc3VwcGxpZWQsIG5ldmVyIGluZmVycmVkLicsCiAgICAgICAgJ1RoZSBiZW5jaG1hcmsgdXBkYXRlcyBhIGRpc3Bvc2FibGUgbW9kZWw7IGl0cyBwcm9ncmVzcyBpcyBub3QgcGFydCBvZiBwcmV0cmFpbmluZy4nCiAgICBdKQogICAgd3JpdGVfcmVwb3J0cyhyZXBvcnQsZGlyZWN0b3J5KQogICAgdHJ5OgogICAgICAgIHJhdGU9b3B0aW9ucy5nZXQoJ3VuaXRzX3Blcl9ob3VyJykKICAgICAgICBpZiByYXRlIGlzIG5vdCBOb25lIGFuZCAobm90IG1hdGguaXNmaW5pdGUocmF0ZSkgb3IgcmF0ZSA8IDApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdDb21wdXRlLXVuaXQgcmF0ZSBtdXN0IGJlIG5vbm5lZ2F0aXZlIG9yIG51bGwnKQogICAgICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05vIENVREEgR1BVIGFzc2lnbmVkOyBzZWxlY3QgVDQsIEw0IG9yIEExMDAgaW4gQ29sYWInKQogICAgICAgIHJlcG9ydC51cGRhdGUoZ3B1X25hbWU9dG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoKSwKICAgICAgICAgICAgICAgICAgICAgIGdwdV90b3RhbF9naWI9dG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkudG90YWxfbWVtb3J5LzIqKjMwLAogICAgICAgICAgICAgICAgICAgICAgdG9yY2hfdmVyc2lvbj10b3JjaC5fX3ZlcnNpb25fXyxjdWRhX3ZlcnNpb249dG9yY2gudmVyc2lvbi5jdWRhKQogICAgICAgIHJlcG9ydFsnbnZpZGlhX3NtaSddPXN1YnByb2Nlc3MuY2hlY2tfb3V0cHV0KFsnbnZpZGlhLXNtaSddLHRleHQ9VHJ1ZSkKICAgICAgICBzZWVkPWNmZ1snc2VlZCddCiAgICAgICAgcmFuZG9tLnNlZWQoc2VlZCk7IG5wLnJhbmRvbS5zZWVkKHNlZWQpOyB0b3JjaC5tYW51YWxfc2VlZChzZWVkKTsgdG9yY2guY3VkYS5tYW51YWxfc2VlZF9hbGwoc2VlZCkKICAgICAgICB0b3JjaC5zZXRfbnVtX3RocmVhZHMoNCk7IHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmJlbmNobWFyaz1GYWxzZQogICAgICAgIHRvcmNoLmJhY2tlbmRzLmN1ZGEubWF0bXVsLmFsbG93X3RmMzI9RmFsc2UKICAgICAgICByb3VuZHM9b3B0aW9ucy5nZXQoJ3JvdW5kcycsMykKICAgICAgICB3YXJtdXBfdXBkYXRlcz1vcHRpb25zLmdldCgnd2FybXVwX3VwZGF0ZXMnLDIpCiAgICAgICAgbWVhc3VyZWRfdXBkYXRlcz1vcHRpb25zLmdldCgnbWVhc3VyZWRfdXBkYXRlcycsNikKICAgICAgICBpZiByb3VuZHMgPCAyIG9yIG5vdCAxIDw9IHdhcm11cF91cGRhdGVzIDw9IG1lYXN1cmVkX3VwZGF0ZXM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1VzZSBhdCBsZWFzdCAyIHJvdW5kcywgYW5kIDEgPD0gd2FybXVwX3VwZGF0ZXMgPD0gbWVhc3VyZWRfdXBkYXRlcycpCiAgICAgICAgdHJhaW5fZnJhbWUsdmFsX2ZyYW1lLGF1ZGl0PXByZXBhcmVfc3BsaXRzKGNmZ1snaW5kZXgnXSxjZmdbJ2ltYWdlX3Jvb3QnXSxkaXJlY3RvcnkvJ2F1ZGl0JywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZWVkLGNmZy5nZXQoJ2hvbGRvdXRfY3N2JyksY2ZnWydzc2xfdmFsX2ZyYWN0aW9uJ10pCiAgICAgICAgcmVwb3J0WydzcGxpdF9hdWRpdCddPWF1ZGl0CiAgICAgICAgc2FtcGxlX2NvdW50PWNmZ1snYmF0Y2hfc2l6ZSddKmNmZ1snYWNjdW11bGF0aW9uJ10qbWVhc3VyZWRfdXBkYXRlcwogICAgICAgIGlmIGxlbih0cmFpbl9mcmFtZSkgPCBzYW1wbGVfY291bnQ6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0luc3VmZmljaWVudCBpbWFnZXMgZm9yIGNvbmZpZ3VyZWQgYmVuY2htYXJrIHNhbXBsZTsgcmVkdWNlIG1lYXN1cmVkX3VwZGF0ZXMnKQogICAgICAgIHNhbXBsZWQ9dHJhaW5fZnJhbWUuc2FtcGxlKG49c2FtcGxlX2NvdW50LHJhbmRvbV9zdGF0ZT1zZWVkKS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICAgICAgc2FtcGxlZF92YWw9dmFsX2ZyYW1lLnNhbXBsZShuPW1pbihsZW4odmFsX2ZyYW1lKSxvcHRpb25zLmdldCgndmFsaWRhdGlvbl9pbWFnZXMnLDY0KSkscmFuZG9tX3N0YXRlPXNlZWQpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgICAgICBzYW1wbGVfaWRzPWRpY3QodHJhaW49c2FtcGxlZC5mY21hZV9yZWNvcmRfaWQudG9saXN0KCksdmFsaWRhdGlvbj1zYW1wbGVkX3ZhbC5mY21hZV9yZWNvcmRfaWQudG9saXN0KCkpCiAgICAgICAgc291cmNlX2hhc2hlcz17c3RyKHAucmVsYXRpdmVfdG8oUGF0aChfX2ZpbGVfXykucGFyZW50KSk6c2hhMjU2KHApIGZvciBwIGluIFBhdGgoX19maWxlX18pLnBhcmVudC5yZ2xvYignKi5weScpfQogICAgICAgIGtleXM9WydpbnB1dF9zaXplJywnYmF0Y2hfc2l6ZScsJ2FjY3VtdWxhdGlvbicsJ3dvcmtlcnMnLCdhbXAnLCdzZWVkJywnbGVhcm5pbmdfcmF0ZScsCiAgICAgICAgICAgICAgJ2Vwb2NocycsJ3dhcm11cF9lcG9jaHMnLCdzYXZlX2V2ZXJ5JywnY2hlY2twb2ludF9ldmVyeV91cGRhdGVzJywnY2hlY2twb2ludF9ldmVyeV9taW51dGVzJywKICAgICAgICAgICAgICAnY2xpcF9ncmFkJywnc3NsX3ZhbF9mcmFjdGlvbiddCiAgICAgICAgY29tcGFyaXNvbj1kaWN0KGNvbmZpZz17azpjZmdba10gZm9yIGsgaW4ga2V5c30sc291cmNlX2hhc2hlcz1zb3VyY2VfaGFzaGVzLAogICAgICAgICAgICAgICAgICAgICAgICBpbmRleF9zaGEyNTY9YXVkaXRbJ2luZGV4X3NoYTI1NiddLGhvbGRvdXRfc2hhMjU2PWF1ZGl0Wydob2xkb3V0X3NoYTI1NiddLAogICAgICAgICAgICAgICAgICAgICAgICBjaGVja3BvaW50X3NoYTI1Nj1zaGEyNTYoY2ZnWydjaGVja3BvaW50J10pLHNhbXBsZV9pZHM9c2FtcGxlX2lkcywKICAgICAgICAgICAgICAgICAgICAgICAgcm91bmRzPXJvdW5kcyx3YXJtdXBfdXBkYXRlcz13YXJtdXBfdXBkYXRlcyxtZWFzdXJlZF91cGRhdGVzPW1lYXN1cmVkX3VwZGF0ZXMsCiAgICAgICAgICAgICAgICAgICAgICAgIHJ1bnRpbWU9ZGljdCh0b3JjaD1zdHIodG9yY2guX192ZXJzaW9uX18pLGN1ZGE9dG9yY2gudmVyc2lvbi5jdWRhLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdGltbT1pbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbigndGltbScpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWlua293c2tpPWltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKCdNaW5rb3dza2lFbmdpbmUnKSkpCiAgICAgICAgcmVwb3J0Wydjb21wYXJpc29uX2tleSddPWhhc2hsaWIuc2hhMjU2KGpzb24uZHVtcHMoY29tcGFyaXNvbixzb3J0X2tleXM9VHJ1ZSkuZW5jb2RlKCkpLmhleGRpZ2VzdCgpCiAgICAgICAgcmVwb3J0Wydjb21wYXJpc29uX2lucHV0cyddPWNvbXBhcmlzb24KICAgICAgICB3cml0ZV9yZXBvcnRzKHJlcG9ydCxkaXJlY3RvcnkpCiAgICAgICAgZnJvbSByZWN0X2ZjbWFlIGltcG9ydCBSZWN0RkNNQUUsIGxvYWRfaW5pdGlhbCwgZXhwb3J0X2VuY29kZXIKICAgICAgICBmcm9tIHRyYWluIGltcG9ydCBwcmVmbGlnaHQsIHZpc3VhbGl6ZQogICAgICAgIG1vZGVsPVJlY3RGQ01BRSgpLmN1ZGEoKQogICAgICAgIHJlcG9ydFsnaW5pdGlhbGl6YXRpb24nXT1sb2FkX2luaXRpYWwobW9kZWwsY2ZnWydjaGVja3BvaW50J10pCiAgICAgICAgcHJlZmxpZ2h0KG1vZGVsLCdjdWRhJykKICAgICAgICByZXBvcnRbJ3NwYXJzZV9wcmVmbGlnaHRfcGFzc2VkJ109VHJ1ZQogICAgICAgIGRlY2F5LG5vX2RlY2F5PVtdLFtdCiAgICAgICAgZm9yIG5hbWUscGFyYW0gaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgICAgICAobm9fZGVjYXkgaWYgcGFyYW0ubmRpbTw9MSBvciBuYW1lLmVuZHN3aXRoKCdiaWFzJykgb3IgJ2dybicgaW4gbmFtZSBlbHNlIGRlY2F5KS5hcHBlbmQocGFyYW0pCiAgICAgICAgb3B0aW1pemVyPXRvcmNoLm9wdGltLkFkYW1XKFt7J3BhcmFtcyc6ZGVjYXksJ3dlaWdodF9kZWNheSc6LjA1fSx7J3BhcmFtcyc6bm9fZGVjYXksJ3dlaWdodF9kZWNheSc6MC59XSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbHI9Y2ZnWydsZWFybmluZ19yYXRlJ10sYmV0YXM9KC45LC45NSkpCiAgICAgICAgc2NhbGVyPXRvcmNoLmN1ZGEuYW1wLkdyYWRTY2FsZXIoZW5hYmxlZD1jZmdbJ2FtcCddKQogICAgICAgIGJlbmNoX2NmZz1kaWN0KGNmZyxjaGVja3BvaW50X2V2ZXJ5X3VwZGF0ZXM9MCxjaGVja3BvaW50X2V2ZXJ5X21pbnV0ZXM9MCkKICAgICAgICB1cGRhdGVzX3Blcl9lcG9jaD1tYXRoLmNlaWwobGVuKHRyYWluX2ZyYW1lKS8oY2ZnWydiYXRjaF9zaXplJ10qY2ZnWydhY2N1bXVsYXRpb24nXSkpCiAgICAgICAgdG90YWxfc3RlcHM9Y2ZnWydlcG9jaHMnXSp1cGRhdGVzX3Blcl9lcG9jaAogICAgICAgIHdhcm11cD1taW4odG90YWxfc3RlcHMtMSxjZmdbJ3dhcm11cF9lcG9jaHMnXSp1cGRhdGVzX3Blcl9lcG9jaCkKICAgICAgICBkZWYgcHJvZ3Jlc3Moc3RlcD0wKToKICAgICAgICAgICAgcmV0dXJuIGRpY3QobmV4dF9iYXRjaD0wLHN0ZXA9c3RlcCxsb3NzX3N1bT0wLixzZWVuPTAsc2Vjb25kcz0wLixwZWFrX2dwdV9naWI9MC4pCiAgICAgICAgIyBOZXcgc2NyYXRjaCBkaXJlY3RvcnkgZW5zdXJlcyBhIGNvbGQgcmVzaXplZC1pbWFnZSBjYWNoZSBldmVyeSBHUFUgcnVuLgogICAgICAgICMgT3JpZ2luYWwgUE5HcyBtYXkgYWxyZWFkeSBiZSBpbiB0aGUgT1MgY2FjaGU7IHRoaXMgaXMgZGlzY2xvc2VkIGJlbG93LgogICAgICAgIHdpdGggdGVtcGZpbGUuVGVtcG9yYXJ5RGlyZWN0b3J5KHByZWZpeD0nZmNtYWVfYmVuY2htYXJrXycpIGFzIHNjcmF0Y2g6CiAgICAgICAgICAgIGNhY2hlPVBhdGgoc2NyYXRjaCkvJ2NhY2hlJwogICAgICAgICAgICB0cmFpbl9kcz1FQ0dEYXRhc2V0KHNhbXBsZWQsY2ZnWydpbWFnZV9yb290J10sY2ZnWydpbnB1dF9zaXplJ10sY2FjaGUpCiAgICAgICAgICAgIHZhbF9kcz1FQ0dEYXRhc2V0KHNhbXBsZWRfdmFsLGNmZ1snaW1hZ2Vfcm9vdCddLGNmZ1snaW5wdXRfc2l6ZSddLGNhY2hlKQogICAgICAgICAgICBkZWYgbG9hZGVyKGRhdGFzZXQsb2Zmc2V0KToKICAgICAgICAgICAgICAgIHJldHVybiBEYXRhTG9hZGVyKGRhdGFzZXQsYmF0Y2hfc2l6ZT1jZmdbJ2JhdGNoX3NpemUnXSxzaHVmZmxlPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbnVtX3dvcmtlcnM9Y2ZnWyd3b3JrZXJzJ10scGluX21lbW9yeT1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGVyc2lzdGVudF93b3JrZXJzPWNmZ1snd29ya2VycyddPjAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBnZW5lcmF0b3I9dG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoc2VlZCtvZmZzZXQpKQogICAgICAgICAgICB0cmFpbl9sb2FkZXI9bG9hZGVyKHRyYWluX2RzLDMwMDAwMCk7IHZhbF9sb2FkZXI9bG9hZGVyKHZhbF9kcyw0MDAwMDApCiAgICAgICAgICAgIGNhY2hlX3RpbWluZ3M9W10KICAgICAgICAgICAgZm9yIGxhYmVsLGRhdGFfbG9hZGVyIGluIFsoJ3RyYWluJyx0cmFpbl9sb2FkZXIpLCgndmFsaWRhdGlvbicsdmFsX2xvYWRlcildOgogICAgICAgICAgICAgICAgdD10aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgICAgICBmb3IgXyBpbiBkYXRhX2xvYWRlcjoKICAgICAgICAgICAgICAgICAgICBwYXNzCiAgICAgICAgICAgICAgICBjYWNoZV90aW1pbmdzLmFwcGVuZChkaWN0KHNwbGl0PWxhYmVsLGltYWdlcz1sZW4oZGF0YV9sb2FkZXIuZGF0YXNldCksc2Vjb25kcz10aW1lLnBlcmZfY291bnRlcigpLXQpKQogICAgICAgICAgICBjYWNoZV9yYXRlPXN1bShyWydzZWNvbmRzJ10gZm9yIHIgaW4gY2FjaGVfdGltaW5ncykvc3VtKHJbJ2ltYWdlcyddIGZvciByIGluIGNhY2hlX3RpbWluZ3MpCiAgICAgICAgICAgIHJlcG9ydFsnY2FjaGVfcHJlcGFyYXRpb24nXT1kaWN0KHNhbXBsZXM9Y2FjaGVfdGltaW5ncywKICAgICAgICAgICAgICAgICBlc3RpbWF0ZWRfZnVsbF9zZWNvbmRzPWNhY2hlX3JhdGUqKGxlbih0cmFpbl9mcmFtZSkrbGVuKHZhbF9mcmFtZSkpLAogICAgICAgICAgICAgICAgIGxpbWl0YXRpb249J0ZyZXNoIHJlc2l6ZWQgY2FjaGU7IG1hc3Rlci1pbWFnZSBPUyBjYWNoZSBjYW5ub3QgYmUgZ3VhcmFudGVlZCBjb2xkLiBJbmNsdWRlcyBzYW1wbGUgd29ya2VyIHN0YXJ0dXAuJykKICAgICAgICAgICAgd2FybV9zYW1wbGVzPXdhcm11cF91cGRhdGVzKmNmZ1snYmF0Y2hfc2l6ZSddKmNmZ1snYWNjdW11bGF0aW9uJ10KICAgICAgICAgICAgcHJpbnQoZidHUFUgd2FybXVwOiB7d2FybXVwX3VwZGF0ZXN9IG9wdGltaXplciB1cGRhdGVzJyxmbHVzaD1UcnVlKQogICAgICAgICAgICBzdGF0ZT10cmFpbl9lcG9jaChtb2RlbCxpdGVydG9vbHMuaXNsaWNlKHRyYWluX2xvYWRlcix3YXJtdXBfdXBkYXRlcypjZmdbJ2FjY3VtdWxhdGlvbiddKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb3B0aW1pemVyLHNjYWxlcixiZW5jaF9jZmcsJ2N1ZGEnLDAsd2FybV9zYW1wbGVzLHByb2dyZXNzKCksdG90YWxfc3RlcHMsd2FybXVwLGxhbWJkYSBwOk5vbmUpCiAgICAgICAgICAgIHN0ZXA9c3RhdGVbJ3N0ZXAnXTsgdHJhaW5pbmc9W107IHZhbGlkYXRpb249W107IHBlYWs9MC4KICAgICAgICAgICAgZm9yIHJvdW5kX2luZGV4IGluIHJhbmdlKHJvdW5kcyk6CiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7IHQ9dGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICAgICAgc3RhdGU9dHJhaW5fZXBvY2gobW9kZWwsdHJhaW5fbG9hZGVyLG9wdGltaXplcixzY2FsZXIsYmVuY2hfY2ZnLCdjdWRhJywwLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbGVuKHRyYWluX2RzKSxwcm9ncmVzcyhzdGVwKSx0b3RhbF9zdGVwcyx3YXJtdXAsbGFtYmRhIHA6Tm9uZSkKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTsgZHVyYXRpb249dGltZS5wZXJmX2NvdW50ZXIoKS10CiAgICAgICAgICAgICAgICBzdGVwPXN0YXRlWydzdGVwJ10KICAgICAgICAgICAgICAgIHRyYWluaW5nLmFwcGVuZChkaWN0KHJvdW5kPXJvdW5kX2luZGV4KzEsaW1hZ2VzPWxlbih0cmFpbl9kcyksc2Vjb25kcz1kdXJhdGlvbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNlY29uZHNfcGVyX2ltYWdlPWR1cmF0aW9uL2xlbih0cmFpbl9kcykpKQogICAgICAgICAgICAgICAgcGVhaz1tYXgocGVhayx0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkvMioqMzApCiAgICAgICAgICAgICAgICBtb2RlbC5ldmFsKCk7IHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTsgdD10aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgICAgICB2YWxfcm5nPXRvcmNoLkdlbmVyYXRvcihkZXZpY2U9J2N1ZGEnKS5tYW51YWxfc2VlZChzZWVkKzEwMDApCiAgICAgICAgICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgICAgICAgICBmb3IgaW1hZ2VzLHZhbGlkLF8gaW4gdmFsX2xvYWRlcjoKICAgICAgICAgICAgICAgICAgICAgICAgd2l0aCB0b3JjaC5hdXRvY2FzdCgnY3VkYScsZHR5cGU9dG9yY2guZmxvYXQxNixlbmFibGVkPWNmZ1snYW1wJ10pOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgbG9zcyxfLF89bW9kZWwoaW1hZ2VzLmN1ZGEoKSx2YWxpZC5jdWRhKCksZ2VuZXJhdG9yPXZhbF9ybmcpCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIG5vdCB0b3JjaC5pc2Zpbml0ZShsb3NzKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcignTm9uZmluaXRlIGJlbmNobWFyayB2YWxpZGF0aW9uIGxvc3MnKQogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpOyBkdXJhdGlvbj10aW1lLnBlcmZfY291bnRlcigpLXQKICAgICAgICAgICAgICAgIHZhbGlkYXRpb24uYXBwZW5kKGRpY3Qocm91bmQ9cm91bmRfaW5kZXgrMSxpbWFnZXM9bGVuKHZhbF9kcyksc2Vjb25kcz1kdXJhdGlvbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc2Vjb25kc19wZXJfaW1hZ2U9ZHVyYXRpb24vbGVuKHZhbF9kcykpKQogICAgICAgICAgICAgICAgcGVhaz1tYXgocGVhayx0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkvMioqMzApCiAgICAgICAgICAgICAgICBwcmludChmIlJvdW5kIHtyb3VuZF9pbmRleCsxfTogdHJhaW49e3RyYWluaW5nWy0xXVsnc2Vjb25kcyddOi4yZn1zLCB2YWxpZGF0aW9uPXtkdXJhdGlvbjouMmZ9cyIsZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgcmVwb3J0LnVwZGF0ZSh0cmFpbmluZ19yb3VuZHM9dHJhaW5pbmcsdmFsaWRhdGlvbl9yb3VuZHM9dmFsaWRhdGlvbixwZWFrX2dwdV9naWI9cGVhaykKICAgICAgICAgICAgIyBFeGVyY2lzZSBhY3R1YWwgbW91bnRlZC1Ecml2ZSBzZXJpYWxpemF0aW9uLCB0aGVuIHJlbW92ZSBkaXNwb3NhYmxlCiAgICAgICAgICAgICMgYmVuY2htYXJrIHdlaWdodHMuIFRoZXkgYXJlIG5ldmVyIG5hbWVkIGFzIHByb2R1Y3Rpb24gY2hlY2twb2ludHMuCiAgICAgICAgICAgIGlvX2Rpcj1kaXJlY3RvcnkvJ2Rpc3Bvc2FibGVfaW8nOyBpb19kaXIubWtkaXIoKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBzYW1wbGVfc3RhdGU9c25hcHNob3QobW9kZWwsb3B0aW1pemVyLHNjYWxlciwnQkVOQ0hNQVJLX05PVF9BX1RSQUlOSU5HX0NIRUNLUE9JTlQnLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIDAsRmFsc2UsdG9yY2guYXJhbmdlKGxlbih0cmFpbl9mcmFtZSkpLHN0YXRlLGZsb2F0KCdpbmYnKSxbXSkKICAgICAgICAgICAgICAgIGNoZWNrcG9pbnRfdGltZXM9W10KICAgICAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKDIpOgogICAgICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKTsgdD10aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgICAgICAgICAgc2F2ZV9hdG9taWMoc2FtcGxlX3N0YXRlLGlvX2Rpci8nbGFzdC5wdCcsa2VlcF9wcmV2aW91cz1UcnVlKQogICAgICAgICAgICAgICAgICAgIGNoZWNrcG9pbnRfdGltZXMuYXBwZW5kKHRpbWUucGVyZl9jb3VudGVyKCktdCkKICAgICAgICAgICAgICAgIGNoZWNrcG9pbnRfYnl0ZXM9KGlvX2Rpci8nbGFzdC5wdCcpLnN0YXQoKS5zdF9zaXplCiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7IHQ9dGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICAgICAgc2F2ZV9hdG9taWMoeydtb2RlbCc6ZXhwb3J0X2VuY29kZXIobW9kZWwpLCdraW5kJzonRElTUE9TQUJMRSBCRU5DSE1BUksnfSxpb19kaXIvJ2VuY29kZXIucHQnKQogICAgICAgICAgICAgICAgZXhwb3J0X3NlY29uZHM9dGltZS5wZXJmX2NvdW50ZXIoKS10CiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7IHQ9dGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICAgICAgdmlzdWFsaXplKG1vZGVsLHZhbF9kcyxpb19kaXIvJ3ByZXZpZXcucG5nJyxzZWVkKzIwMDAsJ2N1ZGEnKQogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpOyBwcmV2aWV3X3NlY29uZHM9dGltZS5wZXJmX2NvdW50ZXIoKS10CiAgICAgICAgICAgICAgICBpbz1kaWN0KGNoZWNrcG9pbnRfc2Vjb25kcz1zdW0oY2hlY2twb2ludF90aW1lcykvbGVuKGNoZWNrcG9pbnRfdGltZXMpLAogICAgICAgICAgICAgICAgICAgICAgICBjaGVja3BvaW50X3RyaWFsc19zZWNvbmRzPWNoZWNrcG9pbnRfdGltZXMsY2hlY2twb2ludF9ieXRlcz1jaGVja3BvaW50X2J5dGVzLAogICAgICAgICAgICAgICAgICAgICAgICBleHBvcnRfc2Vjb25kcz1leHBvcnRfc2Vjb25kcyxwcmV2aWV3X3NlY29uZHM9cHJldmlld19zZWNvbmRzKQogICAgICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICAgICAgZm9yIHBhdGggaW4gaW9fZGlyLml0ZXJkaXIoKToKICAgICAgICAgICAgICAgICAgICBwYXRoLnVubGluaygpCiAgICAgICAgICAgICAgICBpb19kaXIucm1kaXIoKQogICAgICAgICAgICByZXBvcnRbJ2RyaXZlX2lvJ109aW8KICAgICAgICAgICAgcHJlcGFyYXRpb249b3B0aW9ucy5nZXQoJ3ByZXBhcmF0aW9uX3RpbWluZ3MnLHt9KQogICAgICAgICAgICByZXBvcnRbJ3ByZXBhcmF0aW9uX3RpbWluZ3MnXT1wcmVwYXJhdGlvbgogICAgICAgICAgICByZXBvcnRbJ3Byb2plY3Rpb25zJ109cHJvamVjdGlvbnMoY2ZnLGF1ZGl0LFtyWydzZWNvbmRzX3Blcl9pbWFnZSddIGZvciByIGluIHRyYWluaW5nXSwKICAgICAgICAgICAgICAgICAgICBbclsnc2Vjb25kc19wZXJfaW1hZ2UnXSBmb3IgciBpbiB2YWxpZGF0aW9uXSxpbyxzdW0ocHJlcGFyYXRpb24udmFsdWVzKCkpLAogICAgICAgICAgICAgICAgICAgIHJlcG9ydFsnY2FjaGVfcHJlcGFyYXRpb24nXVsnZXN0aW1hdGVkX2Z1bGxfc2Vjb25kcyddLHJhdGUpCiAgICAgICAgICAgICMgRW5kIHBlcnNpc3RlbnQgd29ya2VycyBiZWZvcmUgcmVtb3ZpbmcgdGhlaXIgc2NyYXRjaCBjYWNoZS4KICAgICAgICAgICAgZGVsIHRyYWluX2xvYWRlcix2YWxfbG9hZGVyCiAgICAgICAgcmVwb3J0WydzdGF0dXMnXT0nY29tcGxldGUnCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICByZXBvcnQudXBkYXRlKHN0YXR1cz0nZmFpbGVkJyxlcnJvcj1mJ3t0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfScsdHJhY2ViYWNrPXRyYWNlYmFjay5mb3JtYXRfZXhjKCkpCiAgICAgICAgdHJhY2ViYWNrLnByaW50X2V4YygpCiAgICBmaW5hbGx5OgogICAgICAgIHJlcG9ydFsnYmVuY2htYXJrX3dhbGxfc2Vjb25kcyddPXRpbWUucGVyZl9jb3VudGVyKCktc3RhcnRlZAogICAgICAgIHJlcG9ydFsnYmVuY2htYXJrX2NvbXB1dGVfdW5pdHMnXT1Ob25lIGlmIHJlcG9ydFsndW5pdHNfcGVyX2hvdXInXSBpcyBOb25lIGVsc2UgcmVwb3J0WydiZW5jaG1hcmtfd2FsbF9zZWNvbmRzJ10vMzYwMCpyZXBvcnRbJ3VuaXRzX3Blcl9ob3VyJ10KICAgICAgICB3cml0ZV9yZXBvcnRzKHJlcG9ydCxkaXJlY3RvcnkpCiAgICAgICAgcHJpbnQoJ0JlbmNobWFyayByZXBvcnQ6JyxkaXJlY3RvcnkvJ3N1bW1hcnkubWQnLGZsdXNoPVRydWUpCiAgICByZXR1cm4gcmVwb3J0WydzdGF0dXMnXT09J2NvbXBsZXRlJwoKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOgogICAgcGFyc2VyPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tY29uZmlnJyxyZXF1aXJlZD1UcnVlKTsgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1vcHRpb25zJyxyZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS1vdXRwdXQnLHJlcXVpcmVkPVRydWUpCiAgICBhcmdzPXBhcnNlci5wYXJzZV9hcmdzKCkKICAgIG9rPWJlbmNobWFyayhqc29uLmxvYWRzKFBhdGgoYXJncy5jb25maWcpLnJlYWRfdGV4dCgpKSxqc29uLmxvYWRzKFBhdGgoYXJncy5vcHRpb25zKS5yZWFkX3RleHQoKSksYXJncy5vdXRwdXQpCiAgICByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgb2sgZWxzZSAxKQo=', 'benchmark_report.py': 'IiIiUHVyZS1QeXRob24gdGltaW5nL2Nvc3QgcHJvamVjdGlvbnM7IG5vIEdQVSBwcmljZXMgb3Igc3BlZWQgcmF0aW9zIGFzc3VtZWQuIiIiCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKZGVmIHByb2plY3QoY2ZnLCBuX3RyYWluLCBuX3ZhbCwgdHJhaW5fc2Vjb25kc19wZXJfaW1hZ2UsIHZhbF9zZWNvbmRzX3Blcl9pbWFnZSwKICAgICAgICAgICAgY2hlY2twb2ludF9zZWNvbmRzLCBleHBvcnRfc2Vjb25kcywgcHJldmlld19zZWNvbmRzLCBlcG9jaHMsCiAgICAgICAgICAgIHVuaXRzX3Blcl9ob3VyPU5vbmUpOgogICAgaWYgbl90cmFpbiA8PSAwIG9yIG5fdmFsIDw9IDAgb3IgZXBvY2hzIDw9IDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUG9zaXRpdmUgZGF0YXNldCBjb3VudHMgYW5kIGVwb2NocyByZXF1aXJlZCcpCiAgICBpZiBhbnkodiA8IDAgZm9yIHYgaW4gW3RyYWluX3NlY29uZHNfcGVyX2ltYWdlLCB2YWxfc2Vjb25kc19wZXJfaW1hZ2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgIGNoZWNrcG9pbnRfc2Vjb25kcywgZXhwb3J0X3NlY29uZHMsIHByZXZpZXdfc2Vjb25kc10pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1RpbWluZ3MgY2Fubm90IGJlIG5lZ2F0aXZlJykKICAgIGlmIHVuaXRzX3Blcl9ob3VyIGlzIG5vdCBOb25lIGFuZCAobm90IG1hdGguaXNmaW5pdGUodW5pdHNfcGVyX2hvdXIpIG9yIHVuaXRzX3Blcl9ob3VyIDwgMCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRW50ZXIgYSBub25uZWdhdGl2ZSBvYnNlcnZlZCBjb21wdXRlLXVuaXQgcmF0ZSwgb3IgbGVhdmUgYmxhbmsnKQogICAgZ3JvdXAgPSBjZmdbJ2JhdGNoX3NpemUnXSpjZmdbJ2FjY3VtdWxhdGlvbiddCiAgICB1cGRhdGVzID0gbWF0aC5jZWlsKG5fdHJhaW4vZ3JvdXApCiAgICB1cGRhdGVfaW50ZXJ2YWwgPSBjZmcuZ2V0KCdjaGVja3BvaW50X2V2ZXJ5X3VwZGF0ZXMnLDEwMCkKICAgIHRpbWVfaW50ZXJ2YWwgPSBjZmcuZ2V0KCdjaGVja3BvaW50X2V2ZXJ5X21pbnV0ZXMnLDE1KSo2MAogICAgZ2xvYmFsX3N0ZXAsIHNhdmVzLCBleHBvcnRzLCBwcmV2aWV3cyA9IDAsIDAsIDEsIDAgICMgSW5pdGlhbCBjb250cm9sIGV4cG9ydC4KICAgIG1pbGVzdG9uZXMgPSAwCiAgICBmb3IgZXBvY2ggaW4gcmFuZ2UoZXBvY2hzKToKICAgICAgICBzaW5jZV9zYXZlID0gMC4KICAgICAgICBmb3IgdXBkYXRlIGluIHJhbmdlKHVwZGF0ZXMpOgogICAgICAgICAgICBzaW5jZV9zYXZlICs9IG1pbihncm91cCxuX3RyYWluLXVwZGF0ZSpncm91cCkqdHJhaW5fc2Vjb25kc19wZXJfaW1hZ2UKICAgICAgICAgICAgZ2xvYmFsX3N0ZXAgKz0gMQogICAgICAgICAgICBpZiAoKHVwZGF0ZV9pbnRlcnZhbCBhbmQgZ2xvYmFsX3N0ZXAldXBkYXRlX2ludGVydmFsID09IDApCiAgICAgICAgICAgICAgICBvciAodGltZV9pbnRlcnZhbCBhbmQgc2luY2Vfc2F2ZSA+PSB0aW1lX2ludGVydmFsKQogICAgICAgICAgICAgICAgb3IgdXBkYXRlKzEgPT0gdXBkYXRlcyk6CiAgICAgICAgICAgICAgICBzYXZlcyArPSAxCiAgICAgICAgICAgICAgICBzaW5jZV9zYXZlID0gMC4KICAgICAgICBzYXZlcyArPSAxICAjIENvbXBsZXRlZCBlcG9jaCwgYWZ0ZXIgdmFsaWRhdGlvbi9hcnRpZmFjdHMuCiAgICAgICAgZXhwb3J0cyArPSAxICAjIENvbnNlcnZhdGl2ZSBhc3N1bXB0aW9uOiBiZXN0LVNTTCBpbXByb3ZlcyBldmVyeSBlcG9jaC4KICAgICAgICBpZiAoZXBvY2grMSklY2ZnWydzYXZlX2V2ZXJ5J10gPT0gMCBvciBlcG9jaCsxID09IGNmZ1snZXBvY2hzJ106CiAgICAgICAgICAgIHNhdmVzICs9IDE7IGV4cG9ydHMgKz0gMTsgcHJldmlld3MgKz0gMTsgbWlsZXN0b25lcyArPSAxCiAgICB0cmFpbmluZyA9IG5fdHJhaW4qdHJhaW5fc2Vjb25kc19wZXJfaW1hZ2UqZXBvY2hzCiAgICB2YWxpZGF0aW9uID0gbl92YWwqdmFsX3NlY29uZHNfcGVyX2ltYWdlKmVwb2NocwogICAgd3JpdGVzID0gc2F2ZXMqY2hlY2twb2ludF9zZWNvbmRzICsgZXhwb3J0cypleHBvcnRfc2Vjb25kcyArIHByZXZpZXdzKnByZXZpZXdfc2Vjb25kcwogICAgc2Vjb25kcyA9IHRyYWluaW5nK3ZhbGlkYXRpb24rd3JpdGVzCiAgICByZXR1cm4gZGljdChlcG9jaHM9ZXBvY2hzLCB0cmFpbmluZ19zZWNvbmRzPXRyYWluaW5nLCB2YWxpZGF0aW9uX3NlY29uZHM9dmFsaWRhdGlvbiwKICAgICAgICAgICAgICAgIGNoZWNrcG9pbnRfYW5kX2FydGlmYWN0X3NlY29uZHM9d3JpdGVzLCBjaGVja3BvaW50X3dyaXRlcz1zYXZlcywKICAgICAgICAgICAgICAgIG1pbGVzdG9uZV9jaGVja3BvaW50cz1taWxlc3RvbmVzLCBlbmNvZGVyX2V4cG9ydHM9ZXhwb3J0cywKICAgICAgICAgICAgICAgIHByZXZpZXdfd3JpdGVzPXByZXZpZXdzLCBzdGVhZHlfdG90YWxfaG91cnM9c2Vjb25kcy8zNjAwLAogICAgICAgICAgICAgICAgYXZlcmFnZV9lcG9jaF9taW51dGVzPXNlY29uZHMvZXBvY2hzLzYwLAogICAgICAgICAgICAgICAgc3RlYWR5X2NvbXB1dGVfdW5pdHM9Tm9uZSBpZiB1bml0c19wZXJfaG91ciBpcyBOb25lIGVsc2Ugc2Vjb25kcy8zNjAwKnVuaXRzX3Blcl9ob3VyKQoKCmRlZiBwcm9qZWN0aW9ucyhjZmcsIGNvdW50cywgdHJhaW5fcmF0ZXMsIHZhbF9yYXRlcywgaW8sIHByZXBhcmF0aW9uX3NlY29uZHMsCiAgICAgICAgICAgICAgICBjYWNoZV9zZWNvbmRzLCB1bml0c19wZXJfaG91cik6CiAgICByb3dzPVtdCiAgICBmb3IgZXBvY2hzIGluIHNvcnRlZCh7MSxtaW4oMjAsY2ZnWydlcG9jaHMnXSksY2ZnWydlcG9jaHMnXX0pOgogICAgICAgIGZvciBsYWJlbCwgcmVkdWNlciBpbiBbKCdmYXN0X29ic2VydmVkJyxtaW4pLCgnbWVhbl9vYnNlcnZlZCcsbGFtYmRhIHhzOnN1bSh4cykvbGVuKHhzKSksKCdzbG93X29ic2VydmVkJyxtYXgpXToKICAgICAgICAgICAgcm93PXByb2plY3QoY2ZnLGNvdW50c1snc3NsX3RyYWluJ10sY291bnRzWydzc2xfdmFsaWRhdGlvbiddLAogICAgICAgICAgICAgICAgICAgICAgICByZWR1Y2VyKHRyYWluX3JhdGVzKSxyZWR1Y2VyKHZhbF9yYXRlcyksaW9bJ2NoZWNrcG9pbnRfc2Vjb25kcyddLAogICAgICAgICAgICAgICAgICAgICAgICBpb1snZXhwb3J0X3NlY29uZHMnXSxpb1sncHJldmlld19zZWNvbmRzJ10sZXBvY2hzLHVuaXRzX3Blcl9ob3VyKQogICAgICAgICAgICByb3dbJ3NjZW5hcmlvJ109bGFiZWwKICAgICAgICAgICAgcm93WydwbGFubmluZ190b3RhbF9ob3VycyddPXJvd1snc3RlYWR5X3RvdGFsX2hvdXJzJ10rKHByZXBhcmF0aW9uX3NlY29uZHMrY2FjaGVfc2Vjb25kcykvMzYwMAogICAgICAgICAgICByb3dbJ3BsYW5uaW5nX2NvbXB1dGVfdW5pdHMnXT1Ob25lIGlmIHVuaXRzX3Blcl9ob3VyIGlzIE5vbmUgZWxzZSByb3dbJ3BsYW5uaW5nX3RvdGFsX2hvdXJzJ10qdW5pdHNfcGVyX2hvdXIKICAgICAgICAgICAgcm93cy5hcHBlbmQocm93KQogICAgcmV0dXJuIHJvd3MKCgpkZWYgd3JpdGVfcmVwb3J0cyhyZXBvcnQsIGRpcmVjdG9yeSk6CiAgICBkaXJlY3Rvcnk9UGF0aChkaXJlY3RvcnkpOyBkaXJlY3RvcnkubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICAoZGlyZWN0b3J5LydyZXBvcnQuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsaW5kZW50PTIsYWxsb3dfbmFuPUZhbHNlKSsnXG4nKQogICAgcm93cz1yZXBvcnQuZ2V0KCdwcm9qZWN0aW9ucycsW10pCiAgICBpZiByb3dzOgogICAgICAgIHdpdGggb3BlbihkaXJlY3RvcnkvJ2VzdGltYXRlcy5jc3YnLCd3JyxuZXdsaW5lPScnKSBhcyBzdHJlYW06CiAgICAgICAgICAgIHdyaXRlcj1jc3YuRGljdFdyaXRlcihzdHJlYW0sZmllbGRuYW1lcz1saXN0KHJvd3NbMF0pKTt3cml0ZXIud3JpdGVoZWFkZXIoKTt3cml0ZXIud3JpdGVyb3dzKHJvd3MpCiAgICBsaW5lcz1bJyMgRkNNQUUgR1BVIGJlbmNobWFyaycsICcnLCBmIlN0YXR1czogKip7cmVwb3J0WydzdGF0dXMnXX0qKiIsJycsCiAgICAgICAgICAgZiJHUFU6IHtyZXBvcnQuZ2V0KCdncHVfbmFtZScsJ3VuYXZhaWxhYmxlJyl9IiwKICAgICAgICAgICBmIkNvbXB1dGUgdW5pdHMvaG91ciAoZW50ZXJlZCBieSB1c2VyKToge3JlcG9ydC5nZXQoJ3VuaXRzX3Blcl9ob3VyJyl9IiwKICAgICAgICAgICBmIkNvbXBhcmlzb24gZ3JvdXA6IGB7cmVwb3J0LmdldCgnY29tcGFyaXNvbl9rZXknLCd1bmF2YWlsYWJsZScpfWAiLCcnXQogICAgaWYgcmVwb3J0LmdldCgnZXJyb3InKToKICAgICAgICBsaW5lcyArPSBbZiJFcnJvcjoge3JlcG9ydFsnZXJyb3InXX0iLCcnXQogICAgaWYgcm93czoKICAgICAgICBsaW5lcyArPSBbJ3wgRXBvY2hzIHwgTWVhbiBob3Vycy9lcG9jaCB8IFRvdGFsIGhvdXJzIGluY2wuIHByZXBhcmF0aW9uL2NhY2hlIGVzdGltYXRlIHwgQ29tcHV0ZSB1bml0cyB8JywKICAgICAgICAgICAgICAgICAgJ3wtLS06fC0tLTp8LS0tOnwtLS06fCddCiAgICAgICAgZm9yIHIgaW4gcm93czoKICAgICAgICAgICAgaWYgclsnc2NlbmFyaW8nXT09J21lYW5fb2JzZXJ2ZWQnOgogICAgICAgICAgICAgICAgdW5pdHM9J25vdCBzdXBwbGllZCcgaWYgclsncGxhbm5pbmdfY29tcHV0ZV91bml0cyddIGlzIE5vbmUgZWxzZSBmIntyWydwbGFubmluZ19jb21wdXRlX3VuaXRzJ106LjJmfSIKICAgICAgICAgICAgICAgIGxpbmVzLmFwcGVuZChmInwge3JbJ2Vwb2NocyddfSB8IHtyWydhdmVyYWdlX2Vwb2NoX21pbnV0ZXMnXS82MDouMmZ9IHwge3JbJ3BsYW5uaW5nX3RvdGFsX2hvdXJzJ106LjJmfSB8IHt1bml0c30gfCIpCiAgICBsaW5lcyArPSBbJycsICdOb3RlczonLCAnJ10rW2YnLSB7bm90ZX0nIGZvciBub3RlIGluIHJlcG9ydC5nZXQoJ25vdGVzJyxbXSldCiAgICAoZGlyZWN0b3J5LydzdW1tYXJ5Lm1kJykud3JpdGVfdGV4dCgnXG4nLmpvaW4obGluZXMpKydcbicpCgoKZGVmIGNvbXBhcmlzb25fcm93cyhyb290KToKICAgIHJvd3M9W10KICAgIGZvciBwYXRoIGluIHNvcnRlZChQYXRoKHJvb3QpLmdsb2IoJyovcmVwb3J0Lmpzb24nKSk6CiAgICAgICAgcmVwb3J0PWpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoKSkKICAgICAgICByb3c9ZGljdChncHU9cmVwb3J0LmdldCgnZ3B1X25hbWUnKSxzdGF0dXM9cmVwb3J0WydzdGF0dXMnXSwKICAgICAgICAgICAgICAgICBjb21wYXJpc29uX2tleT1yZXBvcnQuZ2V0KCdjb21wYXJpc29uX2tleScpLHVuaXRzX3Blcl9ob3VyPXJlcG9ydC5nZXQoJ3VuaXRzX3Blcl9ob3VyJyksCiAgICAgICAgICAgICAgICAgcGVha19ncHVfZ2liPXJlcG9ydC5nZXQoJ3BlYWtfZ3B1X2dpYicpLGVwb2NoX21pbnV0ZXM9Tm9uZSwKICAgICAgICAgICAgICAgICBlcG9jaF9jb21wdXRlX3VuaXRzPU5vbmUscGxhbm5lZF9ob3Vycz1Ob25lLHBsYW5uZWRfY29tcHV0ZV91bml0cz1Ob25lLAogICAgICAgICAgICAgICAgIHJlcG9ydD1zdHIocGF0aCkpCiAgICAgICAgbWVhbnM9W3IgZm9yIHIgaW4gcmVwb3J0LmdldCgncHJvamVjdGlvbnMnLFtdKSBpZiByWydzY2VuYXJpbyddPT0nbWVhbl9vYnNlcnZlZCddCiAgICAgICAgaWYgbWVhbnM6CiAgICAgICAgICAgIGZpcnN0PW5leHQociBmb3IgciBpbiBtZWFucyBpZiByWydlcG9jaHMnXT09MSkKICAgICAgICAgICAgZmluYWw9bWF4KG1lYW5zLGtleT1sYW1iZGEgcjpyWydlcG9jaHMnXSkKICAgICAgICAgICAgcm93LnVwZGF0ZShlcG9jaF9taW51dGVzPWZpcnN0WydhdmVyYWdlX2Vwb2NoX21pbnV0ZXMnXSwKICAgICAgICAgICAgICAgICAgICAgICBlcG9jaF9jb21wdXRlX3VuaXRzPWZpcnN0WydzdGVhZHlfY29tcHV0ZV91bml0cyddLAogICAgICAgICAgICAgICAgICAgICAgIHBsYW5uZWRfaG91cnM9ZmluYWxbJ3BsYW5uaW5nX3RvdGFsX2hvdXJzJ10scGxhbm5lZF9jb21wdXRlX3VuaXRzPWZpbmFsWydwbGFubmluZ19jb21wdXRlX3VuaXRzJ10pCiAgICAgICAgcm93cy5hcHBlbmQocm93KQogICAgcmV0dXJuIHJvd3MK', 'test_benchmark_report.py': 'IiIiQ2hlY2sgY2hlY2twb2ludC1hd2FyZSBlc3RpbWF0ZXMgYW5kIHBvcnRhYmxlIGNvbXBhcmlzb24gcmVwb3J0cyBvbiBDUFUuIiIiCmltcG9ydCBqc29uCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdW5pdHRlc3QKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gYmVuY2htYXJrX3JlcG9ydCBpbXBvcnQgcHJvamVjdCwgcHJvamVjdGlvbnMsIHdyaXRlX3JlcG9ydHMsIGNvbXBhcmlzb25fcm93cwoKCmNsYXNzIEJlbmNobWFya1JlcG9ydFRlc3RzKHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiBzZXRVcChzZWxmKToKICAgICAgICBzZWxmLmNmZyA9IGRpY3QoYmF0Y2hfc2l6ZT0yLCBhY2N1bXVsYXRpb249MiwgZXBvY2hzPTIwLCBzYXZlX2V2ZXJ5PTEwLAogICAgICAgICAgICAgICAgICAgICAgICBjaGVja3BvaW50X2V2ZXJ5X3VwZGF0ZXM9MiwgY2hlY2twb2ludF9ldmVyeV9taW51dGVzPTApCgogICAgZGVmIHRlc3Rfc2hvcnRfZ3JvdXBfYW5kX2dsb2JhbF91cGRhdGVfY2hlY2twb2ludF9jb3VudChzZWxmKToKICAgICAgICAjIFRlbiBpbWFnZXM6IHVwZGF0ZXMgb2YgNCwgNCwgMi4gRXBvY2ggb25lIHNhdmVzIGF0IDIvMzsKICAgICAgICAjIGVwb2NoIHR3byBzYXZlcyBhdCBnbG9iYWwgc3RlcHMgNC82LCBwbHVzIG9uZSBjb21wbGV0aW9uIGVhY2guCiAgICAgICAgciA9IHByb2plY3Qoc2VsZi5jZmcsIDEwLCAyLCAxLCAyLCAzLCA1LCA3LCAyLCA0KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoclsnY2hlY2twb2ludF93cml0ZXMnXSwgNikKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJbJ2VuY29kZXJfZXhwb3J0cyddLCAzKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoclsndHJhaW5pbmdfc2Vjb25kcyddLCAyMCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJbJ3ZhbGlkYXRpb25fc2Vjb25kcyddLCA4KQogICAgICAgIHNlbGYuYXNzZXJ0QWxtb3N0RXF1YWwoclsnc3RlYWR5X3RvdGFsX2hvdXJzJ10sICgyMCs4KzE4KzE1KS8zNjAwKQogICAgICAgIHNlbGYuYXNzZXJ0QWxtb3N0RXF1YWwoclsnc3RlYWR5X2NvbXB1dGVfdW5pdHMnXSwgNjEvMzYwMCo0KQoKICAgIGRlZiB0ZXN0X3RpbWVyX2FuZF9maW5hbF9lcG9jaF9hcnRpZmFjdHMoc2VsZik6CiAgICAgICAgY2ZnID0gZGljdChzZWxmLmNmZywgZXBvY2hzPTEsIGNoZWNrcG9pbnRfZXZlcnlfdXBkYXRlcz0wLAogICAgICAgICAgICAgICAgICAgY2hlY2twb2ludF9ldmVyeV9taW51dGVzPTUvNjApCiAgICAgICAgIyBUaW1lciB0cmlnZ2VycyBhZnRlciBlaWdodCBzZWNvbmRzOyBmaW5hbCBzaG9ydCBncm91cCBzdGlsbCBzYXZlcy4KICAgICAgICByID0gcHJvamVjdChjZmcsIDEwLCAyLCAxLCAxLCAzLCA1LCA3LCAxKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoclsnY2hlY2twb2ludF93cml0ZXMnXSwgNCkKICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJbJ2VuY29kZXJfZXhwb3J0cyddLCAzKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoclsncHJldmlld193cml0ZXMnXSwgMSkKICAgICAgICBzZWxmLmFzc2VydElzTm9uZShyWydzdGVhZHlfY29tcHV0ZV91bml0cyddKQoKICAgIGRlZiB0ZXN0X3plcm9fcmF0ZV9pc19kaXN0aW5jdF9mcm9tX3Vua25vd24oc2VsZik6CiAgICAgICAgYXJncyA9IChzZWxmLmNmZywgMTAsIDIsIDEsIDEsIDEsIDEsIDEsIDEpCiAgICAgICAgc2VsZi5hc3NlcnRJc05vbmUocHJvamVjdCgqYXJncylbJ3N0ZWFkeV9jb21wdXRlX3VuaXRzJ10pCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChwcm9qZWN0KCphcmdzLCB1bml0c19wZXJfaG91cj0wKVsnc3RlYWR5X2NvbXB1dGVfdW5pdHMnXSwgMCkKICAgICAgICBmb3IgcmF0ZSBpbiAoLTEsIGZsb2F0KCduYW4nKSwgZmxvYXQoJ2luZicpKToKICAgICAgICAgICAgd2l0aCBzZWxmLmFzc2VydFJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICAgICAgICAgIHByb2plY3QoKmFyZ3MsIHVuaXRzX3Blcl9ob3VyPXJhdGUpCgogICAgZGVmIHRlc3RfcHJlcGFyYXRpb25fY2hhcmdlZF9vbmNlX2FuZF9vYnNlcnZlZF9yYW5nZShzZWxmKToKICAgICAgICByb3dzID0gcHJvamVjdGlvbnMoc2VsZi5jZmcsIHsnc3NsX3RyYWluJzoxMCwgJ3NzbF92YWxpZGF0aW9uJzoyfSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgWzEsMiwzXSwgWzEsMiwzXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgZGljdChjaGVja3BvaW50X3NlY29uZHM9MCwgZXhwb3J0X3NlY29uZHM9MCwgcHJldmlld19zZWNvbmRzPTApLAogICAgICAgICAgICAgICAgICAgICAgICAgICA2MDAsIDEyMDAsIDIpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChsZW4ocm93cyksIDYpICAjIEhvcml6b25zIDEgYW5kIDIwLCBubyBkdXBsaWNhdGUgMjAuCiAgICAgICAgZm9yIHIgaW4gcm93czoKICAgICAgICAgICAgc2VsZi5hc3NlcnRBbG1vc3RFcXVhbChyWydwbGFubmluZ190b3RhbF9ob3VycyddLXJbJ3N0ZWFkeV90b3RhbF9ob3VycyddLCAuNSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRBbG1vc3RFcXVhbChyWydwbGFubmluZ19jb21wdXRlX3VuaXRzJ10sIHJbJ3BsYW5uaW5nX3RvdGFsX2hvdXJzJ10qMikKICAgICAgICBtZWFucyA9IFtyIGZvciByIGluIHJvd3MgaWYgclsnc2NlbmFyaW8nXT09J21lYW5fb2JzZXJ2ZWQnXQogICAgICAgIHNlbGYuYXNzZXJ0QWxtb3N0RXF1YWwobWVhbnNbMV1bJ3RyYWluaW5nX3NlY29uZHMnXSwgNDAwKQogICAgICAgIHNlbGYuYXNzZXJ0TGVzcyhyb3dzWzBdWydzdGVhZHlfdG90YWxfaG91cnMnXSwgcm93c1sxXVsnc3RlYWR5X3RvdGFsX2hvdXJzJ10pCiAgICAgICAgc2VsZi5hc3NlcnRMZXNzKHJvd3NbMV1bJ3N0ZWFkeV90b3RhbF9ob3VycyddLCByb3dzWzJdWydzdGVhZHlfdG90YWxfaG91cnMnXSkKCiAgICBkZWYgdGVzdF9yZXBvcnRzX3ByZXNlcnZlX2NvbXBhcmlzb25fZ3JvdXBzX2FuZF9mYWlsdXJlcyhzZWxmKToKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRlbXBvcmFyeToKICAgICAgICAgICAgcm9vdCA9IFBhdGgodGVtcG9yYXJ5KQogICAgICAgICAgICByb3dzID0gcHJvamVjdGlvbnMoc2VsZi5jZmcsIHsnc3NsX3RyYWluJzoxMCwgJ3NzbF92YWxpZGF0aW9uJzoyfSwgWzFdLCBbMV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBkaWN0KGNoZWNrcG9pbnRfc2Vjb25kcz0wLCBleHBvcnRfc2Vjb25kcz0wLCBwcmV2aWV3X3NlY29uZHM9MCksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAwLCAwLCBOb25lKQogICAgICAgICAgICB3cml0ZV9yZXBvcnRzKGRpY3Qoc3RhdHVzPSdjb21wbGV0ZScsIGdwdV9uYW1lPSdUNCcsIGNvbXBhcmlzb25fa2V5PSdvbmUnLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdW5pdHNfcGVyX2hvdXI9Tm9uZSwgcHJvamVjdGlvbnM9cm93cyksIHJvb3QvJ3Q0JykKICAgICAgICAgICAgd3JpdGVfcmVwb3J0cyhkaWN0KHN0YXR1cz0nZmFpbGVkJywgZ3B1X25hbWU9J0w0JywgY29tcGFyaXNvbl9rZXk9J3R3bycsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlcnJvcj0nQ1VEQSBvdXQgb2YgbWVtb3J5JyksIHJvb3QvJ2w0JykKICAgICAgICAgICAgcmVzdWx0ID0gY29tcGFyaXNvbl9yb3dzKHJvb3QpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoe3JbJ2NvbXBhcmlzb25fa2V5J10gZm9yIHIgaW4gcmVzdWx0fSwgeydvbmUnLCd0d28nfSkKICAgICAgICAgICAgZmFpbGVkID0gbmV4dChyIGZvciByIGluIHJlc3VsdCBpZiByWydzdGF0dXMnXT09J2ZhaWxlZCcpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SXNOb25lKGZhaWxlZFsncGxhbm5lZF9ob3VycyddKQogICAgICAgICAgICBzZWxmLmFzc2VydEluKCdub3Qgc3VwcGxpZWQnLCAocm9vdC8ndDQvc3VtbWFyeS5tZCcpLnJlYWRfdGV4dCgpKQogICAgICAgICAgICBzZWxmLmFzc2VydFRydWUoKHJvb3QvJ3Q0L2VzdGltYXRlcy5jc3YnKS5leGlzdHMoKSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChqc29uLmxvYWRzKChyb290LydsNC9yZXBvcnQuanNvbicpLnJlYWRfdGV4dCgpKVsnZXJyb3InXSwgJ0NVREEgb3V0IG9mIG1lbW9yeScpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'downstream_holdouts_v2.csv': '...', 'holdout_provenance.json': 'ewogICJzb3VyY2UiOiAiZGF0YXNldC1jb21iaW5hdGlvbi1jb2RlL2VjZ19kYXRhc2V0X3BpcGVsaW5lL2RhdGEvcmVsZWFzZXMvdjIvaGFybW9uaXplZC9oYXJtb25pemVkX2VjZ19tZXRhZGF0YS5jc3YiLAogICJzb3VyY2Vfc2hhMjU2IjogImEwZmIyYzY2OTc4MGY0YTQyNmNjYmM0Y2I5MjRjOWQ5YWU5NTg3MDI0ZmJmNDk5YzM5OTgzYzA5ZDNiZTRiYzAiLAogICJob2xkb3V0X3JlY29yZHMiOiAzOTI1LAogICJjb3VudHMiOiB7CiAgICAicHRiLXhsIjogMjc4NCwKICAgICJjaGFwbWFuX3NoYW94aW5nX25pbmdibyI6IDg1MSwKICAgICJTVEVNSSI6IDI5MAogIH0sCiAgImxpbWl0YXRpb24iOiAiT25seSBlc3RhYmxpc2hlZCBzb3VyY2UgbmFtZXNwYWNlcyBhcmUgbWF0Y2hlZC4gVW5rbm93biBhbmQgY3Jvc3Mtc291cmNlIHBhdGllbnQgbGlua2FnZSByZW1haW5zIHVuYXZhaWxhYmxlLiIKfQo=', 'vendor/LICENSE': '...', 'vendor/README.md': 'IyMgQ29udk5lWHQgVjI8YnI+PHN1Yj5PZmZpY2lhbCBQeVRvcmNoIEltcGxlbWVudGF0aW9uPC9zdWI+CgpUaGlzIHJlcG8gY29udGFpbnMgdGhlIFB5VG9yY2ggdmVyc2lvbiBvZiAqOCogbW9kZWwgZGVmaW5pdGlvbnMgKCpBdHRvLCBGZW10bywgUGljbywgTmFubywgVGlueSwgQmFzZSwgTGFyZ2UsIEh1Z2UqKSwgcHJlLXRyYWluaW5nL2ZpbmUtdHVuaW5nIGNvZGUgYW5kIHByZS10cmFpbmVkIHdlaWdodHMgKGNvbnZlcnRlZCBmcm9tIEpBWCB3ZWlnaHRzIHRyYWluZWQgb24gVFBVKSBmb3Igb3VyIENvbnZOZVh0IFYyIHBhcGVyLgoKPiBbKipDb252TmVYdCBWMjogQ28tZGVzaWduaW5nIGFuZCBTY2FsaW5nIENvbnZOZXRzIHdpdGggTWFza2VkIEF1dG9lbmNvZGVycyoqXShodHRwOi8vYXJ4aXYub3JnL2Ficy8yMzAxLjAwODA4KTxicj4KPiBbU2FuZ2h5dW4gV29vXShodHRwczovL3NpdGVzLmdvb2dsZS5jb20vdmlldy9zYW5naHl1bndvby8pLCBbU2hvdWJoaWsgRGVibmF0aF0oaHR0cHM6Ly93d3cubGlua2VkaW4uY29tL2luL3Nob3ViaGlrLWRlYm5hdGgtNDEyNjg1NzAvKSwgW1JvbmdoYW5nIEh1XShodHRwczovL3JvbmdoYW5naHUuY29tLyksIFtYaW5sZWkgQ2hlbl0oaHR0cHM6Ly94aW5sZWljLnh5ei8pLCBbWmh1YW5nIExpdV0oaHR0cHM6Ly9saXV6aHVhbmcxMy5naXRodWIuaW8vKSwgW0luIFNvIEt3ZW9uXShodHRwczovL3NjaG9sYXIuZ29vZ2xlLmNvbS9jaXRhdGlvbnM/dXNlcj1YQThFT2xFQUFBQUomaGw9ZW4pIGFuZCBbU2FpbmluZyBYaWVdKGh0dHBzOi8vc2FpbmluZ3hpZS5jb20pXAo+IDxicj5LQUlTVCwgTWV0YSBBSSBhbmQgTmV3IFlvcmsgVW5pdmVyc2l0eTxicj4KCldlIHByb3Bvc2UgYSBmdWxseSBjb252b2x1dGlvbmFsIG1hc2tlZCBhdXRvZW5jb2RlciBmcmFtZXdvcmsgKEZDTUFFKSBhbmQgYSBuZXcgR2xvYmFsIFJlc3BvbnNlIE5vcm1hbGl6YXRpb24gKEdSTikgbGF5ZXIgdGhhdCBjYW4gYmUgYWRkZWQgdG8gdGhlIENvbnZOZVh0IGFyY2hpdGVjdHVyZSB0byBlbmhhbmNlIGludGVyLWNoYW5uZWwgZmVhdHVyZSBjb21wZXRpdGlvbi4gVGhpcyBjby1kZXNpZ24gb2Ygc2VsZi1zdXBlcnZpc2VkIGxlYXJuaW5nIHRlY2huaXF1ZXMgYW5kIGFyY2hpdGVjdHVyYWwgaW1wcm92ZW1lbnQgcmVzdWx0cyBpbiBhIG5ldyBtb2RlbCBmYW1pbHkgY2FsbGVkIENvbnZOZVh0IFYyLCB3aGljaCBzaWduaWZpY2FudGx5IGltcHJvdmVzIHRoZSBwZXJmb3JtYW5jZSBvZiBwdXJlIENvbnZOZXRzIG9uIHZhcmlvdXMgcmVjb2duaXRpb24gYmVuY2htYXJrcy4gV2UgYWxzbyBwcm92aWRlIHByZS10cmFpbmVkIENvbnZOZVh0IFYyIG1vZGVscyBvZiB2YXJpb3VzIHNpemVzLgoKPHAgYWxpZ249ImNlbnRlciI+CjxpbWcgc3JjPSJmaWd1cmVzL2ZjbWFlX2NvbnZuZXh0djIucG5nIiB3aWR0aD0iNzAlIiBoZWlnaHQ9IjcwJSIgY2xhc3M9ImNlbnRlciI+CjwvcD4KCjxwIGFsaWduPSJjZW50ZXIiPgo8aW1nIHNyYz0iZmlndXJlcy9tb2RlbF9zY2FsaW5nLnBuZyIgd2lkdGg9IjUwJSIgaGVpZ2h0PSI1MCUiIGNsYXNzPSJjZW50ZXIiPgo8L3A+CgojIyBSZXN1bHRzIGFuZCBQcmUtdHJhaW5lZCBNb2RlbHMKIyMjIEltYWdlTmV0LTFLIEZDTUFFIHByZS10cmFpbmVkIHdlaWdodHMgKCpzZWxmLXN1cGVydmlzZWQqKQp8IG5hbWUgfCByZXNvbHV0aW9uIHwgI3BhcmFtcyB8IG1vZGVsIHwKfDotLS06fDotLS06fDotLS06fDotLS06fAp8IENvbnZOZVh0IFYyLUEgfCAyMjR4MjI0IHwgMy43TSB8IFttb2RlbF0oaHR0cHM6Ly9kbC5mYmFpcHVibGljZmlsZXMuY29tL2NvbnZuZXh0L2NvbnZuZXh0djIvcHRfb25seS9jb252bmV4dHYyX2F0dG9fMWtfMjI0X2ZjbWFlLnB0KSB8CnwgQ29udk5lWHQgVjItRiB8IDIyNHgyMjQgfCA1LjJNIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9wdF9vbmx5L2NvbnZuZXh0djJfZmVtdG9fMWtfMjI0X2ZjbWFlLnB0KSB8CnwgQ29udk5lWHQgVjItUCB8IDIyNHgyMjQgfCA5LjFNIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9wdF9vbmx5L2NvbnZuZXh0djJfcGljb18xa18yMjRfZmNtYWUucHQpIHwKfCBDb252TmVYdCBWMi1OIHwgMjI0eDIyNCB8IDE1LjZNfCBbbW9kZWxdKGh0dHBzOi8vZGwuZmJhaXB1YmxpY2ZpbGVzLmNvbS9jb252bmV4dC9jb252bmV4dHYyL3B0X29ubHkvY29udm5leHR2Ml9uYW5vXzFrXzIyNF9mY21hZS5wdCkgfAp8IENvbnZOZVh0IFYyLVQgfCAyMjR4MjI0IHwgMjguNk18IFttb2RlbF0oaHR0cHM6Ly9kbC5mYmFpcHVibGljZmlsZXMuY29tL2NvbnZuZXh0L2NvbnZuZXh0djIvcHRfb25seS9jb252bmV4dHYyX3RpbnlfMWtfMjI0X2ZjbWFlLnB0KSB8CnwgQ29udk5lWHQgVjItQiB8IDIyNHgyMjQgfCA4OU0gIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9wdF9vbmx5L2NvbnZuZXh0djJfYmFzZV8xa18yMjRfZmNtYWUucHQpIHwKfCBDb252TmVYdCBWMi1MIHwgMjI0eDIyNCB8IDE5OE0gfCBbbW9kZWxdKGh0dHBzOi8vZGwuZmJhaXB1YmxpY2ZpbGVzLmNvbS9jb252bmV4dC9jb252bmV4dHYyL3B0X29ubHkvY29udm5leHR2Ml9sYXJnZV8xa18yMjRfZmNtYWUucHQpIHwKfCBDb252TmVYdCBWMi1IIHwgMjI0eDIyNCB8IDY2ME0gfCBbbW9kZWxdKGh0dHBzOi8vZGwuZmJhaXB1YmxpY2ZpbGVzLmNvbS9jb252bmV4dC9jb252bmV4dHYyL3B0X29ubHkvY29udm5leHR2Ml9odWdlXzFrXzIyNF9mY21hZS5wdCkgfAoKIyMjIEltYWdlTmV0LTFLIGZpbmUtdHVuZWQgbW9kZWxzCgp8IG5hbWUgfCByZXNvbHV0aW9uIHxhY2NAMSB8ICNwYXJhbXMgfCBGTE9QcyB8IG1vZGVsIHwKfDotLS06fDotLS06fDotLS06fDotLS06fCA6LS0tOnw6LS0tOnwKfCBDb252TmVYdCBWMi1BIHwgMjI0eDIyNCB8IDc2LjcgfCAzLjdNICB8IDAuNTVHIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9pbTFrL2NvbnZuZXh0djJfYXR0b18xa18yMjRfZW1hLnB0KSB8CnwgQ29udk5lWHQgVjItRiB8IDIyNHgyMjQgfCA3OC41IHwgNS4yTSAgfCAwLjc4RyB8IFttb2RlbF0oaHR0cHM6Ly9kbC5mYmFpcHVibGljZmlsZXMuY29tL2NvbnZuZXh0L2NvbnZuZXh0djIvaW0xay9jb252bmV4dHYyX2ZlbXRvXzFrXzIyNF9lbWEucHQpIHwKfCBDb252TmVYdCBWMi1QIHwgMjI0eDIyNCB8IDgwLjMgfCA5LjFNICB8IDEuMzdHIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9pbTFrL2NvbnZuZXh0djJfcGljb18xa18yMjRfZW1hLnB0KSB8CnwgQ29udk5lWHQgVjItTiB8IDIyNHgyMjQgfCA4MS45IHwgMTUuNk0gfCAyLjQ1RyB8IFttb2RlbF0oaHR0cHM6Ly9kbC5mYmFpcHVibGljZmlsZXMuY29tL2NvbnZuZXh0L2NvbnZuZXh0djIvaW0xay9jb252bmV4dHYyX25hbm9fMWtfMjI0X2VtYS5wdCkgfAp8IENvbnZOZVh0IFYyLVQgfCAyMjR4MjI0IHwgODMuMCB8IDI4LjZNIHwgNC40N0cgfCBbbW9kZWxdKGh0dHBzOi8vZGwuZmJhaXB1YmxpY2ZpbGVzLmNvbS9jb252bmV4dC9jb252bmV4dHYyL2ltMWsvY29udm5leHR2Ml90aW55XzFrXzIyNF9lbWEucHQpIHwKfCBDb252TmVYdCBWMi1CIHwgMjI0eDIyNCB8IDg0LjkgfCA4OU0gICB8IDE1LjRHIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9pbTFrL2NvbnZuZXh0djJfYmFzZV8xa18yMjRfZW1hLnB0KSB8CnwgQ29udk5lWHQgVjItTCB8IDIyNHgyMjQgfCA4NS44IHwgMTk4TSAgfCAzNC40RyB8IFttb2RlbF0oaHR0cHM6Ly9kbC5mYmFpcHVibGljZmlsZXMuY29tL2NvbnZuZXh0L2NvbnZuZXh0djIvaW0xay9jb252bmV4dHYyX2xhcmdlXzFrXzIyNF9lbWEucHQpIHwKfCBDb252TmVYdCBWMi1IIHwgMjI0eDIyNCB8IDg2LjMgfCA2NjBNICB8IDExNUcgIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9pbTFrL2NvbnZuZXh0djJfaHVnZV8xa18yMjRfZW1hLnB0KSB8CgojIyMgSW1hZ2VOZXQtMjJLIGZpbmUtdHVuZWQgbW9kZWxzCgp8IG5hbWUgfCByZXNvbHV0aW9uIHxhY2NAMSB8ICNwYXJhbXMgfCBGTE9QcyB8IG1vZGVsIHwKfDotLS06fDotLS06fDotLS06fDotLS06fCA6LS0tOnwgOi0tLTp8CnwgQ29udk5lWHQgVjItTiB8IDIyNHgyMjQgfCA4Mi4xIHwgMTUuNk0gfCAyLjQ1RyAgIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9pbTIyay9jb252bmV4dHYyX25hbm9fMjJrXzIyNF9lbWEucHQpfAp8IENvbnZOZVh0IFYyLU4gfCAzODR4Mzg0IHwgODMuNCB8IDE1LjZNIHwgNy4yMUcgICB8IFttb2RlbF0oaHR0cHM6Ly9kbC5mYmFpcHVibGljZmlsZXMuY29tL2NvbnZuZXh0L2NvbnZuZXh0djIvaW0yMmsvY29udm5leHR2Ml9uYW5vXzIya18zODRfZW1hLnB0KXwKfCBDb252TmVYdCBWMi1UIHwgMjI0eDIyNCB8IDgzLjkgfCAyOC42TSB8IDQuNDdHICAgfCBbbW9kZWxdKGh0dHBzOi8vZGwuZmJhaXB1YmxpY2ZpbGVzLmNvbS9jb252bmV4dC9jb252bmV4dHYyL2ltMjJrL2NvbnZuZXh0djJfdGlueV8yMmtfMjI0X2VtYS5wdCl8CnwgQ29udk5lWHQgVjItVCB8IDM4NHgzODQgfCA4NS4xIHwgMjguNk0gfCAxMy4xRyAgfCBbbW9kZWxdKGh0dHBzOi8vZGwuZmJhaXB1YmxpY2ZpbGVzLmNvbS9jb252bmV4dC9jb252bmV4dHYyL2ltMjJrL2NvbnZuZXh0djJfdGlueV8yMmtfMzg0X2VtYS5wdCl8CnwgQ29udk5lWHQgVjItQiB8IDIyNHgyMjQgfCA4Ni44IHwgODlNICAgfCAxNS40RyAgIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9pbTIyay9jb252bmV4dHYyX2Jhc2VfMjJrXzIyNF9lbWEucHQpfAp8IENvbnZOZVh0IFYyLUIgfCAzODR4Mzg0IHwgODcuNyB8IDg5TSAgIHwgNDUuMkcgIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9pbTIyay9jb252bmV4dHYyX2Jhc2VfMjJrXzM4NF9lbWEucHQpfAp8IENvbnZOZVh0IFYyLUwgfCAyMjR4MjI0IHwgODcuMyB8IDE5OE0gIHwgMzQuNEcgICB8IFttb2RlbF0oaHR0cHM6Ly9kbC5mYmFpcHVibGljZmlsZXMuY29tL2NvbnZuZXh0L2NvbnZuZXh0djIvaW0yMmsvY29udm5leHR2Ml9sYXJnZV8yMmtfMjI0X2VtYS5wdCl8CnwgQ29udk5lWHQgVjItTCB8IDM4NHgzODQgfCA4OC4yIHwgMTk4TSAgfCAxMDEuMUcgIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9pbTIyay9jb252bmV4dHYyX2xhcmdlXzIya18zODRfZW1hLnB0KXwKfCBDb252TmVYdCBWMi1IIHwgMzg0eDM4NCB8IDg4LjcgfCA2NjBNICB8IDMzNy45RyAgfCBbbW9kZWxdKGh0dHBzOi8vZGwuZmJhaXB1YmxpY2ZpbGVzLmNvbS9jb252bmV4dC9jb252bmV4dHYyL2ltMjJrL2NvbnZuZXh0djJfaHVnZV8yMmtfMzg0X2VtYS5wdCl8CnwgQ29udk5lWHQgVjItSCB8IDUxMng1MTIgfCA4OC45IHwgNjYwTSAgfCA2MDAuOEcgIHwgW21vZGVsXShodHRwczovL2RsLmZiYWlwdWJsaWNmaWxlcy5jb20vY29udm5leHQvY29udm5leHR2Mi9pbTIyay9jb252bmV4dHYyX2h1Z2VfMjJrXzUxMl9lbWEucHQpfAoKIyMgSW5zdGFsbGF0aW9uClBsZWFzZSBjaGVjayBbSU5TVEFMTC5tZF0oSU5TVEFMTC5tZCkgZm9yIGluc3RhbGxhdGlvbiBpbnN0cnVjdGlvbnMuIAoKIyMgRXZhbHVhdGlvbgpXZSBwcm92aWRlIGV4YW1wbGUgZXZhbHVhdGlvbiBjb21tYW5kcyBmb3IgQ29udk5lWHQgVjItQmFzZToKClNpbmdsZS1HUFUKYGBgCnB5dGhvbiBtYWluX2ZpbmV0dW5lLnB5IFwKLS1tb2RlbCBjb252bmV4dHYyX2Jhc2UgXAotLWV2YWwgdHJ1ZSBcCi0tcmVzdW1lIC9wYXRoL3RvL2NoZWNrcG9pbnQgXAotLWlucHV0X3NpemUgMjI0IFwKLS1kYXRhX3BhdGggL3BhdGgvdG8vaW1hZ2VuZXQtMWsgXApgYGAKTXVsdGktR1BVCmBgYApweXRob24gLW0gdG9yY2guZGlzdHJpYnV0ZWQubGF1bmNoIC0tbnByb2NfcGVyX25vZGU9OCBtYWluX2ZpbmV0dW5lLnB5IFwKLS1tb2RlbCBjb252bmV4dHYyX2Jhc2UgXAotLWV2YWwgdHJ1ZSBcCi0tcmVzdW1lIC9wYXRoL3RvL2NoZWNrcG9pbnQgXAotLWlucHV0X3NpemUgMjI0IFwKLS1kYXRhX3BhdGggL3BhdGgvdG8vaW1hZ2VuZXQtMWsgXApgYGAKCi0gRm9yIGV2YWx1YXRpbmcgb3RoZXIgbW9kZWwgdmFyaWFudHMsIGNoYW5nZSBgLS1tb2RlbGAsIGAtLXJlc3VtZWAsIGAtLWlucHV0X3NpemVgIGFjY29yZGluZ2x5LiBVUkxzIGZvciB0aGUgcHJlLXRyYWluZWQgbW9kZWxzIGNhbiBiZSBmb3VuZCBmcm9tIHRoZSByZXN1bHQgdGFibGVzLiAKLSBTZXR0aW5nIG1vZGVsLXNwZWNpZmljIGAtLWRyb3BfcGF0aGAgaXMgbm90IHN0cmljdGx5IHJlcXVpcmVkIGluIGV2YWx1YXRpb24sIGFzIHRoZSBgRHJvcFBhdGhgIG1vZHVsZSBpbiB0aW1tIGJlaGF2ZXMgdGhlIHNhbWUgZHVyaW5nIGV2YWx1YXRpb247IGJ1dCBpdCBpcyByZXF1aXJlZCBpbiB0cmFpbmluZy4gU2VlIFtUUkFJTklORy5tZF0oVFJBSU5JTkcubWQpIG9yIG91ciBwYXBlciAoYXBwZW5kaXgpIGZvciB0aGUgdmFsdWVzIHVzZWQgZm9yIGRpZmZlcmVudCBtb2RlbHMuCgojIyBUcmFpbmluZwpTZWUgW1RSQUlOSU5HLm1kXShUUkFJTklORy5tZCkgZm9yIHByZS10cmFpbmluZyBhbmQgZmluZS10dW5pbmcgaW5zdHJ1Y3Rpb25zLgoKIyMgQWNrbm93bGVkZ2VtZW50ClRoaXMgcmVwb3NpdG9yeSBib3Jyb3dzIGZyb20gW3RpbW1dKGh0dHBzOi8vZ2l0aHViLmNvbS9yd2lnaHRtYW4vcHl0b3JjaC1pbWFnZS1tb2RlbHMpLCBbQ29udk5lWHRdKGh0dHBzOi8vZ2l0aHViLmNvbS9mYWNlYm9va3Jlc2VhcmNoL0NvbnZOZVh0KSBhbmQgW01BRV0oaHR0cHM6Ly9naXRodWIuY29tL2ZhY2Vib29rcmVzZWFyY2gvbWFlKS4KCldlIHRoYW5rIFJvc3MgV2lnaHRtYW4gZm9yIHRoZSBpbml0aWFsIGRlc2lnbiBvZiB0aGUgc21hbGwtY29tcHV0ZSBDb252TmVYdCBtb2RlbCB2YXJpYW50cyBhbmQgdGhlIGFzc29jaWF0ZWQgdHJhaW5pbmcgcmVjaXBlLiBXZSBhbHNvIGFwcHJlY2lhdGUgdGhlIGhlbHBmdWwgZGlzY3Vzc2lvbnMgYW5kIGZlZWRiYWNrIHByb3ZpZGVkIGJ5IEthaW1pbmcgSGUuCgojIyBMaWNlbnNlClRoaXMgcHJvamVjdCBpcyByZWxlYXNlZCB1bmRlciB0aGUgTUlUIGxpY2Vuc2UgZXhjZXB0IEltYWdlTmV0IHByZS10cmFpbmVkIGFuZCBmaW5lLXR1bmVkIG1vZGVscyB3aGljaCBhcmUgbGljZW5zZWQgdW5kZXIgYSBDQy1CWS1OQy4gUGxlYXNlIHNlZSB0aGUgW0xJQ0VOU0VdKExJQ0VOU0UpIGZpbGUgZm9yIG1vcmUgaW5mb3JtYXRpb24uCgojIyBDaXRhdGlvbgpJZiB5b3UgZmluZCB0aGlzIHJlcG9zaXRvcnkgaGVscGZ1bCwgcGxlYXNlIGNvbnNpZGVyIGNpdGluZzoKYGBgYmlidGV4CkBhcnRpY2xle1dvbzIwMjNDb252TmVYdFYyLAogIHRpdGxlPXtDb252TmVYdCBWMjogQ28tZGVzaWduaW5nIGFuZCBTY2FsaW5nIENvbnZOZXRzIHdpdGggTWFza2VkIEF1dG9lbmNvZGVyc30sCiAgYXV0aG9yPXtTYW5naHl1biBXb28sIFNob3ViaGlrIERlYm5hdGgsIFJvbmdoYW5nIEh1LCBYaW5sZWkgQ2hlbiwgWmh1YW5nIExpdSwgSW4gU28gS3dlb24gYW5kIFNhaW5pbmcgWGllfSwKICB5ZWFyPXsyMDIzfSwKICBqb3VybmFsPXthclhpdiBwcmVwcmludCBhclhpdjoyMzAxLjAwODA4fSwKfQpgYGAK', 'vendor/models/__init__.py': '', 'vendor/models/convnextv2.py': 'IyBDb3B5cmlnaHQgKGMpIE1ldGEgUGxhdGZvcm1zLCBJbmMuIGFuZCBhZmZpbGlhdGVzLgoKIyBBbGwgcmlnaHRzIHJlc2VydmVkLgoKIyBUaGlzIHNvdXJjZSBjb2RlIGlzIGxpY2Vuc2VkIHVuZGVyIHRoZSBsaWNlbnNlIGZvdW5kIGluIHRoZQojIExJQ0VOU0UgZmlsZSBpbiB0aGUgcm9vdCBkaXJlY3Rvcnkgb2YgdGhpcyBzb3VyY2UgdHJlZS4KCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHRpbW0ubW9kZWxzLmxheWVycyBpbXBvcnQgdHJ1bmNfbm9ybWFsXywgRHJvcFBhdGgKZnJvbSAudXRpbHMgaW1wb3J0IExheWVyTm9ybSwgR1JOCgpjbGFzcyBCbG9jayhubi5Nb2R1bGUpOgogICAgIiIiIENvbnZOZVh0VjIgQmxvY2suCiAgICAKICAgIEFyZ3M6CiAgICAgICAgZGltIChpbnQpOiBOdW1iZXIgb2YgaW5wdXQgY2hhbm5lbHMuCiAgICAgICAgZHJvcF9wYXRoIChmbG9hdCk6IFN0b2NoYXN0aWMgZGVwdGggcmF0ZS4gRGVmYXVsdDogMC4wCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkaW0sIGRyb3BfcGF0aD0wLik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5kd2NvbnYgPSBubi5Db252MmQoZGltLCBkaW0sIGtlcm5lbF9zaXplPTcsIHBhZGRpbmc9MywgZ3JvdXBzPWRpbSkgIyBkZXB0aHdpc2UgY29udgogICAgICAgIHNlbGYubm9ybSA9IExheWVyTm9ybShkaW0sIGVwcz0xZS02KQogICAgICAgIHNlbGYucHdjb252MSA9IG5uLkxpbmVhcihkaW0sIDQgKiBkaW0pICMgcG9pbnR3aXNlLzF4MSBjb252cywgaW1wbGVtZW50ZWQgd2l0aCBsaW5lYXIgbGF5ZXJzCiAgICAgICAgc2VsZi5hY3QgPSBubi5HRUxVKCkKICAgICAgICBzZWxmLmdybiA9IEdSTig0ICogZGltKQogICAgICAgIHNlbGYucHdjb252MiA9IG5uLkxpbmVhcig0ICogZGltLCBkaW0pCiAgICAgICAgc2VsZi5kcm9wX3BhdGggPSBEcm9wUGF0aChkcm9wX3BhdGgpIGlmIGRyb3BfcGF0aCA+IDAuIGVsc2Ugbm4uSWRlbnRpdHkoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIGlucHV0ID0geAogICAgICAgIHggPSBzZWxmLmR3Y29udih4KQogICAgICAgIHggPSB4LnBlcm11dGUoMCwgMiwgMywgMSkgIyAoTiwgQywgSCwgVykgLT4gKE4sIEgsIFcsIEMpCiAgICAgICAgeCA9IHNlbGYubm9ybSh4KQogICAgICAgIHggPSBzZWxmLnB3Y29udjEoeCkKICAgICAgICB4ID0gc2VsZi5hY3QoeCkKICAgICAgICB4ID0gc2VsZi5ncm4oeCkKICAgICAgICB4ID0gc2VsZi5wd2NvbnYyKHgpCiAgICAgICAgeCA9IHgucGVybXV0ZSgwLCAzLCAxLCAyKSAjIChOLCBILCBXLCBDKSAtPiAoTiwgQywgSCwgVykKCiAgICAgICAgeCA9IGlucHV0ICsgc2VsZi5kcm9wX3BhdGgoeCkKICAgICAgICByZXR1cm4geAoKY2xhc3MgQ29udk5lWHRWMihubi5Nb2R1bGUpOgogICAgIiIiIENvbnZOZVh0IFYyCiAgICAgICAgCiAgICBBcmdzOgogICAgICAgIGluX2NoYW5zIChpbnQpOiBOdW1iZXIgb2YgaW5wdXQgaW1hZ2UgY2hhbm5lbHMuIERlZmF1bHQ6IDMKICAgICAgICBudW1fY2xhc3NlcyAoaW50KTogTnVtYmVyIG9mIGNsYXNzZXMgZm9yIGNsYXNzaWZpY2F0aW9uIGhlYWQuIERlZmF1bHQ6IDEwMDAKICAgICAgICBkZXB0aHMgKHR1cGxlKGludCkpOiBOdW1iZXIgb2YgYmxvY2tzIGF0IGVhY2ggc3RhZ2UuIERlZmF1bHQ6IFszLCAzLCA5LCAzXQogICAgICAgIGRpbXMgKGludCk6IEZlYXR1cmUgZGltZW5zaW9uIGF0IGVhY2ggc3RhZ2UuIERlZmF1bHQ6IFs5NiwgMTkyLCAzODQsIDc2OF0KICAgICAgICBkcm9wX3BhdGhfcmF0ZSAoZmxvYXQpOiBTdG9jaGFzdGljIGRlcHRoIHJhdGUuIERlZmF1bHQ6IDAuCiAgICAgICAgaGVhZF9pbml0X3NjYWxlIChmbG9hdCk6IEluaXQgc2NhbGluZyB2YWx1ZSBmb3IgY2xhc3NpZmllciB3ZWlnaHRzIGFuZCBiaWFzZXMuIERlZmF1bHQ6IDEuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaGFucz0zLCBudW1fY2xhc3Nlcz0xMDAwLCAKICAgICAgICAgICAgICAgICBkZXB0aHM9WzMsIDMsIDksIDNdLCBkaW1zPVs5NiwgMTkyLCAzODQsIDc2OF0sIAogICAgICAgICAgICAgICAgIGRyb3BfcGF0aF9yYXRlPTAuLCBoZWFkX2luaXRfc2NhbGU9MS4KICAgICAgICAgICAgICAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZGVwdGhzID0gZGVwdGhzCiAgICAgICAgc2VsZi5kb3duc2FtcGxlX2xheWVycyA9IG5uLk1vZHVsZUxpc3QoKSAjIHN0ZW0gYW5kIDMgaW50ZXJtZWRpYXRlIGRvd25zYW1wbGluZyBjb252IGxheWVycwogICAgICAgIHN0ZW0gPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoaW5fY2hhbnMsIGRpbXNbMF0sIGtlcm5lbF9zaXplPTQsIHN0cmlkZT00KSwKICAgICAgICAgICAgTGF5ZXJOb3JtKGRpbXNbMF0sIGVwcz0xZS02LCBkYXRhX2Zvcm1hdD0iY2hhbm5lbHNfZmlyc3QiKQogICAgICAgICkKICAgICAgICBzZWxmLmRvd25zYW1wbGVfbGF5ZXJzLmFwcGVuZChzdGVtKQogICAgICAgIGZvciBpIGluIHJhbmdlKDMpOgogICAgICAgICAgICBkb3duc2FtcGxlX2xheWVyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgICAgICAgICBMYXllck5vcm0oZGltc1tpXSwgZXBzPTFlLTYsIGRhdGFfZm9ybWF0PSJjaGFubmVsc19maXJzdCIpLAogICAgICAgICAgICAgICAgICAgIG5uLkNvbnYyZChkaW1zW2ldLCBkaW1zW2krMV0sIGtlcm5lbF9zaXplPTIsIHN0cmlkZT0yKSwKICAgICAgICAgICAgKQogICAgICAgICAgICBzZWxmLmRvd25zYW1wbGVfbGF5ZXJzLmFwcGVuZChkb3duc2FtcGxlX2xheWVyKQoKICAgICAgICBzZWxmLnN0YWdlcyA9IG5uLk1vZHVsZUxpc3QoKSAjIDQgZmVhdHVyZSByZXNvbHV0aW9uIHN0YWdlcywgZWFjaCBjb25zaXN0aW5nIG9mIG11bHRpcGxlIHJlc2lkdWFsIGJsb2NrcwogICAgICAgIGRwX3JhdGVzPVt4Lml0ZW0oKSBmb3IgeCBpbiB0b3JjaC5saW5zcGFjZSgwLCBkcm9wX3BhdGhfcmF0ZSwgc3VtKGRlcHRocykpXSAKICAgICAgICBjdXIgPSAwCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UoNCk6CiAgICAgICAgICAgIHN0YWdlID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgICAgICpbQmxvY2soZGltPWRpbXNbaV0sIGRyb3BfcGF0aD1kcF9yYXRlc1tjdXIgKyBqXSkgZm9yIGogaW4gcmFuZ2UoZGVwdGhzW2ldKV0KICAgICAgICAgICAgKQogICAgICAgICAgICBzZWxmLnN0YWdlcy5hcHBlbmQoc3RhZ2UpCiAgICAgICAgICAgIGN1ciArPSBkZXB0aHNbaV0KCiAgICAgICAgc2VsZi5ub3JtID0gbm4uTGF5ZXJOb3JtKGRpbXNbLTFdLCBlcHM9MWUtNikgIyBmaW5hbCBub3JtIGxheWVyCiAgICAgICAgc2VsZi5oZWFkID0gbm4uTGluZWFyKGRpbXNbLTFdLCBudW1fY2xhc3NlcykKCiAgICAgICAgc2VsZi5hcHBseShzZWxmLl9pbml0X3dlaWdodHMpCiAgICAgICAgc2VsZi5oZWFkLndlaWdodC5kYXRhLm11bF8oaGVhZF9pbml0X3NjYWxlKQogICAgICAgIHNlbGYuaGVhZC5iaWFzLmRhdGEubXVsXyhoZWFkX2luaXRfc2NhbGUpCgogICAgZGVmIF9pbml0X3dlaWdodHMoc2VsZiwgbSk6CiAgICAgICAgaWYgaXNpbnN0YW5jZShtLCAobm4uQ29udjJkLCBubi5MaW5lYXIpKToKICAgICAgICAgICAgdHJ1bmNfbm9ybWFsXyhtLndlaWdodCwgc3RkPS4wMikKICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8obS5iaWFzLCAwKQoKICAgIGRlZiBmb3J3YXJkX2ZlYXR1cmVzKHNlbGYsIHgpOgogICAgICAgIGZvciBpIGluIHJhbmdlKDQpOgogICAgICAgICAgICB4ID0gc2VsZi5kb3duc2FtcGxlX2xheWVyc1tpXSh4KQogICAgICAgICAgICB4ID0gc2VsZi5zdGFnZXNbaV0oeCkKICAgICAgICByZXR1cm4gc2VsZi5ub3JtKHgubWVhbihbLTIsIC0xXSkpICMgZ2xvYmFsIGF2ZXJhZ2UgcG9vbGluZywgKE4sIEMsIEgsIFcpIC0+IChOLCBDKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHggPSBzZWxmLmZvcndhcmRfZmVhdHVyZXMoeCkKICAgICAgICB4ID0gc2VsZi5oZWFkKHgpCiAgICAgICAgcmV0dXJuIHgKCmRlZiBjb252bmV4dHYyX2F0dG8oKiprd2FyZ3MpOgogICAgbW9kZWwgPSBDb252TmVYdFYyKGRlcHRocz1bMiwgMiwgNiwgMl0sIGRpbXM9WzQwLCA4MCwgMTYwLCAzMjBdLCAqKmt3YXJncykKICAgIHJldHVybiBtb2RlbAoKZGVmIGNvbnZuZXh0djJfZmVtdG8oKiprd2FyZ3MpOgogICAgbW9kZWwgPSBDb252TmVYdFYyKGRlcHRocz1bMiwgMiwgNiwgMl0sIGRpbXM9WzQ4LCA5NiwgMTkyLCAzODRdLCAqKmt3YXJncykKICAgIHJldHVybiBtb2RlbAoKZGVmIGNvbnZuZXh0X3BpY28oKiprd2FyZ3MpOgogICAgbW9kZWwgPSBDb252TmVYdFYyKGRlcHRocz1bMiwgMiwgNiwgMl0sIGRpbXM9WzY0LCAxMjgsIDI1NiwgNTEyXSwgKiprd2FyZ3MpCiAgICByZXR1cm4gbW9kZWwKCmRlZiBjb252bmV4dHYyX25hbm8oKiprd2FyZ3MpOgogICAgbW9kZWwgPSBDb252TmVYdFYyKGRlcHRocz1bMiwgMiwgOCwgMl0sIGRpbXM9WzgwLCAxNjAsIDMyMCwgNjQwXSwgKiprd2FyZ3MpCiAgICByZXR1cm4gbW9kZWwKCmRlZiBjb252bmV4dHYyX3RpbnkoKiprd2FyZ3MpOgogICAgbW9kZWwgPSBDb252TmVYdFYyKGRlcHRocz1bMywgMywgOSwgM10sIGRpbXM9Wzk2LCAxOTIsIDM4NCwgNzY4XSwgKiprd2FyZ3MpCiAgICByZXR1cm4gbW9kZWwKCmRlZiBjb252bmV4dHYyX2Jhc2UoKiprd2FyZ3MpOgogICAgbW9kZWwgPSBDb252TmVYdFYyKGRlcHRocz1bMywgMywgMjcsIDNdLCBkaW1zPVsxMjgsIDI1NiwgNTEyLCAxMDI0XSwgKiprd2FyZ3MpCiAgICByZXR1cm4gbW9kZWwKCmRlZiBjb252bmV4dHYyX2xhcmdlKCoqa3dhcmdzKToKICAgIG1vZGVsID0gQ29udk5lWHRWMihkZXB0aHM9WzMsIDMsIDI3LCAzXSwgZGltcz1bMTkyLCAzODQsIDc2OCwgMTUzNl0sICoqa3dhcmdzKQogICAgcmV0dXJuIG1vZGVsCgpkZWYgY29udm5leHR2Ml9odWdlKCoqa3dhcmdzKToKICAgIG1vZGVsID0gQ29udk5lWHRWMihkZXB0aHM9WzMsIDMsIDI3LCAzXSwgZGltcz1bMzUyLCA3MDQsIDE0MDgsIDI4MTZdLCAqKmt3YXJncykKICAgIHJldHVybiBtb2RlbA==', 'vendor/models/convnextv2_sparse.py': 'IyBDb3B5cmlnaHQgKGMpIE1ldGEgUGxhdGZvcm1zLCBJbmMuIGFuZCBhZmZpbGlhdGVzLgoKIyBBbGwgcmlnaHRzIHJlc2VydmVkLgoKIyBUaGlzIHNvdXJjZSBjb2RlIGlzIGxpY2Vuc2VkIHVuZGVyIHRoZSBsaWNlbnNlIGZvdW5kIGluIHRoZQojIExJQ0VOU0UgZmlsZSBpbiB0aGUgcm9vdCBkaXJlY3Rvcnkgb2YgdGhpcyBzb3VyY2UgdHJlZS4KCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmZyb20gdGltbS5tb2RlbHMubGF5ZXJzIGltcG9ydCB0cnVuY19ub3JtYWxfCgpmcm9tIC51dGlscyBpbXBvcnQgKAogICAgTGF5ZXJOb3JtLAogICAgTWlua293c2tpTGF5ZXJOb3JtLAogICAgTWlua293c2tpR1JOLAogICAgTWlua293c2tpRHJvcFBhdGgKKQpmcm9tIE1pbmtvd3NraUVuZ2luZSBpbXBvcnQgKAogICAgTWlua293c2tpQ29udm9sdXRpb24sCiAgICBNaW5rb3dza2lEZXB0aHdpc2VDb252b2x1dGlvbiwKICAgIE1pbmtvd3NraUxpbmVhciwKICAgIE1pbmtvd3NraUdFTFUKKQpmcm9tIE1pbmtvd3NraU9wcyBpbXBvcnQgKAogICAgdG9fc3BhcnNlLAopCgpjbGFzcyBCbG9jayhubi5Nb2R1bGUpOgogICAgIiIiIFNwYXJzZSBDb252TmVYdFYyIEJsb2NrLiAKCiAgICBBcmdzOgogICAgICAgIGRpbSAoaW50KTogTnVtYmVyIG9mIGlucHV0IGNoYW5uZWxzLgogICAgICAgIGRyb3BfcGF0aCAoZmxvYXQpOiBTdG9jaGFzdGljIGRlcHRoIHJhdGUuIERlZmF1bHQ6IDAuMAogICAgICAgIGxheWVyX3NjYWxlX2luaXRfdmFsdWUgKGZsb2F0KTogSW5pdCB2YWx1ZSBmb3IgTGF5ZXIgU2NhbGUuIERlZmF1bHQ6IDFlLTYuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkaW0sIGRyb3BfcGF0aD0wLiwgRD0zKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmR3Y29udiA9IE1pbmtvd3NraURlcHRod2lzZUNvbnZvbHV0aW9uKGRpbSwga2VybmVsX3NpemU9NywgYmlhcz1UcnVlLCBkaW1lbnNpb249RCkKICAgICAgICBzZWxmLm5vcm0gPSBNaW5rb3dza2lMYXllck5vcm0oZGltLCAxZS02KQogICAgICAgIHNlbGYucHdjb252MSA9IE1pbmtvd3NraUxpbmVhcihkaW0sIDQgKiBkaW0pICAgCiAgICAgICAgc2VsZi5hY3QgPSBNaW5rb3dza2lHRUxVKCkKICAgICAgICBzZWxmLnB3Y29udjIgPSBNaW5rb3dza2lMaW5lYXIoNCAqIGRpbSwgZGltKQogICAgICAgIHNlbGYuZ3JuID0gTWlua293c2tpR1JOKDQgICogZGltKQogICAgICAgIHNlbGYuZHJvcF9wYXRoID0gTWlua293c2tpRHJvcFBhdGgoZHJvcF9wYXRoKQogICAgCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBpbnB1dCA9IHgKICAgICAgICB4ID0gc2VsZi5kd2NvbnYoeCkKICAgICAgICB4ID0gc2VsZi5ub3JtKHgpCiAgICAgICAgeCA9IHNlbGYucHdjb252MSh4KQogICAgICAgIHggPSBzZWxmLmFjdCh4KQogICAgICAgIHggPSBzZWxmLmdybih4KQogICAgICAgIHggPSBzZWxmLnB3Y29udjIoeCkKICAgICAgICB4ID0gaW5wdXQgKyBzZWxmLmRyb3BfcGF0aCh4KQogICAgICAgIHJldHVybiB4CgpjbGFzcyBTcGFyc2VDb252TmVYdFYyKG5uLk1vZHVsZSk6CiAgICAiIiIgU3BhcnNlIENvbnZOZVh0VjIuCiAgICAgICAgCiAgICBBcmdzOgogICAgICAgIGluX2NoYW5zIChpbnQpOiBOdW1iZXIgb2YgaW5wdXQgaW1hZ2UgY2hhbm5lbHMuIERlZmF1bHQ6IDMKICAgICAgICBudW1fY2xhc3NlcyAoaW50KTogTnVtYmVyIG9mIGNsYXNzZXMgZm9yIGNsYXNzaWZpY2F0aW9uIGhlYWQuIERlZmF1bHQ6IDEwMDAKICAgICAgICBkZXB0aHMgKHR1cGxlKGludCkpOiBOdW1iZXIgb2YgYmxvY2tzIGF0IGVhY2ggc3RhZ2UuIERlZmF1bHQ6IFszLCAzLCA5LCAzXQogICAgICAgIGRpbXMgKGludCk6IEZlYXR1cmUgZGltZW5zaW9uIGF0IGVhY2ggc3RhZ2UuIERlZmF1bHQ6IFs5NiwgMTkyLCAzODQsIDc2OF0KICAgICAgICBkcm9wX3BhdGhfcmF0ZSAoZmxvYXQpOiBTdG9jaGFzdGljIGRlcHRoIHJhdGUuIERlZmF1bHQ6IDAuCiAgICAgICAgaGVhZF9pbml0X3NjYWxlIChmbG9hdCk6IEluaXQgc2NhbGluZyB2YWx1ZSBmb3IgY2xhc3NpZmllciB3ZWlnaHRzIGFuZCBiaWFzZXMuIERlZmF1bHQ6IDEuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCAKICAgICAgICAgICAgICAgICBpbl9jaGFucz0zLCAKICAgICAgICAgICAgICAgICBudW1fY2xhc3Nlcz0xMDAwLCAKICAgICAgICAgICAgICAgICBkZXB0aHM9WzMsIDMsIDksIDNdLCAKICAgICAgICAgICAgICAgICBkaW1zPVs5NiwgMTkyLCAzODQsIDc2OF0sIAogICAgICAgICAgICAgICAgIGRyb3BfcGF0aF9yYXRlPTAuLCAKICAgICAgICAgICAgICAgICBEPTMpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZGVwdGhzID0gZGVwdGhzCiAgICAgICAgc2VsZi5udW1fY2xhc3NlcyA9IG51bV9jbGFzc2VzCiAgICAgICAgc2VsZi5kb3duc2FtcGxlX2xheWVycyA9IG5uLk1vZHVsZUxpc3QoKSAjIHN0ZW0gYW5kIDMgaW50ZXJtZWRpYXRlIGRvd25zYW1wbGluZyBjb252IGxheWVycwogICAgICAgIHN0ZW0gPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoaW5fY2hhbnMsIGRpbXNbMF0sIGtlcm5lbF9zaXplPTQsIHN0cmlkZT00KSwKICAgICAgICAgICAgTGF5ZXJOb3JtKGRpbXNbMF0sIGVwcz0xZS02LCBkYXRhX2Zvcm1hdD0iY2hhbm5lbHNfZmlyc3QiKQogICAgICAgICkKICAgICAgICBzZWxmLmRvd25zYW1wbGVfbGF5ZXJzLmFwcGVuZChzdGVtKQogICAgICAgIGZvciBpIGluIHJhbmdlKDMpOgogICAgICAgICAgICBkb3duc2FtcGxlX2xheWVyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgICAgIE1pbmtvd3NraUxheWVyTm9ybShkaW1zW2ldLCBlcHM9MWUtNiksCiAgICAgICAgICAgICAgICBNaW5rb3dza2lDb252b2x1dGlvbihkaW1zW2ldLCBkaW1zW2krMV0sIGtlcm5lbF9zaXplPTIsIHN0cmlkZT0yLCBiaWFzPVRydWUsIGRpbWVuc2lvbj1EKQogICAgICAgICAgICApCiAgICAgICAgICAgIHNlbGYuZG93bnNhbXBsZV9sYXllcnMuYXBwZW5kKGRvd25zYW1wbGVfbGF5ZXIpCiAgICAgICAgCiAgICAgICAgc2VsZi5zdGFnZXMgPSBubi5Nb2R1bGVMaXN0KCkgIyA0IGZlYXR1cmUgcmVzb2x1dGlvbiBzdGFnZXMsIGVhY2ggY29uc2lzdGluZyBvZiBtdWx0aXBsZSByZXNpZHVhbCBibG9ja3MKICAgICAgICBkcF9yYXRlcz1beC5pdGVtKCkgZm9yIHggaW4gdG9yY2gubGluc3BhY2UoMCwgZHJvcF9wYXRoX3JhdGUsIHN1bShkZXB0aHMpKV0gCiAgICAgICAgY3VyID0gMAogICAgICAgIGZvciBpIGluIHJhbmdlKDQpOgogICAgICAgICAgICBzdGFnZSA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgICAgICAqW0Jsb2NrKGRpbT1kaW1zW2ldLCBkcm9wX3BhdGg9ZHBfcmF0ZXNbY3VyICsgal0sIEQ9RCkgZm9yIGogaW4gcmFuZ2UoZGVwdGhzW2ldKV0KICAgICAgICAgICAgKQogICAgICAgICAgICBzZWxmLnN0YWdlcy5hcHBlbmQoc3RhZ2UpCiAgICAgICAgICAgIGN1ciArPSBkZXB0aHNbaV0KCiAgICAgICAgc2VsZi5hcHBseShzZWxmLl9pbml0X3dlaWdodHMpCiAgICAgICAgCiAgICBkZWYgX2luaXRfd2VpZ2h0cyhzZWxmLCBtKToKICAgICAgICBpZiBpc2luc3RhbmNlKG0sIE1pbmtvd3NraUNvbnZvbHV0aW9uKToKICAgICAgICAgICAgdHJ1bmNfbm9ybWFsXyhtLmtlcm5lbCwgc3RkPS4wMikKICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8obS5iaWFzLCAwKQogICAgICAgIGlmIGlzaW5zdGFuY2UobSwgTWlua293c2tpRGVwdGh3aXNlQ29udm9sdXRpb24pOgogICAgICAgICAgICB0cnVuY19ub3JtYWxfKG0ua2VybmVsLCBzdGQ9LjAyKQogICAgICAgICAgICBubi5pbml0LmNvbnN0YW50XyhtLmJpYXMsIDApCiAgICAgICAgaWYgaXNpbnN0YW5jZShtLCBNaW5rb3dza2lMaW5lYXIpOgogICAgICAgICAgICB0cnVuY19ub3JtYWxfKG0ubGluZWFyLndlaWdodCwgc3RkPS4wMikKICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8obS5saW5lYXIuYmlhcywgMCkKCiAgICBkZWYgdXBzYW1wbGVfbWFzayhzZWxmLCBtYXNrLCBzY2FsZSk6CiAgICAgICAgYXNzZXJ0IGxlbihtYXNrLnNoYXBlKSA9PSAyCiAgICAgICAgcCA9IGludChtYXNrLnNoYXBlWzFdICoqIC41KQogICAgICAgIHJldHVybiBtYXNrLnJlc2hhcGUoLTEsIHAsIHApLlwKICAgICAgICAgICAgICAgICAgICByZXBlYXRfaW50ZXJsZWF2ZShzY2FsZSwgYXhpcz0xKS5cCiAgICAgICAgICAgICAgICAgICAgcmVwZWF0X2ludGVybGVhdmUoc2NhbGUsIGF4aXM9MikKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4LCBtYXNrKToKICAgICAgICBudW1fc3RhZ2VzID0gbGVuKHNlbGYuc3RhZ2VzKQogICAgICAgIG1hc2sgPSBzZWxmLnVwc2FtcGxlX21hc2sobWFzaywgMioqKG51bV9zdGFnZXMtMSkpICAgICAgICAKICAgICAgICBtYXNrID0gbWFzay51bnNxdWVlemUoMSkudHlwZV9hcyh4KQogICAgICAgIAogICAgICAgICMgcGF0Y2ggZW1iZWRkaW5nCiAgICAgICAgeCA9IHNlbGYuZG93bnNhbXBsZV9sYXllcnNbMF0oeCkKICAgICAgICB4ICo9ICgxLi1tYXNrKQogICAgICAgIAogICAgICAgICMgc3BhcnNlIGVuY29kaW5nCiAgICAgICAgeCA9IHRvX3NwYXJzZSh4KQogICAgICAgIGZvciBpIGluIHJhbmdlKDQpOgogICAgICAgICAgICB4ID0gc2VsZi5kb3duc2FtcGxlX2xheWVyc1tpXSh4KSBpZiBpID4gMCBlbHNlIHgKICAgICAgICAgICAgeCA9IHNlbGYuc3RhZ2VzW2ldKHgpCiAgICAgICAgCiAgICAgICAgIyBkZW5zaWZ5CiAgICAgICAgeCA9IHguZGVuc2UoKVswXQogICAgICAgIHJldHVybiB4', 'vendor/models/fcmae.py': 'IyBDb3B5cmlnaHQgKGMpIE1ldGEgUGxhdGZvcm1zLCBJbmMuIGFuZCBhZmZpbGlhdGVzLgoKIyBBbGwgcmlnaHRzIHJlc2VydmVkLgoKIyBUaGlzIHNvdXJjZSBjb2RlIGlzIGxpY2Vuc2VkIHVuZGVyIHRoZSBsaWNlbnNlIGZvdW5kIGluIHRoZQojIExJQ0VOU0UgZmlsZSBpbiB0aGUgcm9vdCBkaXJlY3Rvcnkgb2YgdGhpcyBzb3VyY2UgdHJlZS4KCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgpmcm9tIE1pbmtvd3NraUVuZ2luZSBpbXBvcnQgKAogICAgTWlua293c2tpQ29udm9sdXRpb24sCiAgICBNaW5rb3dza2lEZXB0aHdpc2VDb252b2x1dGlvbiwKICAgIE1pbmtvd3NraUxpbmVhciwKKQoKZnJvbSB0aW1tLm1vZGVscy5sYXllcnMgaW1wb3J0IHRydW5jX25vcm1hbF8KZnJvbSAuY29udm5leHR2Ml9zcGFyc2UgaW1wb3J0IFNwYXJzZUNvbnZOZVh0VjIKZnJvbSAuY29udm5leHR2MiBpbXBvcnQgQmxvY2sKCmNsYXNzIEZDTUFFKG5uLk1vZHVsZSk6CiAgICAiIiIgRnVsbHkgQ29udm9sdXRpb25hbCBNYXNrZWQgQXV0b2VuY29kZXIgd2l0aCBDb252TmVYdFYyIGJhY2tib25lCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXygKICAgICAgICAgICAgICAgIHNlbGYsCiAgICAgICAgICAgICAgICBpbWdfc2l6ZT0yMjQsCiAgICAgICAgICAgICAgICBpbl9jaGFucz0zLAogICAgICAgICAgICAgICAgZGVwdGhzPVszLCAzLCA5LCAzXSwKICAgICAgICAgICAgICAgIGRpbXM9Wzk2LCAxOTIsIDM4NCwgNzY4XSwKICAgICAgICAgICAgICAgIGRlY29kZXJfZGVwdGg9MSwKICAgICAgICAgICAgICAgIGRlY29kZXJfZW1iZWRfZGltPTUxMiwKICAgICAgICAgICAgICAgIHBhdGNoX3NpemU9MzIsCiAgICAgICAgICAgICAgICBtYXNrX3JhdGlvPTAuNiwKICAgICAgICAgICAgICAgIG5vcm1fcGl4X2xvc3M9RmFsc2UpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQoKICAgICAgICAjIGNvbmZpZ3MKICAgICAgICBzZWxmLmltZ19zaXplID0gaW1nX3NpemUKICAgICAgICBzZWxmLmRlcHRocyA9IGRlcHRocwogICAgICAgIHNlbGYuaW1kcyA9IGRpbXMKICAgICAgICBzZWxmLnBhdGNoX3NpemUgPSBwYXRjaF9zaXplCiAgICAgICAgc2VsZi5tYXNrX3JhdGlvID0gbWFza19yYXRpbwogICAgICAgIHNlbGYubnVtX3BhdGNoZXMgPSAoaW1nX3NpemUgLy8gcGF0Y2hfc2l6ZSkgKiogMgogICAgICAgIHNlbGYuZGVjb2Rlcl9lbWJlZF9kaW0gPSBkZWNvZGVyX2VtYmVkX2RpbQogICAgICAgIHNlbGYuZGVjb2Rlcl9kZXB0aCA9IGRlY29kZXJfZGVwdGgKICAgICAgICBzZWxmLm5vcm1fcGl4X2xvc3MgPSBub3JtX3BpeF9sb3NzCgogICAgICAgICMgZW5jb2RlcgogICAgICAgIHNlbGYuZW5jb2RlciA9IFNwYXJzZUNvbnZOZVh0VjIoCiAgICAgICAgICAgIGluX2NoYW5zPWluX2NoYW5zLCBkZXB0aHM9ZGVwdGhzLCBkaW1zPWRpbXMsIEQ9MikKICAgICAgICAjIGRlY29kZXIKICAgICAgICBzZWxmLnByb2ogPSBubi5Db252MmQoCiAgICAgICAgICAgIGluX2NoYW5uZWxzPWRpbXNbLTFdLCAKICAgICAgICAgICAgb3V0X2NoYW5uZWxzPWRlY29kZXJfZW1iZWRfZGltLCAKICAgICAgICAgICAga2VybmVsX3NpemU9MSkKICAgICAgICAjIG1hc2sgdG9rZW5zCiAgICAgICAgc2VsZi5tYXNrX3Rva2VuID0gbm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKDEsIGRlY29kZXJfZW1iZWRfZGltLCAxLCAxKSkKICAgICAgICBkZWNvZGVyID0gW0Jsb2NrKAogICAgICAgICAgICBkaW09ZGVjb2Rlcl9lbWJlZF9kaW0sIAogICAgICAgICAgICBkcm9wX3BhdGg9MC4pIGZvciBpIGluIHJhbmdlKGRlY29kZXJfZGVwdGgpXQogICAgICAgIHNlbGYuZGVjb2RlciA9IG5uLlNlcXVlbnRpYWwoKmRlY29kZXIpCiAgICAgICAgIyBwcmVkCiAgICAgICAgc2VsZi5wcmVkID0gbm4uQ29udjJkKAogICAgICAgICAgICBpbl9jaGFubmVscz1kZWNvZGVyX2VtYmVkX2RpbSwKICAgICAgICAgICAgb3V0X2NoYW5uZWxzPXBhdGNoX3NpemUgKiogMiAqIGluX2NoYW5zLAogICAgICAgICAgICBrZXJuZWxfc2l6ZT0xKQoKICAgICAgICBzZWxmLmFwcGx5KHNlbGYuX2luaXRfd2VpZ2h0cykKCiAgICBkZWYgX2luaXRfd2VpZ2h0cyhzZWxmLCBtKToKICAgICAgICBpZiBpc2luc3RhbmNlKG0sIE1pbmtvd3NraUNvbnZvbHV0aW9uKToKICAgICAgICAgICAgdHJ1bmNfbm9ybWFsXyhtLmtlcm5lbCwgc3RkPS4wMikKICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8obS5iaWFzLCAwKQogICAgICAgIGlmIGlzaW5zdGFuY2UobSwgTWlua293c2tpRGVwdGh3aXNlQ29udm9sdXRpb24pOgogICAgICAgICAgICB0cnVuY19ub3JtYWxfKG0ua2VybmVsKQogICAgICAgICAgICBubi5pbml0LmNvbnN0YW50XyhtLmJpYXMsIDApCiAgICAgICAgaWYgaXNpbnN0YW5jZShtLCBNaW5rb3dza2lMaW5lYXIpOgogICAgICAgICAgICB0cnVuY19ub3JtYWxfKG0ubGluZWFyLndlaWdodCkKICAgICAgICAgICAgbm4uaW5pdC5jb25zdGFudF8obS5saW5lYXIuYmlhcywgMCkKICAgICAgICBpZiBpc2luc3RhbmNlKG0sIG5uLkNvbnYyZCk6CiAgICAgICAgICAgIHcgPSBtLndlaWdodC5kYXRhCiAgICAgICAgICAgIHRydW5jX25vcm1hbF8ody52aWV3KFt3LnNoYXBlWzBdLCAtMV0pKQogICAgICAgICAgICBubi5pbml0LmNvbnN0YW50XyhtLmJpYXMsIDApCiAgICAgICAgaWYgaXNpbnN0YW5jZShtLCBubi5MYXllck5vcm0pOgogICAgICAgICAgICBubi5pbml0LmNvbnN0YW50XyhtLmJpYXMsIDApCiAgICAgICAgICAgIG5uLmluaXQuY29uc3RhbnRfKG0ud2VpZ2h0LCAxLjApCiAgICAgICAgaWYgaGFzYXR0cihzZWxmLCAnbWFza190b2tlbicpOiAgICAKICAgICAgICAgICAgdG9yY2gubm4uaW5pdC5ub3JtYWxfKHNlbGYubWFza190b2tlbiwgc3RkPS4wMikKICAgIAogICAgZGVmIHBhdGNoaWZ5KHNlbGYsIGltZ3MpOgogICAgICAgICIiIgogICAgICAgIGltZ3M6IChOLCAzLCBILCBXKQogICAgICAgIHg6IChOLCBMLCBwYXRjaF9zaXplKioyICozKQogICAgICAgICIiIgogICAgICAgIHAgPSBzZWxmLnBhdGNoX3NpemUKICAgICAgICBhc3NlcnQgaW1ncy5zaGFwZVsyXSA9PSBpbWdzLnNoYXBlWzNdIGFuZCBpbWdzLnNoYXBlWzJdICUgcCA9PSAwCgogICAgICAgIGggPSB3ID0gaW1ncy5zaGFwZVsyXSAvLyBwCiAgICAgICAgeCA9IGltZ3MucmVzaGFwZShzaGFwZT0oaW1ncy5zaGFwZVswXSwgMywgaCwgcCwgdywgcCkpCiAgICAgICAgeCA9IHRvcmNoLmVpbnN1bSgnbmNocHdxLT5uaHdwcWMnLCB4KQogICAgICAgIHggPSB4LnJlc2hhcGUoc2hhcGU9KGltZ3Muc2hhcGVbMF0sIGggKiB3LCBwKioyICogMykpCiAgICAgICAgcmV0dXJuIHgKCiAgICBkZWYgdW5wYXRjaGlmeShzZWxmLCB4KToKICAgICAgICAiIiIKICAgICAgICB4OiAoTiwgTCwgcGF0Y2hfc2l6ZSoqMiAqMykKICAgICAgICBpbWdzOiAoTiwgMywgSCwgVykKICAgICAgICAiIiIKICAgICAgICBwID0gc2VsZi5wYXRjaF9zaXplCiAgICAgICAgaCA9IHcgPSBpbnQoeC5zaGFwZVsxXSoqLjUpCiAgICAgICAgYXNzZXJ0IGggKiB3ID09IHguc2hhcGVbMV0KICAgICAgICAKICAgICAgICB4ID0geC5yZXNoYXBlKHNoYXBlPSh4LnNoYXBlWzBdLCBoLCB3LCBwLCBwLCAzKSkKICAgICAgICB4ID0gdG9yY2guZWluc3VtKCduaHdwcWMtPm5jaHB3cScsIHgpCiAgICAgICAgaW1ncyA9IHgucmVzaGFwZShzaGFwZT0oeC5zaGFwZVswXSwgMywgaCAqIHAsIGggKiBwKSkKICAgICAgICByZXR1cm4gaW1ncwoKICAgIGRlZiBnZW5fcmFuZG9tX21hc2soc2VsZiwgeCwgbWFza19yYXRpbyk6CiAgICAgICAgTiA9IHguc2hhcGVbMF0KICAgICAgICBMID0gKHguc2hhcGVbMl0gLy8gc2VsZi5wYXRjaF9zaXplKSAqKiAyCiAgICAgICAgbGVuX2tlZXAgPSBpbnQoTCAqICgxIC0gbWFza19yYXRpbykpCgogICAgICAgIG5vaXNlID0gdG9yY2gucmFuZG4oTiwgTCwgZGV2aWNlPXguZGV2aWNlKQoKICAgICAgICAjIHNvcnQgbm9pc2UgZm9yIGVhY2ggc2FtcGxlCiAgICAgICAgaWRzX3NodWZmbGUgPSB0b3JjaC5hcmdzb3J0KG5vaXNlLCBkaW09MSkKICAgICAgICBpZHNfcmVzdG9yZSA9IHRvcmNoLmFyZ3NvcnQoaWRzX3NodWZmbGUsIGRpbT0xKQoKICAgICAgICAjIGdlbmVyYXRlIHRoZSBiaW5hcnkgbWFzazogMCBpcyBrZWVwIDEgaXMgcmVtb3ZlCiAgICAgICAgbWFzayA9IHRvcmNoLm9uZXMoW04sIExdLCBkZXZpY2U9eC5kZXZpY2UpCiAgICAgICAgbWFza1s6LCA6bGVuX2tlZXBdID0gMAogICAgICAgICMgdW5zaHVmZmxlIHRvIGdldCB0aGUgYmluYXJ5IG1hc2sKICAgICAgICBtYXNrID0gdG9yY2guZ2F0aGVyKG1hc2ssIGRpbT0xLCBpbmRleD1pZHNfcmVzdG9yZSkKICAgICAgICByZXR1cm4gbWFzawoKICAgIGRlZiB1cHNhbXBsZV9tYXNrKHNlbGYsIG1hc2ssIHNjYWxlKToKICAgICAgICBhc3NlcnQgbGVuKG1hc2suc2hhcGUpID09IDIKICAgICAgICBwID0gaW50KG1hc2suc2hhcGVbMV0gKiogLjUpCiAgICAgICAgcmV0dXJuIG1hc2sucmVzaGFwZSgtMSwgcCwgcCkuXAogICAgICAgICAgICAgICAgICAgIHJlcGVhdF9pbnRlcmxlYXZlKHNjYWxlLCBheGlzPTEpLlwKICAgICAgICAgICAgICAgICAgICByZXBlYXRfaW50ZXJsZWF2ZShzY2FsZSwgYXhpcz0yKQogICAgCiAgICBkZWYgZm9yd2FyZF9lbmNvZGVyKHNlbGYsIGltZ3MsIG1hc2tfcmF0aW8pOgogICAgICAgICMgZ2VuZXJhdGUgcmFuZG9tIG1hc2tzCiAgICAgICAgbWFzayA9IHNlbGYuZ2VuX3JhbmRvbV9tYXNrKGltZ3MsIG1hc2tfcmF0aW8pCiAgICAgICAgIyBlbmNvZGluZwogICAgICAgIHggPSBzZWxmLmVuY29kZXIoaW1ncywgbWFzaykKICAgICAgICByZXR1cm4geCwgbWFzawoKICAgIGRlZiBmb3J3YXJkX2RlY29kZXIoc2VsZiwgeCwgbWFzayk6CiAgICAgICAgeCA9IHNlbGYucHJvaih4KQogICAgICAgICMgYXBwZW5kIG1hc2sgdG9rZW4KICAgICAgICBuLCBjLCBoLCB3ID0geC5zaGFwZQogICAgICAgIG1hc2sgPSBtYXNrLnJlc2hhcGUoLTEsIGgsIHcpLnVuc3F1ZWV6ZSgxKS50eXBlX2FzKHgpCiAgICAgICAgbWFza190b2tlbiA9IHNlbGYubWFza190b2tlbi5yZXBlYXQoeC5zaGFwZVswXSwgMSwgeC5zaGFwZVsyXSwgeC5zaGFwZVszXSkKICAgICAgICB4ID0geCAqICgxLiAtIG1hc2spICsgbWFza190b2tlbiAqIG1hc2sKICAgICAgICAjIGRlY29kaW5nCiAgICAgICAgeCA9IHNlbGYuZGVjb2Rlcih4KQogICAgICAgICMgcHJlZAogICAgICAgIHByZWQgPSBzZWxmLnByZWQoeCkKICAgICAgICByZXR1cm4gcHJlZAoKICAgIGRlZiBmb3J3YXJkX2xvc3Moc2VsZiwgaW1ncywgcHJlZCwgbWFzayk6CiAgICAgICAgIiIiCiAgICAgICAgaW1nczogW04sIDMsIEgsIFddCiAgICAgICAgcHJlZDogW04sIEwsIHAqcCozXQogICAgICAgIG1hc2s6IFtOLCBMXSwgMCBpcyBrZWVwLCAxIGlzIHJlbW92ZQogICAgICAgICIiIgogICAgICAgIGlmIGxlbihwcmVkLnNoYXBlKSA9PSA0OgogICAgICAgICAgICBuLCBjLCBfLCBfID0gcHJlZC5zaGFwZQogICAgICAgICAgICBwcmVkID0gcHJlZC5yZXNoYXBlKG4sIGMsIC0xKQogICAgICAgICAgICBwcmVkID0gdG9yY2guZWluc3VtKCduY2wtPm5sYycsIHByZWQpCgogICAgICAgIHRhcmdldCA9IHNlbGYucGF0Y2hpZnkoaW1ncykKICAgICAgICBpZiBzZWxmLm5vcm1fcGl4X2xvc3M6CiAgICAgICAgICAgIG1lYW4gPSB0YXJnZXQubWVhbihkaW09LTEsIGtlZXBkaW09VHJ1ZSkKICAgICAgICAgICAgdmFyID0gdGFyZ2V0LnZhcihkaW09LTEsIGtlZXBkaW09VHJ1ZSkKICAgICAgICAgICAgdGFyZ2V0ID0gKHRhcmdldCAtIG1lYW4pIC8gKHZhciArIDEuZS02KSoqLjUKICAgICAgICBsb3NzID0gKHByZWQgLSB0YXJnZXQpICoqIDIKICAgICAgICBsb3NzID0gbG9zcy5tZWFuKGRpbT0tMSkgICMgW04sIExdLCBtZWFuIGxvc3MgcGVyIHBhdGNoCgogICAgICAgIGxvc3MgPSAobG9zcyAqIG1hc2spLnN1bSgpIC8gbWFzay5zdW0oKSAgIyBtZWFuIGxvc3Mgb24gcmVtb3ZlZCBwYXRjaGVzCiAgICAgICAgcmV0dXJuIGxvc3MKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbWdzLCBsYWJlbHM9Tm9uZSwgbWFza19yYXRpbz0wLjYpOgogICAgICAgIHgsIG1hc2sgPSBzZWxmLmZvcndhcmRfZW5jb2RlcihpbWdzLCBtYXNrX3JhdGlvKQogICAgICAgIHByZWQgPSBzZWxmLmZvcndhcmRfZGVjb2Rlcih4LCBtYXNrKQogICAgICAgIGxvc3MgPSBzZWxmLmZvcndhcmRfbG9zcyhpbWdzLCBwcmVkLCBtYXNrKQogICAgICAgIHJldHVybiBsb3NzLCBwcmVkLCBtYXNrCgpkZWYgY29udm5leHR2Ml9hdHRvKCoqa3dhcmdzKToKICAgIG1vZGVsID0gRkNNQUUoCiAgICAgICAgZGVwdGhzPVsyLCAyLCA2LCAyXSwgZGltcz1bNDAsIDgwLCAxNjAsIDMyMF0sICoqa3dhcmdzKQogICAgcmV0dXJuIG1vZGVsCgpkZWYgY29udm5leHR2Ml9mZW10bygqKmt3YXJncyk6CiAgICBtb2RlbCA9IEZDTUFFKAogICAgICAgIGRlcHRocz1bMiwgMiwgNiwgMl0sIGRpbXM9WzQ4LCA5NiwgMTkyLCAzODRdLCAqKmt3YXJncykKICAgIHJldHVybiBtb2RlbAoKZGVmIGNvbnZuZXh0djJfcGljbygqKmt3YXJncyk6CiAgICBtb2RlbCA9IEZDTUFFKAogICAgICAgIGRlcHRocz1bMiwgMiwgNiwgMl0sIGRpbXM9WzY0LCAxMjgsIDI1NiwgNTEyXSwgKiprd2FyZ3MpCiAgICByZXR1cm4gbW9kZWwKCmRlZiBjb252bmV4dHYyX25hbm8oKiprd2FyZ3MpOgogICAgbW9kZWwgPSBGQ01BRSgKICAgICAgICBkZXB0aHM9WzIsIDIsIDgsIDJdLCBkaW1zPVs4MCwgMTYwLCAzMjAsIDY0MF0sICoqa3dhcmdzKQogICAgcmV0dXJuIG1vZGVsCgpkZWYgY29udm5leHR2Ml90aW55KCoqa3dhcmdzKToKICAgIG1vZGVsID0gRkNNQUUoCiAgICAgICAgZGVwdGhzPVszLCAzLCA5LCAzXSwgZGltcz1bOTYsIDE5MiwgMzg0LCA3NjhdLCAqKmt3YXJncykKICAgIHJldHVybiBtb2RlbAoKZGVmIGNvbnZuZXh0djJfYmFzZSgqKmt3YXJncyk6CiAgICBtb2RlbCA9IEZDTUFFKAogICAgICAgIGRlcHRocz1bMywgMywgMjcsIDNdLCBkaW1zPVsxMjgsIDI1NiwgNTEyLCAxMDI0XSwgKiprd2FyZ3MpCiAgICByZXR1cm4gbW9kZWwKCmRlZiBjb252bmV4dHYyX2xhcmdlKCoqa3dhcmdzKToKICAgIG1vZGVsID0gRkNNQUUoCiAgICAgICAgZGVwdGhzPVszLCAzLCAyNywgM10sIGRpbXM9WzE5MiwgMzg0LCA3NjgsIDE1MzZdLCAqKmt3YXJncykKICAgIHJldHVybiBtb2RlbAoKZGVmIGNvbnZuZXh0djJfaHVnZSgqKmt3YXJncyk6CiAgICBtb2RlbCA9IEZDTUFFKAogICAgICAgIGRlcHRocz1bMywgMywgMjcsIDNdLCBkaW1zPVszNTIsIDcwNCwgMTQwOCwgMjgxNl0sICoqa3dhcmdzKQogICAgcmV0dXJuIG1vZGVs', 'vendor/models/utils.py': 'IyBDb3B5cmlnaHQgKGMpIE1ldGEgUGxhdGZvcm1zLCBJbmMuIGFuZCBhZmZpbGlhdGVzLgoKIyBBbGwgcmlnaHRzIHJlc2VydmVkLgoKIyBUaGlzIHNvdXJjZSBjb2RlIGlzIGxpY2Vuc2VkIHVuZGVyIHRoZSBsaWNlbnNlIGZvdW5kIGluIHRoZQojIExJQ0VOU0UgZmlsZSBpbiB0aGUgcm9vdCBkaXJlY3Rvcnkgb2YgdGhpcyBzb3VyY2UgdHJlZS4KCgppbXBvcnQgbnVtcHkucmFuZG9tIGFzIHJhbmRvbQoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCmZyb20gTWlua293c2tpRW5naW5lIGltcG9ydCBTcGFyc2VUZW5zb3IKCmNsYXNzIE1pbmtvd3NraUdSTihubi5Nb2R1bGUpOgogICAgIiIiIEdSTiBsYXllciBmb3Igc3BhcnNlIHRlbnNvcnMuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkaW0pOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZ2FtbWEgPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MoMSwgZGltKSkKICAgICAgICBzZWxmLmJldGEgPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MoMSwgZGltKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBjbSA9IHguY29vcmRpbmF0ZV9tYW5hZ2VyCiAgICAgICAgaW5fa2V5ID0geC5jb29yZGluYXRlX21hcF9rZXkKCiAgICAgICAgR3ggPSB0b3JjaC5ub3JtKHguRiwgcD0yLCBkaW09MCwga2VlcGRpbT1UcnVlKQogICAgICAgIE54ID0gR3ggLyAoR3gubWVhbihkaW09LTEsIGtlZXBkaW09VHJ1ZSkgKyAxZS02KQogICAgICAgIHJldHVybiBTcGFyc2VUZW5zb3IoCiAgICAgICAgICAgICAgICBzZWxmLmdhbW1hICogKHguRiAqIE54KSArIHNlbGYuYmV0YSArIHguRiwKICAgICAgICAgICAgICAgIGNvb3JkaW5hdGVfbWFwX2tleT1pbl9rZXksCiAgICAgICAgICAgICAgICBjb29yZGluYXRlX21hbmFnZXI9Y20pCgpjbGFzcyBNaW5rb3dza2lEcm9wUGF0aChubi5Nb2R1bGUpOgogICAgIiIiIERyb3AgUGF0aCBmb3Igc3BhcnNlIHRlbnNvcnMuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZHJvcF9wcm9iOiBmbG9hdCA9IDAuLCBzY2FsZV9ieV9rZWVwOiBib29sID0gVHJ1ZSk6CiAgICAgICAgc3VwZXIoTWlua293c2tpRHJvcFBhdGgsIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmRyb3BfcHJvYiA9IGRyb3BfcHJvYgogICAgICAgIHNlbGYuc2NhbGVfYnlfa2VlcCA9IHNjYWxlX2J5X2tlZXAKICAgIAogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgaWYgc2VsZi5kcm9wX3Byb2IgPT0gMC4gb3Igbm90IHNlbGYudHJhaW5pbmc6CiAgICAgICAgICAgIHJldHVybiB4CiAgICAgICAgY20gPSB4LmNvb3JkaW5hdGVfbWFuYWdlcgogICAgICAgIGluX2tleSA9IHguY29vcmRpbmF0ZV9tYXBfa2V5CiAgICAgICAga2VlcF9wcm9iID0gMSAtIHNlbGYuZHJvcF9wcm9iCiAgICAgICAgbWFzayA9IHRvcmNoLmNhdChbCiAgICAgICAgICAgIHRvcmNoLm9uZXMobGVuKF8pKSBpZiByYW5kb20udW5pZm9ybSgwLCAxKSA+IHNlbGYuZHJvcF9wcm9iCiAgICAgICAgICAgIGVsc2UgdG9yY2guemVyb3MobGVuKF8pKSBmb3IgXyBpbiB4LmRlY29tcG9zZWRfY29vcmRpbmF0ZXMKICAgICAgICBdKS52aWV3KC0xLCAxKS50byh4LmRldmljZSkKICAgICAgICBpZiBrZWVwX3Byb2IgPiAwLjAgYW5kIHNlbGYuc2NhbGVfYnlfa2VlcDoKICAgICAgICAgICAgbWFzay5kaXZfKGtlZXBfcHJvYikKICAgICAgICByZXR1cm4gU3BhcnNlVGVuc29yKAogICAgICAgICAgICAgICAgeC5GICogbWFzaywKICAgICAgICAgICAgICAgIGNvb3JkaW5hdGVfbWFwX2tleT1pbl9rZXksCiAgICAgICAgICAgICAgICBjb29yZGluYXRlX21hbmFnZXI9Y20pCgpjbGFzcyBNaW5rb3dza2lMYXllck5vcm0obm4uTW9kdWxlKToKICAgICIiIiBDaGFubmVsLXdpc2UgbGF5ZXIgbm9ybWFsaXphdGlvbiBmb3Igc3BhcnNlIHRlbnNvcnMuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBub3JtYWxpemVkX3NoYXBlLAogICAgICAgIGVwcz0xZS02LAogICAgKToKICAgICAgICBzdXBlcihNaW5rb3dza2lMYXllck5vcm0sIHNlbGYpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmxuID0gbm4uTGF5ZXJOb3JtKG5vcm1hbGl6ZWRfc2hhcGUsIGVwcz1lcHMpCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbnB1dCk6CiAgICAgICAgb3V0cHV0ID0gc2VsZi5sbihpbnB1dC5GKQogICAgICAgIHJldHVybiBTcGFyc2VUZW5zb3IoCiAgICAgICAgICAgIG91dHB1dCwKICAgICAgICAgICAgY29vcmRpbmF0ZV9tYXBfa2V5PWlucHV0LmNvb3JkaW5hdGVfbWFwX2tleSwKICAgICAgICAgICAgY29vcmRpbmF0ZV9tYW5hZ2VyPWlucHV0LmNvb3JkaW5hdGVfbWFuYWdlcikKICAgICAgICAgICAgCmNsYXNzIExheWVyTm9ybShubi5Nb2R1bGUpOgogICAgIiIiIExheWVyTm9ybSB0aGF0IHN1cHBvcnRzIHR3byBkYXRhIGZvcm1hdHM6IGNoYW5uZWxzX2xhc3QgKGRlZmF1bHQpIG9yIGNoYW5uZWxzX2ZpcnN0LiAKICAgIFRoZSBvcmRlcmluZyBvZiB0aGUgZGltZW5zaW9ucyBpbiB0aGUgaW5wdXRzLiBjaGFubmVsc19sYXN0IGNvcnJlc3BvbmRzIHRvIGlucHV0cyB3aXRoIAogICAgc2hhcGUgKGJhdGNoX3NpemUsIGhlaWdodCwgd2lkdGgsIGNoYW5uZWxzKSB3aGlsZSBjaGFubmVsc19maXJzdCBjb3JyZXNwb25kcyB0byBpbnB1dHMgCiAgICB3aXRoIHNoYXBlIChiYXRjaF9zaXplLCBjaGFubmVscywgaGVpZ2h0LCB3aWR0aCkuCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBub3JtYWxpemVkX3NoYXBlLCBlcHM9MWUtNiwgZGF0YV9mb3JtYXQ9ImNoYW5uZWxzX2xhc3QiKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLndlaWdodCA9IG5uLlBhcmFtZXRlcih0b3JjaC5vbmVzKG5vcm1hbGl6ZWRfc2hhcGUpKQogICAgICAgIHNlbGYuYmlhcyA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyhub3JtYWxpemVkX3NoYXBlKSkKICAgICAgICBzZWxmLmVwcyA9IGVwcwogICAgICAgIHNlbGYuZGF0YV9mb3JtYXQgPSBkYXRhX2Zvcm1hdAogICAgICAgIGlmIHNlbGYuZGF0YV9mb3JtYXQgbm90IGluIFsiY2hhbm5lbHNfbGFzdCIsICJjaGFubmVsc19maXJzdCJdOgogICAgICAgICAgICByYWlzZSBOb3RJbXBsZW1lbnRlZEVycm9yIAogICAgICAgIHNlbGYubm9ybWFsaXplZF9zaGFwZSA9IChub3JtYWxpemVkX3NoYXBlLCApCiAgICAKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIGlmIHNlbGYuZGF0YV9mb3JtYXQgPT0gImNoYW5uZWxzX2xhc3QiOgogICAgICAgICAgICByZXR1cm4gRi5sYXllcl9ub3JtKHgsIHNlbGYubm9ybWFsaXplZF9zaGFwZSwgc2VsZi53ZWlnaHQsIHNlbGYuYmlhcywgc2VsZi5lcHMpCiAgICAgICAgZWxpZiBzZWxmLmRhdGFfZm9ybWF0ID09ICJjaGFubmVsc19maXJzdCI6CiAgICAgICAgICAgIHUgPSB4Lm1lYW4oMSwga2VlcGRpbT1UcnVlKQogICAgICAgICAgICBzID0gKHggLSB1KS5wb3coMikubWVhbigxLCBrZWVwZGltPVRydWUpCiAgICAgICAgICAgIHggPSAoeCAtIHUpIC8gdG9yY2guc3FydChzICsgc2VsZi5lcHMpCiAgICAgICAgICAgIHggPSBzZWxmLndlaWdodFs6LCBOb25lLCBOb25lXSAqIHggKyBzZWxmLmJpYXNbOiwgTm9uZSwgTm9uZV0KICAgICAgICAgICAgcmV0dXJuIHgKCmNsYXNzIEdSTihubi5Nb2R1bGUpOgogICAgIiIiIEdSTiAoR2xvYmFsIFJlc3BvbnNlIE5vcm1hbGl6YXRpb24pIGxheWVyCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBkaW0pOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZ2FtbWEgPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MoMSwgMSwgMSwgZGltKSkKICAgICAgICBzZWxmLmJldGEgPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MoMSwgMSwgMSwgZGltKSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBHeCA9IHRvcmNoLm5vcm0oeCwgcD0yLCBkaW09KDEsMiksIGtlZXBkaW09VHJ1ZSkKICAgICAgICBOeCA9IEd4IC8gKEd4Lm1lYW4oZGltPS0xLCBrZWVwZGltPVRydWUpICsgMWUtNikKICAgICAgICByZXR1cm4gc2VsZi5nYW1tYSAqICh4ICogTngpICsgc2VsZi5iZXRhICsgeA=='}
SOURCE_HASHES = {'data_utils.py': '65422fc0cc6e492b1e6a1d7d30a14a4688e9f3341aa20d51236e17e6df805d93', 'rect_fcmae.py': 'f5858da18c9407f203e872841adda1d8cc290e067c4c935b5322e02501d2fd88', 'checkpoint_utils.py': '7c4b0c415ecf6a7c8525a5733c79fe428e4b96f4fc12f501734c794fb8676d04', 'train.py': 'b87fa25cc4dd9e46c5ff33605aea6904247562a12026a3186deb36625df88e0e', 'setup_runtime.py': '0cf7e6599bd654b9531fd2743de80e1ddc458ed01ecf1b672ca9f5f404b2f9b6', 'test_data_utils.py': 'a4972b4337a28bda10825e4a53147c3fc3428657ecd668274eb5896d73c369b5', 'test_checkpoint_utils.py': '794e8ebcc7f7be99fd190e965b3026f13c0afc87ad4ab33c2ca1157ef28e7201', 'training_state.py': 'fbae5f82479947ab8a44b4b5df56d54fa66dd6499ffd48ca19b6e8b7bfaad71a', 'test_training_state.py': '23fd8b4f3e71d280fe649f72b05940d8ece8952b62e38e2778f215566f1a2025', 'benchmark.py': '27f6d02a75850a496ce949c4c3a7e4f2d6955cb1811226368adb5c2688a9141e', 'benchmark_report.py': '320a2e5d51e6d53bea3b865639b17c3a2a64d5cdb7373ed3a319926891ee71bc', 'test_benchmark_report.py': '6114e19dd7df19034125c264770e89395f3996d0ffb0a30a52b2e32cfaf8792e', 'downstream_holdouts_v2.csv': '93f7b37676507fb55e540b73f5ac78e4ba41da8b0be7d1224b508da26e498895', 'holdout_provenance.json': '0800a46ebba593e23e7cfbe231f196588691bee416c224b4e80893e378a39ff7', 'vendor/LICENSE': '9ab7947f327be897e841ce58b90996c4b814c35127897b383f17616510115b9b', 'vendor/README.md': '4b2148ccd980d7ad0a89b3109681e2f32e2ab2bee425ee10d7bf9059eb389e1f', 'vendor/models/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'vendor/models/convnextv2.py': 'c553af39d6ed5e89d1ceec4598fb48001c3757f7a547c535812291ecb6fc2f1a', 'vendor/models/convnextv2_sparse.py': '32b048337f973bb1685fecc3a3b62d73936e611f37b05fd3dcd90620c5a18881', 'vendor/models/fcmae.py': 'b544d909cb2eb5693bab7732078b7548f0c6d6c50266cee0c30c5c15aaa22cd1', 'vendor/models/utils.py': 'f7eca4be0696ccdae8b7a05387870cacc66d6408a7acc44b0b7f1ae56b2df631'}
for name, encoded in PAYLOAD.items():
    content = base64.b64decode(encoded)
    assert hashlib.sha256(content).hexdigest() == SOURCE_HASHES[name]
    target = CODE/name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
(EXPERIMENT/'source_hashes.json').write_text(json.dumps(SOURCE_HASHES, indent=2))
print('Materialized', len(PAYLOAD), 'source/license files')


## 3. Build the isolated sparse-training runtime — Runtime compatibility fixes v5

This can take several minutes and downloads a compiler/CUDA toolkit. Build logs
are streamed. A successful import is only an environment check; the pilot later
tests actual sparse convolutions and dense encoder export. There is **no fallback
to input-only masking or to a different initialization** if setup fails.

The primary package versions are pinned; the resolved conda package URLs and
`pip freeze` are saved for reproducing the exact successful environment. Colab
images/drivers change, so retain any compiler log if this cell fails.


In [ ]:
stage_started = time.perf_counter()
setup_log = RUN/'runtime_setup.log'
if PYTHON_OVERRIDE:
    TRAIN_PYTHON = str(PYTHON_OVERRIDE)
else:
    with open(setup_log, 'a') as log:
        process = subprocess.Popen([sys.executable, '-u', str(CODE/'setup_runtime.py'), str(RUNTIME)],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        if process.wait() != 0:
            raise RuntimeError(f'Sparse runtime setup failed. See {setup_log}')
    TRAIN_PYTHON = str(RUNTIME/'env/bin/python')

TRAIN_ENV = dict(os.environ, OMP_NUM_THREADS='4', PYTHONUNBUFFERED='1')
if not PYTHON_OVERRIDE:
    TRAIN_ENV['PATH'] = str(RUNTIME/'env/bin')+os.pathsep+os.environ['PATH']
    TRAIN_ENV['LD_LIBRARY_PATH'] = str(RUNTIME/'env/lib')+os.pathsep+os.environ.get('LD_LIBRARY_PATH','')
    TRAIN_ENV['CUDA_HOME'] = str(RUNTIME/'env')
    for name in ['conda-explicit.txt', 'pip-freeze.txt', 'runtime_ready.json']:
        shutil.copy2(RUNTIME/name, RUN/name)
subprocess.run([TRAIN_PYTHON, '-c', 'import torch, MinkowskiEngine; print(torch.__version__, torch.version.cuda, MinkowskiEngine.__version__); assert torch.cuda.is_available()'],
               env=TRAIN_ENV, check=True)
subprocess.run([TRAIN_PYTHON, str(CODE/'test_data_utils.py')], env=TRAIN_ENV, check=True)
subprocess.run([TRAIN_PYTHON, str(CODE/'test_training_state.py')], env=TRAIN_ENV, check=True)
subprocess.run([TRAIN_PYTHON, str(CODE/'test_benchmark_report.py')], env=TRAIN_ENV, check=True)
PREPARATION_TIMINGS['runtime_setup_and_tests_seconds'] = time.perf_counter()-stage_started


## 4. Verify and extract the generated image archive

The checksum is verified before extraction. Only regular files/directories are
accepted; absolute paths, traversal paths, links, and devices are rejected. The
index and all images are kept together. If resuming in a fresh Colab session,
this cell restages the data without changing saved split assignments.


In [ ]:
import tarfile
stage_started = time.perf_counter()
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8<<20), b''):
            digest.update(block)
    return digest.hexdigest()

assert ARCHIVE.is_file() and ARCHIVE_SHA.is_file(), 'Image generation/archive is not ready at configured path'
expected_sha = ARCHIVE_SHA.read_text().split()[0].lower()
assert len(expected_sha) == 64 and all(c in '0123456789abcdef' for c in expected_sha)
local_archive = WORK/ARCHIVE.name
if not local_archive.exists() or file_sha256(local_archive) != expected_sha:
    staging = local_archive.with_suffix('.copying')
    shutil.copyfile(ARCHIVE, staging)
    assert file_sha256(staging) == expected_sha, 'Archive checksum mismatch'
    staging.replace(local_archive)

DATA = WORK/'images_release'
marker = DATA/'.archive_sha256'
if not marker.exists() or marker.read_text().strip() != expected_sha:
    if DATA.exists():
        raise RuntimeError('Incomplete or different extraction exists: choose a new WORK directory or inspect it before removing it')
    stage = WORK/'images_release_extracting'
    if stage.exists():
        raise RuntimeError('Previous extraction was interrupted; inspect/remove images_release_extracting and retry')
    stage.mkdir()
    with tarfile.open(local_archive) as tf:
        members = tf.getmembers()
        required_bytes = sum(m.size for m in members if m.isfile())
        assert shutil.disk_usage(WORK).free > required_bytes*1.1, 'Insufficient extraction space'
        for member in members:
            destination = (stage/member.name).resolve()
            if Path(member.name).is_absolute() or not destination.is_relative_to(stage.resolve()):
                raise ValueError('Unsafe archive path: '+member.name)
            if not (member.isfile() or member.isdir()):
                raise ValueError('Unsupported archive member: '+member.name)
        for member in members:
            destination = stage/member.name
            if member.isdir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with tf.extractfile(member) as source, open(destination, 'wb') as target:
                    shutil.copyfileobj(source, target)
    (stage/'.archive_sha256').write_text(expected_sha)
    stage.rename(DATA)
indexes = [p for p in DATA.rglob('index.csv') if (p.parent/'images').is_dir()]
assert len(indexes) == 1, f'Expected one image release index; found {indexes}'
INDEX = indexes[0]
CONFIG.update(index=str(INDEX), image_root=str(INDEX.parent), archive_sha256=expected_sha,
              resize_cache=str(WORK/'resize_cache'/expected_sha))
print('Image index:', INDEX)
PREPARATION_TIMINGS['archive_copy_verification_extraction_seconds'] = time.perf_counter()-stage_started


## 5. Download the self-supervised ImageNet-1K checkpoint

This URL is the **FCMAE** Tiny checkpoint in Meta's self-supervised table, not
the 1K classifier or 22K fine-tuned checkpoint. The loader rejects missing,
unexpected, or incompatible tensors rather than silently leaving layers random.
The official release contains 196 encoder tensors; the reconstruction decoder
is newly initialized. The downloaded SHA-256 is recorded and checked against
the file inspected while building this notebook (not a publisher-supplied digest).

Meta's code is MIT; the provided ImageNet weights are CC-BY-NC, as stated in the
bundled license. Record that provenance with downstream artifacts.


In [ ]:
import urllib.request
stage_started = time.perf_counter()
CHECKPOINT_URL = 'https://dl.fbaipublicfiles.com/convnext/convnextv2/pt_only/convnextv2_tiny_1k_224_fcmae.pt'
checkpoint = WORK/'convnextv2_tiny_1k_224_fcmae.pt'
if not checkpoint.exists():
    temporary = checkpoint.with_suffix('.downloading')
    with urllib.request.urlopen(CHECKPOINT_URL, timeout=120) as response, open(temporary, 'wb') as target:
        shutil.copyfileobj(response, target)
    temporary.replace(checkpoint)
CONFIG.update(checkpoint=str(checkpoint), checkpoint_url=CHECKPOINT_URL)
verified_checkpoint_sha = 'b25eadb89ad634b5c925ea04cdacebafe3e462486baf9482dc2d832bb9af57ef'
assert file_sha256(checkpoint) == verified_checkpoint_sha, 'Checkpoint differs from inspected official release'
subprocess.run([TRAIN_PYTHON, str(CODE/'test_checkpoint_utils.py'), '--checkpoint', str(checkpoint)],
               env=TRAIN_ENV, check=True)
config_path = WORK/'config.json'
config_path.write_text(json.dumps(CONFIG, indent=2))
print('Checkpoint SHA-256:', file_sha256(checkpoint))
print(json.dumps(CONFIG, indent=2))
PREPARATION_TIMINGS['checkpoint_download_verification_seconds'] = time.perf_counter()-stage_started


## 5A. Benchmark the assigned GPU and save a cost report

Run this on **each GPU you actually obtain** (T4, L4, A100). It cannot measure
unavailable GPUs. Keep `BENCHMARK_ONLY=True` while comparing: subsequent pilot
and full-training cells will be skipped, even with Run all.

Enter the **compute units/hour currently displayed in Colab** in the configuration
cell. No prices are assumed. If left as `None`, timing reports still work and
cost columns remain blank. A rate of 0 means an actually free allocated session,
not guaranteed free access or a guarantee of any session duration.

This runs the real sparse model at the configured resolution/batch/precision:
2 warmup updates, then 3 measured rounds of 6 updates (192 images per round at
the defaults), plus validation and actual mounted-Drive write timings. Expect
several minutes or longer depending on hardware. The GPU synchronization points
ensure elapsed time includes completed GPU work, not just queued operations.

Cache preparation is timed separately using a fresh sample cache. Full-dataset
cache cost is extrapolated; it is not a directly measured first full epoch.
Training uses a disposable model, separate from saved pretraining checkpoints.
The reports include memory, initialization/preflight status, settings, sample
identities, runtime versions, and projections for 1, 20, and the configured total
epochs (100 by default). Repeated-round ranges are **not confidence intervals**.
Costs include estimated checkpoint overhead and one observed preparation session;
they exclude future reconnect/setup repetitions and idle time.

Reports go to `MyDrive/CardioSentry/runs/fcmae_tiny_v1_benchmarks/<GPU_timestamp>/`.
Send **`summary.md` and `report.json`** after each GPU, or the combined
`comparison.csv` after all available GPUs. Compare only matching comparison keys;
do not change batch size, precision, resolution, data, or code between GPU tests.
An out-of-memory or compatibility failure is recorded as a failure, not assigned
an invented runtime estimate. The isolated environment must first build successfully.

Once a GPU is chosen, set `RUN_BENCHMARK=False` and `BENCHMARK_ONLY=False`,
rerun the configuration cell, then run the pilot before enabling `RUN_FULL`.


In [ ]:
from datetime import datetime, timezone
import re, uuid, math
from IPython.display import display, Markdown

BENCHMARK_ROOT = DRIVE/'runs/fcmae_tiny_v1_benchmarks'
BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_BENCHMARK:
    assert COMPUTE_UNITS_PER_HOUR is None or (
        isinstance(COMPUTE_UNITS_PER_HOUR, (int,float)) and math.isfinite(COMPUTE_UNITS_PER_HOUR)
        and COMPUTE_UNITS_PER_HOUR >= 0), 'Enter a nonnegative rate or None'
    gpu_name = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True).splitlines()[0]
    label = re.sub(r'[^A-Za-z0-9_-]+','_',gpu_name)
    benchmark_dir = BENCHMARK_ROOT/(label+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
    benchmark_dir.mkdir()
    benchmark_options = dict(units_per_hour=COMPUTE_UNITS_PER_HOUR,
                             preparation_timings=dict(PREPARATION_TIMINGS),
                             warmup_updates=2, measured_updates=6, rounds=3, validation_images=64)
    options_path = benchmark_dir/'options.json'
    options_path.write_text(json.dumps(benchmark_options, indent=2))
    command = [TRAIN_PYTHON,'-u',str(CODE/'benchmark.py'),'--config',str(config_path),
               '--options',str(options_path),'--output',str(benchmark_dir)]
    with open(benchmark_dir/'console.log','w') as log:
        process = subprocess.Popen(command,cwd=CODE,env=TRAIN_ENV,stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT,text=True)
        try:
            for line in process.stdout:
                print(line,end=''); log.write(line); log.flush()
            benchmark_exit = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
            raise
    if (benchmark_dir/'summary.md').exists():
        display(Markdown((benchmark_dir/'summary.md').read_text()))
    print('Send these files:',benchmark_dir/'summary.md',benchmark_dir/'report.json')
else:
    benchmark_exit = 0
    print('Benchmark disabled; loading existing reports only.')

# The report helper uses only Python's standard library; no kernel packages change.
import importlib.util
spec = importlib.util.spec_from_file_location('fcmae_benchmark_report',CODE/'benchmark_report.py')
report_helpers = importlib.util.module_from_spec(spec); spec.loader.exec_module(report_helpers)
import pandas as pd
comparison = pd.DataFrame(report_helpers.comparison_rows(BENCHMARK_ROOT))
if not comparison.empty:
    comparison.to_csv(BENCHMARK_ROOT/'comparison.csv',index=False)
    display(comparison.drop(columns=['report']))
    print('Compare rows only within the same comparison_key; no automatic cross-setting ranking.')
if benchmark_exit:
    raise RuntimeError('Benchmark failed. Send its report.json and console.log before changing training settings.')


## 6. Run the pilot and inspect reconstruction

The pilot runs the real Tiny model with the configured input size and accumulation.
It checks rectangular output bounds, independence from hidden pixel values,
finite gradients, and exported dense-encoder parity on an all-visible batch of
one. The latter also checks sparse kernel ordering during checkpoint conversion.

It then trains for a short epoch, validates with repeatable masks, and writes
timing, memory, loss, and reconstruction panels under `runs/.../pilot/`.
Resized, unpadded PNGs are cached lazily on local disk, avoiding repeated
600-DPI decoding in later epochs. Reserve additional disk space for this cache.
The reconstruction panel uses **ground-truth patch mean/variance for display**
because the objective predicts normalized pixels; it is not an independent
recovery of absolute brightness or contrast.

If memory is insufficient, lower `batch_size`, adjust accumulation as needed,
and use a new run directory. If sparse FP16 is incompatible with your GPU,
set `amp=False` and rerun in a new run directory. Neither change is automatic.


In [ ]:
def launch_training(pilot):
    command = [TRAIN_PYTHON, '-u', str(CODE/'train.py'), '--config', str(config_path)]
    if pilot:
        command.append('--pilot')
    log_path = RUN/('pilot_console.log' if pilot else 'full_console.log')
    with open(log_path, 'a') as log:
        process = subprocess.Popen(command, cwd=CODE, env=TRAIN_ENV,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end=''); log.write(line); log.flush()
            if process.wait() != 0:
                raise RuntimeError(f'Training failed; inspect {log_path}')
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
            raise

if not BENCHMARK_ONLY:
    launch_training(pilot=True)
    PILOT_PASSED = True
    from IPython.display import display, Image as DisplayImage
    print((RUN/'pilot/split_audit.json').read_text())
    print((RUN/'pilot/metrics.csv').read_text())
    for figure in sorted((RUN/'pilot/previews').glob('reconstruction_*.png')):
        display(DisplayImage(filename=str(figure)))
else:
    print('Pilot skipped for GPU cost comparison. Set BENCHMARK_ONLY=False when ready to train.')


## 7. Full continued pretraining

After inspecting the pilot, set `RUN_FULL=True` in the configuration cell and
supply `HOLDOUT_CSV`, then rerun the configuration-dependent cells. A full run
requires the exclusion audit; a pilot may run without it and is not a reportable
experiment. The runtime refuses mismatched resume configurations.

All weights are saved directly to mounted Google Drive under
`MyDrive/CardioSentry/runs/fcmae_tiny_v1_768_seed42/`, with separate `pilot/` and
`full/` runs. Within each run:

- `checkpoints/last.pt`: latest full training state, saved every 100 optimizer
  updates or approximately 15 minutes (whichever occurs first, at the next
  completed update), before validation, and after each completed epoch.
- `checkpoints/previous.pt`: preceding snapshot, used automatically if `last.pt`
  is absent or unreadable. This is a rolling backup, not an accumulating archive.
- `checkpoints/checkpoint_epoch_XXXX.pt`: full resumable state every 10 epochs
  and at the final epoch, so epoch 20 remains available after training advances.
- `encoders/`: initial, best-SSL, and periodic encoder-only weights.
- `previews/`: reconstruction images. Metrics and audits remain at the run root.

`checkpoints/last.pt` contains model, optimizer, scaler, RNG states, image order,
the next unprocessed batch, and running epoch metrics. It resumes **within the
epoch** from the last saved update. At batch 2 × accumulation 16, 100 updates
usually cover 3,200 images; the time trigger can save sooner. Saving occurs only
after an optimizer update, with no partially accumulated gradients to restore.
Unsaved work since the last snapshot is replayed after a disconnect. If training
finished but validation was interrupted, only validation/artifact writing repeats.

Checkpoints are serialized to a temporary file, flushed, then renamed; the
previous snapshot is retained. A mounted Drive write is not a guarantee that
remote synchronization survives an immediate hard shutdown. Wait for the saved
message and verify the checkpoint appears in Drive before intentionally ending
a session. There is a write-time pause; GPU/Drive timing must be measured.

The sampler records **consumed** batches, not prefetched ones, and loader RNG is
separate from model/masking RNG. This assumes the current deterministic image
preprocessing; adding random worker augmentations requires extending this design.
Set an interval to 0 to disable that trigger, or both to 0 for epoch-boundary
saving only. Old checkpoints from the previous notebook format must be resumed
with that original notebook; this version refuses implicit migration.
The original sparse engine may still have nondeterministic GPU operations;
saved seeds do not imply bitwise reproducibility.

The 100-epoch schedule and 2e-5 learning rate are initial engineering choices.
Do not infer convergence from the scheduled length. SSL reconstruction loss is
a diagnostic; downstream validation selects the useful encoder checkpoint.


In [ ]:
if RUN_FULL and not BENCHMARK_ONLY:
    assert globals().get('PILOT_PASSED'), 'Complete the pilot first'
    assert HOLDOUT_CSV is not None and Path(HOLDOUT_CSV).is_file(), 'Configure the canonical downstream holdout CSV'
    launch_training(pilot=False)
    print('Full pretraining complete:', RUN/'full')
else:
    print('Full training skipped: RUN_FULL is False or BENCHMARK_ONLY is True.')


## 8. Review metrics and use the encoder later

`encoder_initial.pt` is the matched ImageNet-1K FCMAE control before ECG updates.
`encoder_best_ssl.pt` is selected by SSL reconstruction only.
`encoder_epoch_XXXX.pt` files support downstream checkpoint comparisons.

Exports use **official dense ConvNeXt-V2 keys**, not timm's renamed keys. In the
same environment, use `models.convnextv2.convnextv2_tiny(num_classes=...)`, then
load `checkpoint['model']` with `strict=False`. Assert that only
`norm.weight`, `norm.bias`, `head.weight`, and `head.bias` are missing; initialize
those identically for the matched control and adapted model. The sparse encoder
has no pooled classification norm/head. A timm model needs an explicit key map.

Apply the same aspect-preserving input preprocessing during downstream
fine-tuning. Evaluate per-label metrics for STEMI, AF and LVH on downstream
validation; reserve test patients for final evaluation. This notebook does not
train the downstream classifier or claim that reconstruction improves diagnosis.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
if not BENCHMARK_ONLY:
    result = RUN/('full' if RUN_FULL else 'pilot')
    metrics = pd.read_csv(result/'metrics.csv')
    display(metrics)
    ax = metrics.plot(x='epoch', y=['train_loss', 'ssl_val_loss'], marker='o', figsize=(8,4))
    ax.set_ylabel('Masked normalized-pixel MSE'); ax.grid(alpha=.2)
    plt.show()
    print('Artifacts:')
    for path in sorted(result.rglob('*')):
        if path.is_file():
            print(path.relative_to(result))
else:
    print('Benchmark-only session: use the comparison table and benchmark reports above.')


## References and method boundaries

- [Official ConvNeXt-V2 / FCMAE](https://github.com/facebookresearch/ConvNeXt-V2)
- [ConvNeXt V2 paper](https://arxiv.org/abs/2301.00808)
- [HeartBEiT](https://www.nature.com/articles/s41746-023-00840-9)
- [ECG waveform/scalogram MAE](https://pmc.ncbi.nlm.nih.gov/articles/PMC12238358/)

Later controlled experiments: 1152 × 1536 inputs; waveform-stratified masking;
lead-local temporal masking; loss weighting. None is enabled here. The archive
does not need waveform masks for this uniform baseline. Preserve master images
and rendering provenance for those later experiments.
